# Business entity resolution - full pipeline (Kaggle, CPU)

Self-contained: every source file of the repository is written to `/kaggle/working/er` by the cells
below (no internet needed), the attached dataset is located by file name, then the notebook runs the
unit tests, a **smoke test** (whole pipeline on a 0.3 % slice with a scored hold-out, ~3 min) and the
**full pipeline** (~2-2.5 h on the 4 Kaggle CPU threads). Outputs:

* `/kaggle/working/output/matching_results.tsv`, `candidate_pairs.tsv` - the submission
* `/kaggle/working/diagnostics/` - validation report, leakage check, threshold analysis, runtime profile, summary
* `/kaggle/working/logs/` - one log per step

Settings: *Accelerator = None*, *Internet* not required. Use **Save Version -> Save & Run All** for the full run.

In [ ]:
# ---------------------------------------------------------------- settings
SMOKE_ONLY = False          # True: stop after the smoke test (~3 min)
SKIP_SMOKE = False          # True: go straight to the full run
ROUNDS1, ROUNDS2 = 150, 100 # LightGBM boosting rounds (stage 1 / stage 2); 300 / 200 = original, same score, 2x slower
DATASET_ROOT = "/kaggle/input"   # searched recursively for train_source1.tsv etc.
CKPT_NAME = "kaggle_full"        # training checkpoint name (<work>/checkpoints/<name>; re-running resumes it)
CHANNELS = None                  # blocking.py --channels, e.g. "combined=5,nchar=3,addr=2,rev=2"; None = the default of blocking.py

import os, sys, subprocess, time, json, shutil, glob
ER = "/kaggle/working/er"
OUT = "/kaggle/working/output"
LOGS = "/kaggle/working/logs"
DIAG = "/kaggle/working/diagnostics"
WORK = "/kaggle/temp/er_work" if os.access("/kaggle/temp", os.W_OK) else "/kaggle/working/er_work"
for d in (ER, OUT, LOGS, DIAG, WORK, f"{ER}/src", f"{ER}/tools", f"{ER}/tests"):
    os.makedirs(d, exist_ok=True)
print("python", sys.version.split()[0], "| cpus", os.cpu_count(), "| work dir", WORK)

In [ ]:
# ---------------------------------------------------------------- environment check
import importlib
need = {"polars": "1.0", "lightgbm": "4.0", "rapidfuzz": "3.0", "scipy": "1.10", "numpy": "1.24", "pyarrow": "10.0"}
for mod, minv in need.items():
    try:
        m = importlib.import_module(mod)
        v = tuple(int(x) for x in m.__version__.split(".")[:2])
        ok = v >= tuple(int(x) for x in minv.split("."))
        print(f"{mod:10s} {m.__version__:10s} {'ok' if ok else 'TOO OLD (need >= ' + minv + ')'}")
        assert ok
    except ImportError:
        print(f"{mod:10s} MISSING -> pip install {mod}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", mod], check=True)
try:
    import unidecode; print("unidecode  ", unidecode.__version__ if hasattr(unidecode, "__version__") else "ok")
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unidecode"])
    print("unidecode   installed" if r.returncode == 0 else "unidecode   not available (no internet): accent-folding fallback is used")
import pytest; print("pytest      ok")

## Source files

In [ ]:
%%writefile /kaggle/working/er/src/common.py
"""Shared paths, IO helpers and logging."""
import argparse
import os
import sys
import time

import polars as pl

SOURCES = ("source1", "source2", "source3")
_T0 = time.time()


def log(*msg):
    print(f"[{time.time() - _T0:8.1f}s]", *msg, file=sys.stderr, flush=True)


def read_tsv(path):
    """Reads a challenge TSV. quote_char=None: fields are never quoted and may contain quotes."""
    df = pl.read_csv(path, separator="\t", quote_char=None, infer_schema_length=0)  # all columns as strings
    return df.with_columns(pl.when(pl.col(c) == "").then(None).otherwise(pl.col(c)).alias(c) for c in df.columns)


def left_join_ordered(df, other, on):
    """Left join that keeps the row order of `df` on every polars version (the `maintain_order`
    argument of DataFrame.join only exists in recent releases)."""
    idx = "__order__"
    return df.with_row_index(idx).join(other, on=on, how="left").sort(idx).drop(idx)


def source_path(data_dir, split, source):
    return os.path.join(data_dir, split, f"{split}_{source}.tsv")


def base_args(desc):
    ap = argparse.ArgumentParser(description=desc)
    ap.add_argument("--data-dir", required=True, help="the challenge dataset/ directory (contains train/ and test/)")
    ap.add_argument("--work-dir", required=True, help="scratch directory for intermediate artefacts")
    return ap


def split_dir(work_dir, split):
    d = os.path.join(work_dir, split)
    os.makedirs(d, exist_ok=True)
    return d


def n_workers():
    return max(1, (os.cpu_count() or 2))


class Stage:
    """Context manager that appends wall-clock / CPU time of one stage to <work>/profile.json.

        with Stage(work_dir, "blocking_train"):
            ...

    CPU time is the process + children CPU seconds, so cpu/wall > 1 shows the stage actually
    ran in parallel; cpu/wall ~ 1 on a multi-core box means it was effectively sequential."""

    def __init__(self, work_dir, name):
        self.work_dir, self.name = work_dir, name

    def __enter__(self):
        self.t0, self.c0 = time.time(), _cpu_seconds()
        return self

    def __exit__(self, *exc):
        import json
        wall, cpu = time.time() - self.t0, _cpu_seconds() - self.c0
        rec = {"wall_s": round(wall, 2), "cpu_s": round(cpu, 2), "cpu_per_wall": round(cpu / max(wall, 1e-6), 2),
               "workers": n_workers(), "ok": exc[0] is None}
        os.makedirs(self.work_dir, exist_ok=True)
        path = os.path.join(self.work_dir, "profile.json")
        try:
            with open(path) as f:
                prof = json.load(f)
        except (OSError, ValueError):
            prof = {}
        prof[self.name] = rec
        with open(path, "w") as f:
            json.dump(prof, f, indent=1)
        log(f"stage {self.name}: {wall:.1f}s wall, {cpu:.1f}s cpu ({rec['cpu_per_wall']}x)")


def _cpu_seconds():
    t = os.times()
    return t.user + t.system + t.children_user + t.children_system


In [ ]:
%%writefile /kaggle/working/er/src/textnorm.py
"""Country-agnostic text normalisation for business names and addresses.

Everything here is rule based (hand-written abbreviation tables) plus an optional
Indic-script -> Latin lexicon that is *learned from the training ground truth*
(see build_lexicon.py). No external data or services are used.
"""
import re
import unicodedata

from rapidfuzz import fuzz

try:
    from unidecode import unidecode
except ImportError:  # offline Kaggle image without the package: accent folding only
    def unidecode(s):
        """Fallback: strip diacritics of Latin text (NFKD). Indic script cannot be transliterated this
        way; such tokens keep their own script unless the learned lexicon covers them."""
        return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))

INDIC_RE = re.compile(r"[ऀ-෿]")
ZW_RE = re.compile(r"[​-‏⁠﻿­]")
ID_RE = re.compile(r"\(\s*id\s*:\s*\d+\s*\)", re.I)
ALIAS_RE = re.compile(
    r"\s*(?:\||\b(?:d\s*/\s*b\s*/\s*a|a\s*/\s*k\s*/\s*a|f\s*/\s*k\s*/\s*a|t\s*/\s*a)\b\.?"
    r"|\b(?:dba|formerly|aka|fka|also known as|trading as)\b\s*:?)\s*",
    re.I,
)
MS_RE = re.compile(r"\bm\s*/\s*s\b\.?", re.I)
NULL_RE = re.compile(r"<\s*null\s*>|\bnull\b|\bn\s*/\s*a\b|\bnone\b|\bc\s*/\s*o\b\.?", re.I)
# postal boxes / sorting-office codes (PO Box, French BP / CS / Cedex) and the French "N°" number sign
POBOX_RE = re.compile(r"\b(?:p\.?\s*o\.?\s*box|post\s*box|bp|cs)\s*[.:#]?\s*\d+\b|\bcedex(?:\s*\d+)?\b", re.I)
NUMSIGN_RE = re.compile(r"\bn\s*[°º]", re.I)
DOMAIN_RE = re.compile(
    r"^(?:https?://)?(?:www\.)?([a-z0-9][a-z0-9\-]*)\.(?:com|net|org|co\.in|in|co|biz|info|us|fr|io)$"
)
NONALNUM_RE = re.compile(r"[^a-z0-9]+")
DIGITS_RE = re.compile(r"\d+")

# ---------------------------------------------------------------- name tables
# Canonical forms collapse spelling variants onto ONE short form. The direction is always
# long -> short: "technologies" / "technology" -> "tech" (never "tech" -> "technologies",
# which would turn "Tech Mahindra" into "technologies mahindra"). A short token is therefore
# never expanded into something it may not mean; only unambiguous long forms are shortened.
NAME_CANON = {
    "incorporated": "inc", "corporation": "corp", "company": "co", "cos": "co",
    "limited": "ltd", "private": "pvt", "centre": "center", "brothers": "bros",
    "and": "&", "et": "&", "sri": "shri", "shree": "shri", "international": "intl",
    "manufacturing": "mfg", "services": "svcs", "service": "svc", "associates": "assoc",
    "technologies": "tech", "technology": "tech", "hospital": "hosp",
    "etablissements": "ets", "etablissement": "ets", "etabl": "ets", "compagnie": "cie",
    "societe": "ste", "pvtltd": "pvt ltd", "corpn": "corp",
}
# Legal-form tokens. STRICT ones are never anything but a legal form, so they are removed
# from the core name wherever they appear (corrupted records shuffle tokens: "Federal LLC
# Minerals Star"). WEAK ones are also ordinary words or initials ("PC World", "Ag Supply",
# "SA Toys", "Co-op", "PA Impex"), so they are only removed in a legal *position*: at the end
# of the name, right after "&" ("Tiffany & Co"), or next to another legal token ("Co Ltd").
STRICT_LEGAL = {
    "inc", "corp", "ltd", "pvt", "llc", "llp", "plc", "pllc", "ltda", "sarl", "sas", "sasu",
    "eurl", "gmbh", "opc", "selarl", "scop", "gie", "snc", "sca", "scm", "sci",
}
WEAK_LEGAL = {"co", "lp", "pc", "pa", "sa", "ag", "bv", "nv", "ei", "cie"}
LEGAL = STRICT_LEGAL | WEAK_LEGAL
# Articles / connectives / honorifics: dropped from the core name only when at least two
# other tokens remain, so "The One", "La Poste", "El Lincoln" keep their identity, while
# "The Dent Diner" -> "dent diner". They are always kept in the full name.
NAME_STOP = {
    "the", "of", "&", "mr", "mrs", "ms", "dr", "smt", "messrs", "de", "du", "des", "la", "le",
    "les", "el", "los", "las", "d", "l", "a", "an", "en", "au", "aux", "for",
}
MIN_CORE = 2
MIN_CORE_COUNTRY = 3  # tokens that must remain before a trailing country token is treated as a tag
PAREN_RE = re.compile(r"\(([^()]*)\)")

# ------------------------------------------------------------- address tables
ADDR_CANON = {
    "street": "st", "str": "st", "road": "rd", "avenue": "ave", "avenu": "ave",
    "boulevard": "blvd", "boul": "blvd", "bld": "blvd", "drive": "dr", "court": "ct",
    "lane": "ln", "place": "pl", "circle": "cir", "highway": "hwy", "parkway": "pkwy",
    "terrace": "ter", "trail": "trl", "square": "sq", "north": "n", "south": "s", "east": "e",
    "west": "w", "northeast": "ne", "northwest": "nw", "southeast": "se", "southwest": "sw",
    "suite": "ste", "apartment": "apt", "appartement": "apt", "appt": "apt", "building": "bldg",
    "floor": "fl", "flr": "fl", "number": "no", "nr": "near", "opposite": "opp", "saint": "st",
    "sainte": "ste", "mount": "mt", "fort": "ft", "point": "pt", "route": "rte", "expressway": "expy",
    "freeway": "fwy", "crossing": "xing", "impasse": "imp", "allee": "all",
    "allees": "all", "chemin": "che", "cours": "crs", "quai": "qu", "residence": "res",
    "faubourg": "fbg", "chaussee": "chau", "hno": "no", "house": "h", "sector": "sec",
    "sect": "sec", "nagar": "ngr", "marg": "mg", "colony": "col", "district": "dist",
    "township": "twp", "junction": "jn", "extension": "extn", "ext": "extn", "gali": "gali",
    "bengaluru": "bangalore", "gurugram": "gurgaon", "calcutta": "kolkata", "bombay": "mumbai",
    "madras": "chennai", "centre": "center", "first": "1st", "second": "2nd", "third": "3rd",
    "bvld": "blvd", "bvd": "blvd", "etage": "fl", "ndeg": "no",
}
# One- and two-letter French street abbreviations are only unambiguous in France: "R K Puram"
# (an Indian locality) must not become "rue k puram", and "Q" / "CH" are ordinary tokens elsewhere.
ADDR_CANON_FR = {"r": "rue", "q": "qu", "ch": "che", "bd": "blvd", "av": "ave"}
FR_COUNTRY_TOKENS = {"france", "fr"}
ADDR_STOP = {
    "de", "du", "des", "la", "le", "les", "d", "l", "of", "the", "and", "&", "au", "aux",
    "no", "eme",
}
# canonical street-type / direction / unit words: never used as "key" tokens in combined blocking keys
ADDR_GENERIC = set(ADDR_CANON.values()) | set(ADDR_CANON_FR.values()) | {
    "rue", "city", "town", "county", "village", "unit", "apt", "ste", "fl", "bldg", "floor", "po",
    "box", "near", "opp", "road", "main", "cross", "center", "plot", "flat", "shop", "office",
    "phase", "block", "complex", "tower", "society", "market", "chowk", "bazar", "industrial",
    "estate", "area", "layout", "stage", "behind", "post", "village",
}

US_STATES = {
    "alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar", "california": "ca",
    "colorado": "co", "connecticut": "ct", "delaware": "de", "district of columbia": "dc",
    "florida": "fl", "georgia": "ga", "hawaii": "hi", "idaho": "id", "illinois": "il",
    "indiana": "in", "iowa": "ia", "kansas": "ks", "kentucky": "ky", "louisiana": "la",
    "maine": "me", "maryland": "md", "massachusetts": "ma", "michigan": "mi", "minnesota": "mn",
    "mississippi": "ms", "missouri": "mo", "montana": "mt", "nebraska": "ne", "nevada": "nv",
    "new hampshire": "nh", "new jersey": "nj", "new mexico": "nm", "new york": "ny",
    "north carolina": "nc", "north dakota": "nd", "ohio": "oh", "oklahoma": "ok", "oregon": "or",
    "pennsylvania": "pa", "rhode island": "ri", "south carolina": "sc", "south dakota": "sd",
    "tennessee": "tn", "texas": "tx", "utah": "ut", "vermont": "vt", "virginia": "va",
    "washington": "wa", "west virginia": "wv", "wisconsin": "wi", "wyoming": "wy",
    "puerto rico": "pr",
}
IN_STATES = {
    "andhra pradesh": "ap", "arunachal pradesh": "ar", "assam": "as", "bihar": "br",
    "chhattisgarh": "cg", "goa": "ga", "gujarat": "gj", "haryana": "hr", "himachal pradesh": "hp",
    "jharkhand": "jh", "karnataka": "ka", "kerala": "kl", "madhya pradesh": "mp",
    "maharashtra": "mh", "manipur": "mn", "meghalaya": "ml", "mizoram": "mz", "nagaland": "nl",
    "odisha": "od", "orissa": "od", "punjab": "pb", "rajasthan": "rj", "sikkim": "sk",
    "tamil nadu": "tn", "telangana": "tg", "tripura": "tr", "uttar pradesh": "up",
    "uttarakhand": "uk", "uttaranchal": "uk", "west bengal": "wb", "delhi": "dl",
    "nct of delhi": "dl", "jammu and kashmir": "jk", "jammu & kashmir": "jk", "chandigarh": "ch",
    "puducherry": "py", "pondicherry": "py", "ladakh": "la", "dadra and nagar haveli": "dn",
    "daman and diu": "dd", "andaman and nicobar islands": "an", "lakshadweep": "ld",
}

FR_REGIONS = {
    "ara": ["auvergne rhone alpes", "ain", "allier", "ardeche", "cantal", "drome", "isere", "loire",
            "haute loire", "puy de dome", "rhone", "savoie", "haute savoie"],
    "bfc": ["bourgogne franche comte", "cote d or", "doubs", "jura", "nievre", "haute saone",
            "saone et loire", "yonne", "territoire de belfort"],
    "bre": ["bretagne", "cotes d armor", "finistere", "ille et vilaine", "morbihan"],
    "cvl": ["centre val de loire", "cher", "eure et loir", "indre", "indre et loire", "loir et cher", "loiret"],
    "cor": ["corse", "corse du sud", "haute corse"],
    "ges": ["grand est", "ardennes", "aube", "marne", "haute marne", "meurthe et moselle", "meuse",
            "moselle", "bas rhin", "haut rhin", "vosges"],
    "hdf": ["hauts de france", "aisne", "nord", "oise", "pas de calais", "somme"],
    "idf": ["ile de france", "seine et marne", "yvelines", "essonne", "hauts de seine",
            "seine saint denis", "val de marne", "val d oise"],
    "nor": ["normandie", "calvados", "eure", "manche", "orne", "seine maritime"],
    "naq": ["nouvelle aquitaine", "charente", "charente maritime", "correze", "creuse", "dordogne",
            "gironde", "landes", "lot et garonne", "pyrenees atlantiques", "deux sevres", "vienne",
            "haute vienne"],
    "occ": ["occitanie", "ariege", "aude", "aveyron", "gard", "haute garonne", "gers", "herault",
            "lot", "lozere", "hautes pyrenees", "pyrenees orientales", "tarn", "tarn et garonne"],
    "pdl": ["pays de la loire", "loire atlantique", "maine et loire", "mayenne", "sarthe", "vendee"],
    "pac": ["provence alpes cote d azur", "paca", "alpes de haute provence", "hautes alpes",
            "alpes maritimes", "bouches du rhone", "var", "vaucluse"],
}
FR_ADMIN = {name: "fr" + code for code, names in FR_REGIONS.items() for name in names}

# Learned Indic lexicons (populated by set_lexicon in each worker process).
_NAME_LEX = {}
_ADDR_LEX = {}


def set_lexicon(lex):
    global _NAME_LEX, _ADDR_LEX
    if lex:
        _NAME_LEX = lex.get("name", {})
        _ADDR_LEX = lex.get("addr", {})


def indic_key(tok):
    """Canonical lookup key for an Indic-script token/component."""
    tok = ZW_RE.sub("", unicodedata.normalize("NFC", tok))
    return "".join(ch for ch in tok if ch.isalnum() or unicodedata.category(ch)[0] == "M").strip()


def to_ascii(s):
    return unidecode(ZW_RE.sub("", s)).lower()


def latin_tokens(s):
    return [t for t in NONALNUM_RE.split(s) if t]


def _translit_name(raw):
    """Replace Indic tokens with their learned Latin equivalents (fallback: unidecode)."""
    out = []
    for tok in raw.split():
        if INDIC_RE.search(tok):
            lat = _NAME_LEX.get(indic_key(tok))
            out.append(lat if lat is not None else to_ascii(tok))
        else:
            out.append(tok)
    return " ".join(out)


def _country_tag(text, country_toks):
    """True when a parenthesised chunk is the record's own country label, e.g. "(India)",
    including OCR-style corruptions such as "(lndia)" / "(1ndia)" (ratio >= 80)."""
    if not country_toks:
        return False
    toks = latin_tokens(text)
    if not toks:
        return False
    a, b = "".join(toks), "".join(country_toks)
    return a == b or (len(a) >= 3 and fuzz.ratio(a, b) >= 80)


def _merge_initials(toks):
    """Runs of single-letter tokens (optionally glued by "&") become one token:
    "j p morgan" -> "jp morgan", "d & l metro" -> "dl metro", "l a fitness" -> "la fitness".
    A lone single letter next to a longer token is left alone ("orelee s barbershop")."""
    out, run = [], []

    def flush():
        if len(run) >= 2:
            out.append("".join(run))
        else:
            out.extend(run)
        run.clear()

    for t in toks:
        if len(t) == 1 and t != "&":
            run.append(t)
        elif t == "&" and run:
            continue  # "&" between initials is glue; it is dropped from the core anyway
        else:
            flush()
            out.append(t)
    flush()
    return out


def _strip_legal(toks):
    """Removes legal-form tokens from the core name (see STRICT_LEGAL / WEAK_LEGAL).
    Returns (remaining tokens, removed legal tokens)."""
    legal, keep = [], []
    n = len(toks)
    for i, t in enumerate(toks):
        if t in STRICT_LEGAL:
            legal.append(t)
            continue
        if t in WEAK_LEGAL:
            trailing = all(x in LEGAL or x in NAME_STOP for x in toks[i + 1:])
            after_amp = i > 0 and toks[i - 1] == "&"
            next_legal = (i + 1 < n and toks[i + 1] in LEGAL) or (i > 0 and toks[i - 1] in LEGAL)
            if trailing or after_amp or next_legal:
                legal.append(t)
                continue
        keep.append(t)
    return keep, legal


def _strip_trailing_country(toks, country_toks):
    """"Tata Consultancy Services India" -> drop the trailing country token, but only when
    at least MIN_CORE_COUNTRY other content tokens remain: "Air India", "Reliance India" and
    "Toys R Us" keep it. Being conservative is cheap: an extra trailing token on one side is
    handled by the token-set / extra-token features, whereas a deleted token is gone."""
    k = len(country_toks)
    if not k or len(toks) < k + MIN_CORE_COUNTRY:
        return toks
    if toks[-k:] == list(country_toks):
        rest = toks[:-k]
        if sum(1 for t in rest if t not in NAME_STOP) >= MIN_CORE_COUNTRY:
            return rest
    return toks


def _name_tokens(part, country_toks=()):
    """Normalise one alias part of a name -> (full tokens, core tokens, domain stems, legal tokens).

    `country_toks` are the tokens of the record's own country label. A parenthesised country tag
    "(India)" carries no identity and is removed everywhere; a bare country token is kept in the
    full name and only dropped from the core when it is a trailing location tag (see
    _strip_trailing_country) -- "Air India" is not "Air".
    """
    s = to_ascii(part)
    s = PAREN_RE.sub(lambda m: " " if _country_tag(m.group(1), country_toks) else " " + m.group(1) + " ", s)
    s = s.replace("&", " & ").replace("+", " & ")
    domains = []
    toks = []
    for raw in s.split():
        m = DOMAIN_RE.match(raw.strip(".,;:()[]{}<>*#-_\"'"))
        if m:
            domains.append(m.group(1).replace("-", ""))
            continue
        raw = raw.replace(".", "").replace("'", "").replace("\u2019", "")
        for t in latin_tokens(raw) if raw != "&" else ["&"]:
            toks.extend(NAME_CANON.get(t, t).split())
    toks = _merge_initials(toks)
    full = [t for t in toks if t != "&"]
    core, legal = _strip_legal(toks)
    core = _strip_trailing_country(core, tuple(country_toks))
    content = [t for t in core if t not in NAME_STOP]
    if len(content) >= MIN_CORE:
        core = content
    else:
        core = [t for t in core if t != "&"]
    core = list(dict.fromkeys(core))
    legal = [t for t in legal if t != "cie"]
    return full, core, domains, legal


def norm_name(raw, country=None):
    """Returns dict with normalised name fields."""
    if raw is None:
        raw = ""
    country_toks = tuple(latin_tokens(to_ascii(country))) if country else ()
    is_indic = bool(INDIC_RE.search(raw))
    s = ID_RE.sub(" ", raw)
    s = MS_RE.sub(" ", s)
    if is_indic:
        s = _translit_name(s)
    parts = [p for p in ALIAS_RE.split(s) if p and p.strip()]
    if not parts:
        parts = [""]
    fulls, cores, compacts, all_domains, legals = [], [], [], [], []
    for p in parts:
        full, core, domains, legal = _name_tokens(p, country_toks)
        if full or core:
            fulls.append(" ".join(full))
            cores.append(" ".join(core))
            if core:
                compacts.append("".join(core))
            if full:
                compacts.append("".join(full))
        all_domains.extend(domains)
        legals.extend(legal)
    compacts.extend(all_domains)
    # flatten: primary string uses every part (aliases are alternative names of the same entity)
    all_core, seen = [], set()
    for c in cores:
        for t in c.split():
            if t not in seen:
                seen.add(t)
                all_core.append(t)
    return {
        "n_full": " ".join(fulls),
        "n_core": " ".join(all_core),
        "n_parts": "|".join(c for c in cores if c),
        "n_compact": "|".join(dict.fromkeys(c for c in compacts if c)),
        "n_domain": "|".join(all_domains),
        "n_legal": " ".join(sorted(set(legals))),
        "f_indic": is_indic,
        "f_alias": len(parts) > 1,
    }


def _translit_addr_component(comp):
    if INDIC_RE.search(comp):
        lat = _ADDR_LEX.get(indic_key(comp))
        if lat is not None:
            return lat
        return to_ascii(comp)
    return comp


UNIT_WORDS = {"unit", "apt", "ste", "fl", "bldg", "room", "rm", "po", "box"}
STATE_CODES = set(US_STATES.values()) | set(IN_STATES.values()) | set(FR_ADMIN.values())


def norm_addr(raw, country=None):
    """Returns dict: a_norm (tokens), a_comp (components joined by ','), a_num (digit groups),
    a_hn (house number: first digit group of the street line), a_street (non-numeric tokens of
    the street line) and a_key (distinctive non-numeric tokens used in combined blocking keys).

    `country` gates the ambiguous French abbreviations (ADDR_CANON_FR)."""
    if raw is None:
        raw = ""
    canon = ADDR_CANON
    if country and to_ascii(country).strip() in FR_COUNTRY_TOKENS:
        canon = {**ADDR_CANON, **ADDR_CANON_FR}
    s = NULL_RE.sub(" ", raw)
    s = POBOX_RE.sub(" ", s)
    s = NUMSIGN_RE.sub(" ", s)
    comps = []
    for comp in s.split(","):
        comp = _translit_addr_component(comp.strip())
        c = to_ascii(comp).replace("&", " and ").replace("'", " ").replace("’", " ")
        toks = latin_tokens(c)
        if not toks:
            continue
        joined = " ".join(toks)
        st = US_STATES.get(joined) or IN_STATES.get(joined) or FR_ADMIN.get(joined)
        if st:
            comps.append([st])
            continue
        out = []
        for t in toks:
            if t.isdigit():
                t = t.lstrip("0") or "0"
            t = canon.get(t, t)
            if t in ADDR_STOP and len(toks) > 1:
                # articles are dropped inside a component, but a component that IS the token is
                # kept: "DE" (Delaware) and "LA" (Louisiana) are state codes, not French articles
                continue
            out.append(t)
        if out:
            comps.append(out)
    flat = [t for c in comps for t in c]
    nums = DIGITS_RE.findall(" ".join(flat))
    numbered = [c for c in comps if any(ch.isdigit() for t in c for ch in t)]
    street = next((c for c in numbered if c[0] not in UNIT_WORDS), numbered[0] if numbered else [])
    hn = next((m.group(0) for t in street for m in [DIGITS_RE.search(t)] if m), "")
    street_words = [t for t in street if not any(ch.isdigit() for ch in t)]
    key = []
    for t in street_words + [t for c in comps if c is not street for t in c]:
        if len(t) >= 3 and t not in ADDR_GENERIC and t not in STATE_CODES \
                and not any(ch.isdigit() for ch in t) and t not in key:
            key.append(t)
    return {
        "a_norm": " ".join(flat),
        "a_comp": ",".join(" ".join(c) for c in comps),
        "a_num": " ".join(dict.fromkeys(nums)),
        "a_hn": hn,
        "a_street": " ".join(street_words),
        "a_key": " ".join(key[:6]),
    }


NGRAM = 4


def char_ngrams(s, n=NGRAM):
    """Padded character n-grams of one compact string: "#dent", "dentd", ..., "iner#". A string
    shorter than n gives the padded string itself, so very short names still get one feature."""
    s = "#" + s + "#"
    if len(s) <= n:
        return [s]
    return [s[i:i + n] for i in range(len(s) - n + 1)]


def blocking_features(n, a):
    """Space separated blocking features for one record (four blocks, each a retrieval channel):

    name block   : n:<core token>, p:<order-free token pair>, k:<compact-name prefix>
    nchar block  : g:<char 4-gram of the compact core name / alias / domain stem> (typo- and
                   transliteration-tolerant: "co1lege" still shares most grams with "college")
    address block: a:<token>, b:<adjacent bigram>, h:<house number>_<key token>
    cross block  : c:<name token>_<key address token>  (separates namesakes in different places)
    """
    nf, af, cf, gf = [], [], [], []
    for c in n["n_compact"].split("|"):
        if c:
            gf.extend("g:" + g for g in char_ngrams(c))
    for part in n["n_parts"].split("|"):
        toks = part.split()
        for t in toks:
            nf.append("n:" + t)
        head = toks[:6]
        for i in range(len(head)):
            for j in range(i + 1, len(head)):
                x, y = (head[i], head[j]) if head[i] < head[j] else (head[j], head[i])
                nf.append("p:" + x + "_" + y)
    for c in n["n_compact"].split("|"):
        if len(c) >= 5:
            nf.append("k:" + c[:8])
    for comp in a["a_comp"].split(","):
        toks = comp.split()
        for t in toks:
            af.append("a:" + t)
        for i in range(len(toks) - 1):
            af.append("b:" + toks[i] + "_" + toks[i + 1])
    key = a["a_key"].split()[:5]
    if a["a_hn"]:
        for k in key:
            af.append("h:" + a["a_hn"] + "_" + k)
    for t in n["n_core"].split()[:3]:
        for k in key:
            cf.append("c:" + t + "_" + k)
    return " ".join(dict.fromkeys(nf)), " ".join(dict.fromkeys(af)), " ".join(dict.fromkeys(cf)), " ".join(dict.fromkeys(gf))


def normalize_record(name, addr, country=None):
    n = norm_name(name, country)
    a = norm_addr(addr, country)
    fn, fa, fc, fg = blocking_features(n, a)
    n.update(a)
    n["feat_name"] = fn
    n["feat_addr"] = fa
    n["feat_cross"] = fc
    n["feat_nchar"] = fg
    return n


In [ ]:
%%writefile /kaggle/working/er/src/phonetic.py
"""Script-independent phonetic skeleton of a Latin token.

Used as a fallback for Indic-script names whose words are missing from the learned lexicon:
unidecode("मार्केटिंग") = "maarkettinga" and "marketing" both reduce to "mrktng".
"""
import re

_RULES = [
    (re.compile(r"tion"), "sn"), (re.compile(r"ture"), "cr"), (re.compile(r"c(?=[eiy])"), "s"),
    (re.compile(r"x"), "ks"), (re.compile(r"ph"), "f"), (re.compile(r"sh"), "s"),
    (re.compile(r"([bcdgjkptr])h"), r"\1"),
]
_VOWEL_RE = re.compile(r"[aeiouy]")
_REPEAT_RE = re.compile(r"(.)\1+")
_SUBST = str.maketrans({"c": "k", "q": "k", "w": "b", "v": "b", "z": "j", "g": "j"})


def skeleton(tok):
    """First letter + consonants: aspirates merged, similar consonants unified, repeats collapsed."""
    t = tok.lower()
    if len(t) < 3 or not t.isalpha():
        return t
    for rx, rep in _RULES:
        t = rx.sub(rep, t)
    t = t.translate(_SUBST)
    s = t[0] + _VOWEL_RE.sub("", t[1:])
    return _REPEAT_RE.sub(r"\1", s)


In [ ]:
%%writefile /kaggle/working/er/src/metrics.py
"""Macro F_0.5 exactly as defined by the challenge (singletons included)."""
import numpy as np
import polars as pl


def macro_f05(pred, truth, beta=0.5):
    """pred/truth: DataFrames (s1 id column 's', list[str] column 'm'); rows of truth define the eval set.

    Per Source 1 entity: empty truth & empty pred -> 1, empty truth & non-empty pred -> 0,
    otherwise F_beta of the predicted vs. true id sets. Returns (macro F, stats dict).
    """
    b2 = beta * beta
    j = truth.rename({"m": "true"}).join(pred.rename({"m": "pred"}), on="s", how="left").with_columns(
        pl.col("pred").fill_null(pl.lit([], dtype=pl.List(pl.String)))
    )
    j = j.with_columns(
        pl.col("true").list.len().alias("nt"),
        pl.col("pred").list.len().alias("np"),
        pl.col("true").list.set_intersection("pred").list.len().alias("tp"),
    )
    nt = j["nt"].to_numpy().astype(float)
    npd = j["np"].to_numpy().astype(float)
    tp = j["tp"].to_numpy().astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(npd > 0, tp / npd, 0.0)
        r = np.where(nt > 0, tp / nt, 0.0)
        f = np.where((p + r) > 0, (1 + b2) * p * r / (b2 * p + r), 0.0)
    f = np.where((nt == 0) & (npd == 0), 1.0, f)
    f = np.where((nt == 0) & (npd > 0), 0.0, f)
    stats = {
        "n": len(f),
        "macro_f05": float(f.mean()),
        "micro_precision": float(tp.sum() / max(npd.sum(), 1)),
        "micro_recall": float(tp.sum() / max(nt.sum(), 1)),
        "singleton_acc": float(((nt == 0) & (npd == 0)).sum() / max((nt == 0).sum(), 1)),
    }
    return stats["macro_f05"], stats


In [ ]:
%%writefile /kaggle/working/er/src/folds.py
"""Step 0: validation folds, assigned BEFORE any label-derived preprocessing runs.

Why this exists (leakage + realism):
* The Indic-script lexicon (build_lexicon.py) is learned from the training ground truth. It
  used to be learned from *all* of it and then applied to every record, so the validation
  folds were normalised with a lexicon that had seen their own labels. Folds therefore have
  to be known first, so that each fold can be normalised with a lexicon learned from the
  other folds only.
* 38 % of Source 1 records share their exact name with another Source 1 record of the same
  country (chains, franchises, namesakes). Random per-entity folds put "Dent Diner #1" in
  training and "Dent Diner #2" in validation; the model then learns the namesake's
  address pattern from the training copy. Folds are therefore assigned per *name group*
  (normalised core name + country): all namesakes share a fold.
* Matched Source 2/3 records inherit the fold of their Source 1 entity. Unmatched records
  ("decoys") get fold -1 here; train.py assigns them the fold of their best blocking
  candidate so that as many candidate pairs as possible are within one fold.

Schemes (--scheme):
  group   : K folds by hashed name group (default, the honest i.i.d. estimate)
  country : one fold per country label (leave-one-country-out; simulates the unseen
            country of the test split -- France never appears in training)

Writes <work>/train/folds.parquet: entity_id, fold (Int8), group (UInt64 hash).
"""
import os

import polars as pl

import textnorm
from common import Stage, base_args, log, read_tsv, source_path, split_dir


def name_group_key(names, countries):
    """Namesake key: sorted core-name tokens + country. Computed without the Indic lexicon
    (Source 1 names are Latin script; for Source 2/3 the unidecode fallback is used), so it
    depends on no label-derived state."""
    keys = []
    for n, c in zip(names, countries):
        core = textnorm.norm_name(n, c)["n_core"].split()
        keys.append(" ".join(sorted(core)) + "|" + (c or ""))
    return keys


def main():
    ap = base_args(__doc__)
    ap.add_argument("--folds", type=int, default=4)
    ap.add_argument("--scheme", choices=["group", "country"], default="group")
    ap.add_argument("--seed", type=int, default=7)
    args = ap.parse_args()
    with Stage(args.work_dir, "folds"):
        run(args)


def run(args):
    textnorm.set_lexicon({})
    out_dir = split_dir(args.work_dir, "train")

    s1 = read_tsv(source_path(args.data_dir, "train", "source1"))
    s1 = s1.with_columns(pl.Series("group_key", name_group_key(s1["business_name"].to_list(), s1["country"].to_list())))
    if args.scheme == "country":
        countries = sorted(s1["country"].fill_null("").unique().to_list())
        fold_expr = pl.col("country").fill_null("").replace_strict(countries, list(range(len(countries)))).cast(pl.Int8)
        log("country folds:", dict(enumerate(countries)))
    else:
        fold_expr = (pl.col("group_key").hash(seed=args.seed) % args.folds).cast(pl.Int8)
    s1 = s1.with_columns(fold_expr.alias("fold"), pl.col("group_key").hash(seed=args.seed).alias("group"))
    n_groups = s1["group_key"].n_unique()
    log(f"S1 {s1.height} records in {n_groups} name groups; fold sizes {s1['fold'].value_counts().sort('fold')['count'].to_list()}")

    gt = read_tsv(os.path.join(args.data_dir, "train", "train_ground_truth.tsv"))
    matched = (
        gt.with_columns(pl.col("matched_entity_ids").str.split(","))
        .explode("matched_entity_ids")
        .drop_nulls("matched_entity_ids")
        .join(s1.select(pl.col("entity_id").alias("source1_entity_id"), "fold", "group"), on="source1_entity_id")
        .select(pl.col("matched_entity_ids").alias("entity_id"), "fold", "group")
    )
    others = pl.concat([read_tsv(source_path(args.data_dir, "train", s)).select("entity_id") for s in ("source2", "source3")])
    decoys = others.join(matched.select("entity_id"), on="entity_id", how="anti").with_columns(
        pl.lit(-1, pl.Int8).alias("fold"), pl.lit(0, pl.UInt64).alias("group"))
    folds = pl.concat([s1.select("entity_id", "fold", "group"), matched, decoys])
    folds.write_parquet(os.path.join(out_dir, "folds.parquet"))
    log(f"wrote folds.parquet: {s1.height} S1, {matched.height} matched S2/S3, {decoys.height} decoys (fold -1)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/er/src/build_lexicon.py
"""Learns an Indic-script -> Latin lexicon from the *training* ground truth.

Source 2/3 names are sometimes written in an Indic script (Devanagari, Telugu, ...)
while their Source 1 counterpart is in Latin script with the same number of tokens,
so tokens can be aligned by position. Addresses carry Indic-script state/city
components which are aligned against the Source 1 address components by
co-occurrence. Only the provided training data is used.

Leakage fix: the lexicon is label-derived preprocessing, so it must be fit on the training
folds only. This script therefore writes one lexicon per validation fold, learned from the
pairs of the *other* folds ("fold_k"), plus "all" (every training pair) which is the one used
for the test split. prepare.py normalises each training record with the lexicon of its own
fold, so no validation record is ever transliterated with knowledge of its own label.
Requires <work>/train/folds.parquet (folds.py).
"""
import collections
import json
import os

import polars as pl

from common import Stage, base_args, log, read_tsv, source_path, split_dir
from textnorm import INDIC_RE, indic_key, latin_tokens, to_ascii


def count_alignments(j):
    """Token / component co-occurrence counts of the Indic-script pairs in `j`.
    Returns per-pair-fold counters so that leave-one-fold-out lexicons are a subtraction."""
    name_cnt = collections.defaultdict(collections.Counter)
    addr_cnt = collections.defaultdict(collections.Counter)
    for name, addr, n1, a1 in j.select("business_name", "business_address", "n1", "a1").iter_rows():
        if name and INDIC_RE.search(name):
            tt, st = name.split(), n1.split()
            if len(tt) == len(st):
                for u, v in zip(tt, st):
                    if INDIC_RE.search(u):
                        lv = latin_tokens(to_ascii(v))
                        if len(lv) == 1:
                            name_cnt[indic_key(u)][lv[0]] += 1
        if addr and a1 and INDIC_RE.search(addr):
            s1_comps = {" ".join(latin_tokens(to_ascii(c))) for c in a1.split(",")}
            s1_comps.discard("")
            for c in addr.split(","):
                if INDIC_RE.search(c):
                    k = indic_key(c)
                    addr_cnt[k]["__n__"] += 1
                    for v in s1_comps:
                        addr_cnt[k][v] += 1
    return name_cnt, addr_cnt


def merge_counts(parts):
    name_cnt = collections.defaultdict(collections.Counter)
    addr_cnt = collections.defaultdict(collections.Counter)
    for nc, ac in parts:
        for k, c in nc.items():
            name_cnt[k].update(c)
        for k, c in ac.items():
            addr_cnt[k].update(c)
    return name_cnt, addr_cnt


def to_lexicon(name_cnt, addr_cnt, min_count):
    name_lex = {}
    for u, cnt in name_cnt.items():
        v, c = cnt.most_common(1)[0]
        if c >= min_count and c / sum(cnt.values()) >= 0.5:
            name_lex[u] = v
    addr_lex = {}
    for u, cnt in addr_cnt.items():
        cnt = cnt.copy()
        n = cnt.pop("__n__", 0)
        if not cnt or n == 0:
            continue
        v, c = cnt.most_common(1)[0]
        if c >= min_count and c / n >= 0.5:
            addr_lex[u] = v
    return {"name": name_lex, "addr": addr_lex}


def main():
    ap = base_args(__doc__)
    ap.add_argument("--min-count", type=int, default=2)
    args = ap.parse_args()
    with Stage(args.work_dir, "lexicon"):
        run(args)


def run(args):

    s1 = read_tsv(source_path(args.data_dir, "train", "source1"))
    others = pl.concat([read_tsv(source_path(args.data_dir, "train", s)) for s in ("source2", "source3")])
    gt = read_tsv(os.path.join(args.data_dir, "train", "train_ground_truth.tsv"))
    pairs = (
        gt.with_columns(pl.col("matched_entity_ids").str.split(","))
        .explode("matched_entity_ids")
        .drop_nulls("matched_entity_ids")
        .rename({"source1_entity_id": "s1", "matched_entity_ids": "t"})
    )
    folds = pl.read_parquet(os.path.join(split_dir(args.work_dir, "train"), "folds.parquet"), columns=["entity_id", "fold"])
    indic = others.filter(
        pl.col("business_name").str.contains(INDIC_RE.pattern)
        | pl.col("business_address").fill_null("").str.contains(INDIC_RE.pattern)
    )
    j = indic.join(pairs, left_on="entity_id", right_on="t").join(
        s1.select(pl.col("entity_id").alias("s1"), pl.col("business_name").alias("n1"),
                  pl.col("business_address").alias("a1")),
        on="s1",
    ).join(folds.rename({"entity_id": "s1"}), on="s1", how="left").with_columns(pl.col("fold").fill_null(-1))
    log("indic-script pairs", j.height)

    # counts per fold of the Source 1 entity; the lexicon for validation fold k uses every fold but k
    fold_ids = sorted(j["fold"].unique().to_list())
    per_fold = {k: count_alignments(j.filter(pl.col("fold") == k)) for k in fold_ids}
    lexicons = {"all": to_lexicon(*merge_counts(per_fold.values()), args.min_count)}
    for k in fold_ids:
        if k < 0:
            continue
        lexicons[f"fold_{k}"] = to_lexicon(*merge_counts(v for kk, v in per_fold.items() if kk != k), args.min_count)
    for name, lex in lexicons.items():
        log(f"lexicon {name}: name {len(lex['name'])} entries, address {len(lex['addr'])} entries")
    os.makedirs(args.work_dir, exist_ok=True)
    with open(os.path.join(args.work_dir, "lexicon.json"), "w", encoding="utf-8") as f:
        json.dump(lexicons, f, ensure_ascii=False)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/er/src/prepare.py
"""Step 1: normalise every record of a split (train or test) in parallel.

Writes <work>/<split>/records.parquet with one row per record of all three sources:
  rid (global row id), entity_id, src (1/2/3), country, raw fields, normalised fields,
  the blocking-feature strings and (train only) the validation fold.

Leakage fix: a training record is transliterated with the Indic lexicon learned WITHOUT
its own fold ("fold_k" in lexicon.json); decoys (fold -1) and every test record use the
lexicon learned from all training pairs ("all"), exactly as at inference time.
"""
import json
import multiprocessing as mp
import os
import shutil

import polars as pl

import textnorm
from common import Stage, SOURCES, base_args, left_join_ordered, log, n_workers, read_tsv, source_path, split_dir

CHUNK = 50_000


_LEX = {}


def _init(lexicons):
    _LEX.update(lexicons)


def _work(args):
    names, addrs, countries, lex_keys = args
    rows = []
    current = None
    for n, a, c, k in zip(names, addrs, countries, lex_keys):
        if k != current:  # records are grouped by lexicon key, so this switches rarely
            textnorm.set_lexicon(_LEX.get(k) or _LEX["all"])
            current = k
        rows.append(textnorm.normalize_record(n, a, c))
    return {k: [r[k] for r in rows] for k in rows[0]}


def main():
    ap = base_args(__doc__)
    ap.add_argument("--split", required=True, choices=["train", "test"])
    args = ap.parse_args()
    with Stage(args.work_dir, "prepare_" + args.split):
        run(args)


def run(args):
    out_dir = split_dir(args.work_dir, args.split)
    with open(os.path.join(args.work_dir, "lexicon.json"), encoding="utf-8") as f:
        lexicons = json.load(f)
    if "all" not in lexicons:  # lexicon written by the old single-lexicon build_lexicon.py
        lexicons = {"all": lexicons}

    frames = []
    for i, s in enumerate(SOURCES):
        df = read_tsv(source_path(args.data_dir, args.split, s)).with_columns(pl.lit(i + 1, pl.Int8).alias("src"))
        frames.append(df)
    df = pl.concat(frames).with_row_index("rid")
    if args.split == "train":
        folds = pl.read_parquet(os.path.join(out_dir, "folds.parquet"), columns=["entity_id", "fold", "group"])
        df = left_join_ordered(df, folds, "entity_id").with_columns(
            pl.col("fold").fill_null(-1).cast(pl.Int8), pl.col("group").fill_null(0))
        lex_key = pl.when(pl.col("fold") >= 0).then("fold_" + pl.col("fold").cast(pl.String)).otherwise(pl.lit("all"))
    else:
        lex_key = pl.lit("all")
    df = df.with_columns(lex_key.alias("lex_key"))
    log(f"{args.split}: {df.height} records; lexicon keys {df['lex_key'].value_counts().sort('lex_key').rows()}")

    n_rows = df.height
    cols = df.select("business_name", "business_address", "country", "lex_key")

    def jobs():
        # one chunk at a time: materialising all 12.5M rows as Python lists costs several GB
        for i in range(0, n_rows, CHUNK):
            c = cols.slice(i, CHUNK)
            yield (c["business_name"].to_list(), c["business_address"].to_list(), c["country"].to_list(), c["lex_key"].to_list())

    # each normalised chunk is joined to its raw rows and written straight to disk; the parts are then
    # merged with a streaming write, so memory stays at ~one chunk instead of the whole table (the
    # blocking-feature strings of 12.5M records are ~10 GB in RAM)
    parts_dir = os.path.join(out_dir, "records_parts")
    shutil.rmtree(parts_dir, ignore_errors=True)
    os.makedirs(parts_dir)
    raw = df.drop("lex_key")
    with mp.get_context("spawn").Pool(n_workers(), initializer=_init, initargs=(lexicons,)) as pool:
        for k, res in enumerate(pool.imap(_work, jobs(), chunksize=1)):
            part = pl.concat([raw.slice(k * CHUNK, CHUNK), pl.DataFrame(res)], how="horizontal")
            part.write_parquet(os.path.join(parts_dir, f"part_{k:05d}.parquet"))
            if k % 40 == 0:
                log(f"normalised {min((k + 1) * CHUNK, n_rows)}/{n_rows}")
    del raw, df
    out_path = os.path.join(out_dir, "records.parquet")
    pl.scan_parquet(os.path.join(parts_dir, "part_*.parquet")).sink_parquet(out_path)
    shutil.rmtree(parts_dir, ignore_errors=True)
    df = pl.read_parquet(out_path, columns=["rid"])
    log("wrote", os.path.join(out_dir, "records.parquet"), df.shape)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/er/src/blocking.py
"""Step 2: multi-channel candidate generation (blocking).

For every country label (open set, whatever values appear in the data) TF-IDF vectors are built
over the blocking features of textnorm.blocking_features, one *block* per feature family:

  name  : n:<token>, p:<token pair> (order-free), k:<8-char compact-name prefix>
  nchar : g:<char 4-gram of the compact name>          (typo / transliteration tolerant)
  addr  : a:<token>, b:<adjacent bigram inside an address component>, h:<house number>_<key token>
  cross : c:<name token>_<key address token>, i.e. name x place

IDF is computed on Source 1 (the index side); every block is L2-normalised, so a block's
retrieval score is the cosine between the two records restricted to "index" features whose
Source 1 document frequency is <= --df-cap (very common features cost a lot and carry little
evidence, but they still count in the vector norms).

Retrieval channels (each one keeps its own top-k per Source 2/3 record; the candidate set is
the UNION of all channels):

  combined : cos_name + cos_addr + cos_cross               (the single channel of the previous version)
  name     : cos_name alone                                 (address-less / address-conflicting records)
  nchar    : cos over character 4-grams of the compact name (typos, concatenations, transliterations)
  addr     : cos_addr alone                                 (name in another script or heavily corrupted)
  cross    : cos_cross alone
  rare     : sum of idf over shared *rare* name features (Source 1 df <= --rare-df): a shared rare
             token proposes the pair whatever the rest of the name looks like
  hn       : shared h:<house number>_<key token> keys (numeric / address blocking)
  rev      : bidirectional retrieval: every Source 1 record retrieves its top-k Source 2/3 records
             on the combined score (a record crowded out of a namesake's top-k is still proposed
             by the entity that wants it)

--channels chooses the channels and their k, e.g. "combined=5,nchar=3,addr=2,rev=2" or
"combined=3,name=2,nchar=2,addr=1,cross=1,rare=2,hn=2,rev=2"; "combined=3" reproduces the previous
single-channel candidate set exactly (up to the extra columns); the default is "combined=5,nchar=3,addr=2,rev=2".

For every retrieved pair the exact per-block cosines (all shared features, no df cap) are computed;
`score` is the exact combined cosine and `rank` the pair's rank by that score within its Source 2/3
record's candidate list. The per-channel ranks (r_<channel>, -1 = not proposed by that channel) and
the number of channels that proposed the pair (n_ch, "retrieval agreement") are kept as features.

Candidate audit (training split, labels used for measurement only): <work>/train/blocking_recall.json
holds the true-pair recall of the union and of every channel, the share of Source 1 entities whose
complete record set was retrieved, the candidate *oracle* macro F0.5 (a perfect classifier on this
candidate set: the ceiling any matcher can reach) and the mean candidates per record.

Output: <work>/<split>/candidates_raw.parquet
  t_rid, s_rid, score, rank, cos_name, cos_addr, cos_cross, cos_nchar, n_ch, r_<channel>...
"""
import json
import multiprocessing as mp
import os
import shutil

import numpy as np
import polars as pl
import scipy.sparse as sp

from common import Stage, base_args, log, n_workers, split_dir

_G = {}


# ------------------------------------------------------------------ features
BLOCKS = ("feat_name", "feat_addr", "feat_cross", "feat_nchar")
BLOCK_NAMES = ("name", "addr", "cross", "nchar")
COMBINED = ("name", "addr", "cross")  # blocks whose cosines add up to the combined score
CHANNELS = ("combined", "name", "nchar", "addr", "cross", "rare", "hn", "rev")
# default: the configuration validated end to end on the shifted hold-out (docs/REPORT.md 14.4);
# "combined=5" is the cheaper fallback, all channels the most complete candidate set
DEFAULT_CHANNELS = "combined=5,nchar=3,addr=2,rev=2"
# minimum retrieval score per channel (cosines for the tf-idf channels; any shared feature for rare / hn)
MIN_SCORE = {"combined": 0.1, "name": 0.2, "nchar": 0.3, "addr": 0.3, "cross": 0.2, "rare": 1e-6, "hn": 0.5, "rev": 0.1}


def parse_channels(spec):
    """'combined=3,name=2' -> {'combined': 3, 'name': 2} (validated, k > 0 only)."""
    out = {}
    for item in (spec or "").replace(";", ",").split(","):
        item = item.strip()
        if not item:
            continue
        name, _, k = item.partition("=")
        name = name.strip()
        if name not in CHANNELS:
            raise ValueError(f"unknown channel {name!r}; known: {CHANNELS}")
        k = int(k) if k else 3
        if k > 0:
            out[name] = k
    if "combined" not in out:
        raise ValueError("the 'combined' channel is required")
    return out


def explode(df, col, prefix=None):
    """(lrow, h) pairs of one feature block (optionally only the features starting with `prefix`)."""
    q = (
        df.lazy()
        .select(pl.col("lrow"), pl.col(col).str.split(" ").alias("f"))
        .explode("f")
        .filter(pl.col("f").is_not_null() & (pl.col("f") != ""))
    )
    if prefix:
        q = q.filter(pl.col("f").str.starts_with(prefix))
    return q.select("lrow", pl.col("f").hash(seed=17).alias("h")).collect()


def block_stats(trS, trT, nS):
    """IDF (from Source 1) of every feature of one block, and the features shared by both sides."""
    dfS = trS.group_by("h").agg(pl.len().cast(pl.Int64).alias("dfS"))
    dfT = trT.group_by("h").agg(pl.len().cast(pl.Int64).alias("dfT"))
    idf = dfS.with_columns((np.log1p(nS / pl.col("dfS"))).cast(pl.Float32).alias("idf")).select("h", "idf", "dfS")
    # features absent from S1 get the max idf (rare) for the norm computation
    idf_all = pl.concat([
        idf.select("h", "idf"),
        dfT.join(dfS, on="h", how="anti").select("h", pl.lit(float(np.log1p(nS)), pl.Float32).alias("idf")),
    ])
    shared = idf.join(dfT, on="h", how="inner")
    return idf_all, shared


def block_norms(trip, idf, n_rows):
    """Per-row L2 norm of the idf vector of one block."""
    w = trip.join(idf, on="h", how="left").with_columns(pl.col("idf").fill_null(pl.col("idf").max()))
    agg = w.group_by("lrow").agg((pl.col("idf") ** 2).sum().sqrt().alias("nrm"))
    norms = np.ones(n_rows, dtype=np.float32)
    norms[agg["lrow"].to_numpy()] = agg["nrm"].to_numpy()
    return norms


def build_matrix(trip, feat, norms, n_rows, n_cols, weight="idf"):
    """CSR matrix over the features in `feat` (h -> col, idf): idf/norm weights, or plain idf /
    binary weights when `weight` is 'idf_raw' / 'binary' (rare / hn channels: no normalisation)."""
    m = trip.join(feat, on="h", how="inner").sort("lrow")
    rows = m["lrow"].to_numpy()
    cols = m["col"].to_numpy().astype(np.int32)
    if weight == "idf":
        vals = (m["idf"].to_numpy() / norms[rows]).astype(np.float32)
    elif weight == "idf_raw":
        vals = m["idf"].to_numpy().astype(np.float32)
    else:
        vals = np.ones(len(rows), dtype=np.float32)
    del m
    indptr = np.zeros(n_rows + 1, dtype=np.int64)
    indptr[1:] = np.cumsum(np.bincount(rows, minlength=n_rows))
    return sp.csr_matrix((vals, cols, indptr), shape=(n_rows, n_cols))


# --------------------------------------------------------------- retrieval
def _save_csr(mat, prefix):
    np.save(prefix + "_data.npy", mat.data.astype(np.float32))
    np.save(prefix + "_indices.npy", mat.indices.astype(np.int32))
    np.save(prefix + "_indptr.npy", mat.indptr.astype(np.int64))


def _load_csr(prefix, shape):
    d = np.load(prefix + "_data.npy", mmap_mode="r")
    i = np.load(prefix + "_indices.npy", mmap_mode="r")
    p = np.load(prefix + "_indptr.npy", mmap_mode="r")
    return sp.csr_matrix((d, i, p), shape=shape, copy=False)


def _init(tmp, mats, channels, reverse):
    """mats: {name: (T shape, ST shape)} per matrix family; loads the memory-mapped matrices."""
    _G.clear()
    for name, (t_shape, st_shape) in mats.items():
        _G["T_" + name] = _load_csr(os.path.join(tmp, "T_" + name), t_shape)
        _G["ST_" + name] = _load_csr(os.path.join(tmp, "ST_" + name), st_shape)
    _G["channels"] = channels
    _G["reverse"] = reverse


def topk_per_row(c, k, min_score, offset):
    """Top-k entries per row of a sparse product c (rows offset by `offset`)."""
    counts = np.diff(c.indptr)
    rows = np.repeat(np.arange(c.shape[0], dtype=np.int64), counts)
    data, cols = c.data, c.indices
    keep = data >= min_score
    rows, data, cols = rows[keep], data[keep], cols[keep]
    order = np.lexsort((-data, rows))
    rows, data, cols = rows[order], data[order], cols[order]
    counts = np.bincount(rows, minlength=c.shape[0])
    starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
    rank = np.arange(len(rows)) - np.repeat(starts, counts)
    sel = rank < k
    return rows[sel] + offset, cols[sel].astype(np.int32), data[sel].astype(np.float32), rank[sel].astype(np.int16)


def _retrieve(bounds):
    """One chunk of Source 2/3 rows (or of Source 1 rows for the reverse channel): the per-channel
    top-k lists. Returns a list of (channel, rows, cols, score, rank) with rows/cols in (T, S) order."""
    a, b = bounds
    ch = _G["channels"]
    out = []
    if _G["reverse"]:
        # S1 rows against all Source 2/3 rows on the combined score
        c = None
        for name in COMBINED:
            if "T_" + name not in _G:
                continue
            prod = _G["ST_" + name][a:b] @ _G["T_" + name]  # here ST_<name> holds S (rows) and T_<name> holds T^T
            c = prod if c is None else c + prod
        rows, cols, data, rank = topk_per_row(c, ch["rev"], MIN_SCORE["rev"], a)
        out.append(("rev", cols.astype(np.int64), rows.astype(np.int32), data, rank))  # swap to (T, S)
        return out
    prods = {}
    for name in BLOCK_NAMES:
        if "T_" + name in _G:
            prods[name] = _G["T_" + name][a:b] @ _G["ST_" + name]
    comb = None
    for name in COMBINED:
        if name in prods:
            comb = prods[name] if comb is None else comb + prods[name]
    prods["combined"] = comb
    for extra in ("rare", "hn"):
        if "T_" + extra in _G:
            prods[extra] = _G["T_" + extra][a:b] @ _G["ST_" + extra]
    for name, k in ch.items():
        if name == "rev" or name not in prods or prods[name] is None:
            continue
        out.append((name, *topk_per_row(prods[name], k, MIN_SCORE[name], a)))
    return out


def rowwise_dot(A, B, ia, ib, chunk=1_000_000):
    out = np.empty(len(ia), dtype=np.float32)
    for s in range(0, len(ia), chunk):
        x = A[ia[s:s + chunk]].multiply(B[ib[s:s + chunk]])
        out[s:s + chunk] = np.asarray(x.sum(axis=1)).ravel()
    return out


def run_pool(tmp, mats, channels, reverse, bounds, tag):
    res = []
    with mp.get_context("spawn").Pool(n_workers(), initializer=_init, initargs=(tmp, mats, channels, reverse)) as pool:
        for k, r in enumerate(pool.imap_unordered(_retrieve, bounds)):
            res.extend(r)
            if k % 100 == 0:
                log(f"{tag} retrieval {k + 1}/{len(bounds)}")
    return res


def run_country(rec, country, args, tmp_root, channels):
    """Returns the union candidate DataFrame of one country (exact cosines, per-channel ranks)."""
    S = rec.filter((pl.col("src") == 1) & (pl.col("country") == country)).select("rid", *BLOCKS)
    T = rec.filter((pl.col("src") != 1) & (pl.col("country") == country)).select("rid", *BLOCKS)
    nS, nT = S.height, T.height
    log(f"[{country}] S1={nS} T={nT}")
    if nS == 0 or nT == 0:
        return None
    s_rid = S["rid"].to_numpy()
    t_rid = T["rid"].to_numpy()
    S = S.with_row_index("lrow")
    T = T.with_row_index("lrow")
    tmp = os.path.join(tmp_root, f"c_{abs(hash(country)) % 10**8}")
    rtmp = os.path.join(tmp, "rev")
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(rtmp)

    # ---- per block: idf, norms, df-capped index matrices (one block at a time to bound memory)
    mats, block_info = {}, {}
    need_rev = "rev" in channels
    for col, bname in zip(BLOCKS, BLOCK_NAMES):
        trS, trT = explode(S, col), explode(T, col)
        idf_all, shared = block_stats(trS, trT, nS)
        normS, normT = block_norms(trS, idf_all, nS), block_norms(trT, idf_all, nT)
        del idf_all
        capped = shared.filter(pl.col("dfS") <= args.df_cap)
        cost = int(capped.select((pl.col("dfS") * pl.col("dfT")).sum()).item() or 0)
        log(f"[{country}] block {bname}: shared feats={shared.height} capped multiply-adds={cost}")
        feat_ix = capped.with_row_index("col").select("h", pl.col("col").cast(pl.Int64), "idf")
        if bname in COMBINED or bname in channels:  # a block is a retrieval matrix only when some channel uses it
            XS = build_matrix(trS, feat_ix, normS, nS, feat_ix.height)
            XT = build_matrix(trT, feat_ix, normT, nT, feat_ix.height)
            _save_csr(XT, os.path.join(tmp, "T_" + bname))
            _save_csr(XS.T.tocsr(), os.path.join(tmp, "ST_" + bname))
            mats[bname] = ((nT, feat_ix.height), (feat_ix.height, nS))
            if need_rev and bname in COMBINED:
                # the reverse worker reads "ST_<b>" as S (nS x F) and "T_<b>" as T^T (F x nT)
                _save_csr(XS, os.path.join(rtmp, "ST_" + bname))
                _save_csr(XT.T.tocsr(), os.path.join(rtmp, "T_" + bname))
            del XS, XT
        if bname == "name":
            if "rare" in channels:
                rare = shared.filter(pl.col("dfS") <= args.rare_df).with_row_index("col").select("h", pl.col("col").cast(pl.Int64), "idf")
                _save_csr(build_matrix(trT, rare, normT, nT, rare.height, "idf_raw"), os.path.join(tmp, "T_rare"))
                _save_csr(build_matrix(trS, rare, normS, nS, rare.height, "binary").T.tocsr(), os.path.join(tmp, "ST_rare"))
                mats["rare"] = ((nT, rare.height), (rare.height, nS))
                log(f"[{country}] rare channel: {rare.height} features with df <= {args.rare_df}")
        if bname == "addr" and "hn" in channels:
            hS, hT = explode(S, col, "h:"), explode(T, col, "h:")
            hshared = hS.group_by("h").agg(pl.len().alias("dfS")).join(hT.select("h").unique(), on="h").filter(pl.col("dfS") <= args.df_cap)
            hfeat = hshared.with_row_index("col").select("h", pl.col("col").cast(pl.Int64), pl.lit(1.0, pl.Float32).alias("idf"))
            _save_csr(build_matrix(hT, hfeat, normT, nT, hfeat.height, "binary"), os.path.join(tmp, "T_hn"))
            _save_csr(build_matrix(hS, hfeat, normS, nS, hfeat.height, "binary").T.tocsr(), os.path.join(tmp, "ST_hn"))
            mats["hn"] = ((nT, hfeat.height), (hfeat.height, nS))
            log(f"[{country}] hn channel: {hfeat.height} house-number keys")
        block_info[bname] = (shared.select("h", "idf"), normS, normT)
        del trS, trT, feat_ix

    # ---- forward retrieval: every Source 2/3 record against Source 1, all channels
    bounds = [(a, min(a + args.chunk, nT)) for a in range(0, nT, args.chunk)]
    res = run_pool(tmp, mats, channels, False, bounds, f"[{country}]")
    if need_rev:
        # the reverse worker multiplies S (nS x F, read as "ST_<b>") by T^T (F x nT, read as "T_<b>")
        rmats = {b: ((mats[b][0][1], nT), (nS, mats[b][0][1])) for b in COMBINED if b in mats}
        rbounds = [(a, min(a + args.chunk, nS)) for a in range(0, nS, args.chunk)]
        res += run_pool(rtmp, rmats, channels, True, rbounds, f"[{country}] reverse")
    shutil.rmtree(tmp, ignore_errors=True)

    # ---- union of the channels: one row per (t, s) with the rank in every channel
    frames = []
    while res:
        name, rows, cols, score, rank = res.pop()
        if len(rows):
            frames.append(pl.DataFrame({"tl": rows.astype(np.int64), "sl": cols.astype(np.int64),
                                        "ch": np.full(len(rows), CHANNELS.index(name), dtype=np.int8), "rk": rank}))
    if not frames:
        return None
    pairs = pl.concat(frames)
    del frames
    per_ch = pairs.group_by("tl", "sl", "ch").agg(pl.col("rk").min())
    union = per_ch.group_by("tl", "sl").agg(pl.len().cast(pl.Int16).alias("n_ch"))
    for name, cid in ((n, CHANNELS.index(n)) for n in channels):
        r = per_ch.filter(pl.col("ch") == cid).select("tl", "sl", pl.col("rk").alias("r_" + name))
        union = union.join(r, on=["tl", "sl"], how="left").with_columns(pl.col("r_" + name).fill_null(-1).cast(pl.Int16))
    del pairs, per_ch
    union = union.sort("tl", "sl")
    tl, sl = union["tl"].to_numpy(), union["sl"].to_numpy()
    log(f"[{country}] union pairs {union.height} ({union.height / nT:.2f} per Source 2/3 record); "
        + ", ".join(f"{n}={int((union['r_' + n] >= 0).sum())}" for n in channels))

    # ---- exact per-block cosines with all shared features (no df cap)
    cos = {}
    for col, bname in zip(BLOCKS, BLOCK_NAMES):
        shared, normS, normT = block_info[bname]
        feat_all = shared.with_row_index("col").select("h", pl.col("col").cast(pl.Int64), "idf")
        XS = build_matrix(explode(S, col), feat_all, normS, nS, feat_all.height)
        XT = build_matrix(explode(T, col), feat_all, normT, nT, feat_all.height)
        cos[bname] = rowwise_dot(XT, XS, tl, sl)
        del XS, XT
    score = cos["name"] + cos["addr"] + cos["cross"]
    out = pl.DataFrame({
        "t_rid": t_rid[tl].astype(np.uint32), "s_rid": s_rid[sl].astype(np.uint32), "score": score,
        "cos_name": cos["name"], "cos_addr": cos["addr"], "cos_cross": cos["cross"], "cos_nchar": cos["nchar"],
        "n_ch": union["n_ch"], **{c: union[c] for c in union.columns if c.startswith("r_")},
    })
    return out.with_columns((pl.col("score").rank("ordinal", descending=True).over("t_rid").cast(pl.Int16) - 1).alias("rank"))


# ------------------------------------------------------------------ audit
def candidate_audit(cand, truth, s1_rids, channels, ambiguous=None):
    """Retrieval measured against the training labels (diagnostic only): true-pair recall of the
    union and per channel, complete-entity coverage, candidate oracle macro F0.5.

    ambiguous: optional bool per true pair marking pairs no retrieval can single out (an address-less
    Source 2/3 record whose Source 1 entity has exact-name namesakes): the misses are split into
    ambiguous / recoverable so that retrieval work is aimed at the recoverable ones."""
    hit = truth.join(cand.select("t_rid", "s_rid", "n_ch", *[c for c in cand.columns if c.startswith("r_")]), on=["t_rid", "s_rid"], how="left")
    found = hit["n_ch"].is_not_null().to_numpy()
    miss_split = None
    if ambiguous is not None:
        amb = np.asarray(ambiguous, dtype=bool)
        miss_split = {"ambiguous_true_pairs": int(amb.sum()), "missed_ambiguous": int((~found & amb).sum()),
                      "missed_recoverable": int((~found & ~amb).sum()),
                      "recall_on_recoverable": float(found[~amb].mean()) if (~amb).any() else None}
    per_channel = {n: float((hit["r_" + n].fill_null(-1) >= 0).mean()) for n in channels}
    only = {n: int(((hit["r_" + n].fill_null(-1) >= 0) & (hit["n_ch"].fill_null(0) == 1)).sum()) for n in channels}
    # entity level
    s_index = pl.DataFrame({"s_rid": s1_rids.astype(np.uint32)}).with_row_index("s_idx")
    ent = truth.join(s_index, on="s_rid").with_columns(pl.Series("found", found))
    e = ent.group_by("s_idx").agg(pl.len().alias("nt"), pl.col("found").sum().alias("tp"))
    nt = np.zeros(len(s1_rids), dtype=np.int64)
    tp = np.zeros(len(s1_rids), dtype=np.int64)
    nt[e["s_idx"].to_numpy()] = e["nt"].to_numpy()
    tp[e["s_idx"].to_numpy()] = e["tp"].to_numpy()
    with np.errstate(divide="ignore", invalid="ignore"):
        rec = np.where(nt > 0, tp / np.maximum(nt, 1), 0.0)
        f = np.where(rec > 0, 1.25 * 1.0 * rec / (0.25 * 1.0 + rec), 0.0)  # precision 1 (oracle)
    f = np.where(nt == 0, 1.0, f)
    non_single = nt > 0
    return {
        "n_true_pairs": int(truth.height),
        "pair_recall": float(found.mean()),
        "pair_recall_by_channel": per_channel,
        "true_pairs_found_by_one_channel_only": only,
        "entities": int(len(s1_rids)), "entities_with_matches": int(non_single.sum()),
        "entity_complete_coverage": float((tp[non_single] == nt[non_single]).mean()) if non_single.any() else None,
        "entity_zero_coverage": float((tp[non_single] == 0).mean()) if non_single.any() else None,
        "oracle_macro_f05": float(f.mean()),
        "oracle_macro_f05_non_singletons": float(f[non_single].mean()) if non_single.any() else None,
        "candidates_per_record": float(cand.height / max(cand["t_rid"].n_unique(), 1)),
        "candidates_total": int(cand.height),
        "channels": channels,
        "misses": miss_split,
    }


def main():
    ap = base_args(__doc__)
    ap.add_argument("--split", required=True, choices=["train", "test"])
    ap.add_argument("--df-cap", type=int, default=1000)
    ap.add_argument("--channels", default=DEFAULT_CHANNELS, help="channel=k list; 'combined=3' = the previous single-channel retrieval")
    ap.add_argument("--rare-df", type=int, default=5, help="rare channel: name features with Source 1 df <= this")
    ap.add_argument("--chunk", type=int, default=4000)
    args = ap.parse_args()
    with Stage(args.work_dir, "blocking_" + args.split):
        run(args)


def run(args):
    d = split_dir(args.work_dir, args.split)
    channels = parse_channels(args.channels)
    log("channels", channels)
    rec = pl.read_parquet(os.path.join(d, "records.parquet"), columns=["rid", "src", "country", *BLOCKS])
    rec = rec.with_columns(pl.col("country").fill_null(""))
    countries = rec.filter(pl.col("src") == 1)["country"].unique().sort().to_list()
    tmp_root = os.path.join(d, "block_tmp")
    parts_dir = os.path.join(d, "cand_parts")
    shutil.rmtree(parts_dir, ignore_errors=True)
    os.makedirs(parts_dir)
    n_parts = 0
    for i, c in enumerate(countries):
        # each country's candidates go straight to disk: the full data has tens of millions of pairs
        r = run_country(rec, c, args, tmp_root, channels)
        if r is not None:
            r.write_parquet(os.path.join(parts_dir, f"part_{i:03d}.parquet"))
            n_parts += 1
        del r
    del rec
    out_path = os.path.join(d, "candidates_raw.parquet")
    if n_parts == 0:
        raise SystemExit("no candidate pairs retrieved")
    pl.scan_parquet(os.path.join(parts_dir, "part_*.parquet")).sink_parquet(out_path)  # streaming concat
    shutil.rmtree(parts_dir, ignore_errors=True)
    shutil.rmtree(tmp_root, ignore_errors=True)
    n_pairs = pl.scan_parquet(out_path).select(pl.len()).collect().item()
    log("candidates_raw", n_pairs, "pairs")
    if args.split == "train":
        from pair_features import truth_pairs  # noqa: E402  (label use is diagnostic only)
        ids = pl.read_parquet(os.path.join(d, "records.parquet"), columns=["rid", "entity_id", "src", "group", "a_norm"])
        truth = truth_pairs(ids, args.data_dir).select("t_rid", "s_rid")
        s1_rids = ids.filter(pl.col("src") == 1)["rid"].to_numpy().astype(np.uint32)
        cand = pl.read_parquet(out_path, columns=["t_rid", "s_rid", "n_ch"] + ["r_" + n for n in channels])
        # ambiguous true pair: address-less record + Source 1 entity with exact-name namesakes
        namesakes = ids.filter(pl.col("src") == 1).group_by("group").agg(pl.len().alias("n_ns"))
        amb = (truth.join(ids.select(pl.col("rid").cast(pl.UInt32).alias("t_rid"), (pl.col("a_norm").fill_null("") == "").alias("noaddr")), on="t_rid", how="left")
               .join(ids.select(pl.col("rid").cast(pl.UInt32).alias("s_rid"), "group"), on="s_rid", how="left")
               .join(namesakes, on="group", how="left")
               .select((pl.col("noaddr").fill_null(False) & (pl.col("n_ns").fill_null(1) > 1)).alias("amb"))["amb"].to_numpy())
        audit = candidate_audit(cand, truth, s1_rids, channels, amb)
        audit["recall_at_k"] = {}  # kept for older readers of this file
        with open(os.path.join(d, "blocking_recall.json"), "w") as f:
            json.dump(audit, f, indent=1)
        log("candidate audit:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in audit.items() if k not in ("pair_recall_by_channel", "true_pairs_found_by_one_channel_only")})
        log("  recall by channel:", {k: round(v, 4) for k, v in audit["pair_recall_by_channel"].items()})
        log("  true pairs found by one channel only:", audit["true_pairs_found_by_one_channel_only"])


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/er/src/pair_features.py
"""Step 3: candidate selection + pairwise features.

Candidate set (this is exactly what the matching model scores, and what is written
to candidate_pairs.tsv): every pair retrieved by blocking.py (--topk / --min-score there).
--max-rank / --min-score here only exist to *tighten* the set for experiments; by default
nothing is filtered a second time.

Features (all country-agnostic; the country label itself is never a feature):
  retrieval  : score, rank, per-block TF-IDF cosines, gaps to the best candidate of the
               same Source 2/3 record, candidate-list statistics on both sides
  name       : rapidfuzz ratio / token_sort / token_set / partial / Jaro-Winkler on the
               normalised full & core names, alias-part max, compact-name and domain-stem
               similarities, token-set overlaps, length features
  address    : rapidfuzz similarities on the normalised address, token and bigram
               overlaps, house/unit number agreement, missing-field flags
  record     : source of the Source 2/3 record, Indic-script / alias / domain flags
  numbers    : relation between the two house numbers (equal / dropped leading digits / truncated /
               one digit substituted / transposed / other), their numeric offset, cross-containment
  street     : similarity of the street name without its number
  legal      : legal-form tokens on each side, overlap and conflict (LLC vs Inc, SARL vs SAS)
  crowding   : how many Source 1 records share the Source 1 address / street; how many candidates of
               the Source 2/3 record share its exact address or its name
  extra toks : typo-tolerant count of name tokens present on one side only
  channels   : rank of the pair in every retrieval channel (r_<channel>, -1 = not proposed), number of
               channels that proposed it (n_ch), char-4-gram name cosine (cos_nchar) - from blocking.py
  contradict : strong negative evidence: different house numbers on both sides (hn_conflict), different
               unit / flat / suite numbers (unit_conflict), different postal-like codes (pc_conflict),
               no digit group in common although both sides carry numbers (num_conflict), identical
               names at contradicting addresses (na_conflict)
  token idf  : (--token-idf, off by default) a Source 1 - IDF weighted view of the name tokens: the
               rarest token present on one side only (xt_maxidf / xs_maxidf, "rare token conflict") and
               the rarest token both share (common_maxidf). Measured on the 5 % subset: +0.0005 OOF macro
               F0.5 but -0.0010 on the shifted labelled hold-out (the statistics are transductive and do
               not transfer between corpora), so the group is not part of the default feature set.

Output: <work>/<split>/pairs/part_XXXX.parquet (pid, t_rid, s_rid, features..., [label for train])
"""
import multiprocessing as mp
import os

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler

from common import Stage, base_args, log, n_workers, read_tsv, split_dir

REC_COLS = ["rid", "entity_id", "src", "country", "n_full", "n_core", "n_parts", "n_compact", "n_domain", "n_legal",
            "f_indic", "f_alias", "a_norm", "a_comp", "a_num", "a_hn", "a_street", "a_key"]
_POOL = None
# unit / flat / suite / shop numbers inside a normalised address component (textnorm canonical forms)
UNIT_PATTERN = r"\b(?:unit|apt|ste|fl|flat|shop|room|rm|office|door|gala|lot|cabin)\s*(?:no\s*)?[a-z]?\d+[a-z]?\b"
PC_PATTERN = r"\b\d{5,6}\b"  # postal-like code: 5-6 digit group (US ZIP, Indian PIN, French code postal)


def select_candidates(cand, max_rank=None, min_score=None):
    if max_rank is not None:
        cand = cand.filter(pl.col("rank") < max_rank)
    if min_score is not None:
        cand = cand.filter(pl.col("score") >= min_score)
    return cand


def context_features(c):
    """Features describing the candidate lists around each pair."""
    c = c.with_columns(
        pl.col("score").max().over("t_rid").alias("t_best"),
        pl.len().over("t_rid").alias("t_ncand"),
        pl.col("cos_name").max().over("t_rid").alias("t_best_name"),
        pl.col("cos_addr").max().over("t_rid").alias("t_best_addr"),
        pl.col("cos_cross").max().over("t_rid").alias("t_best_cross"),
        pl.col("score").sort(descending=True).over("t_rid", mapping_strategy="join").list.get(1, null_on_oob=True).alias("t_second"),
        pl.len().over("s_rid").alias("s_ncand"),
        pl.col("score").rank("ordinal", descending=True).over("s_rid").alias("s_rank"),
        pl.col("score").max().over("s_rid").alias("s_best"),
        (pl.col("rank") == 0).sum().over("s_rid").alias("s_ntop1"),
    )
    return c.with_columns(
        (pl.col("t_best") - pl.col("score")).alias("gap_best"),
        (pl.col("score") - pl.col("t_second").fill_null(0)).alias("gap_second"),
        (pl.col("t_best_name") - pl.col("cos_name")).alias("gap_name"),
        (pl.col("t_best_addr") - pl.col("cos_addr")).alias("gap_addr"),
        (pl.col("t_best_cross") - pl.col("cos_cross")).alias("gap_cross"),
        (pl.col("s_best") - pl.col("score")).alias("s_gap_best"),
    )


def _sim(scorer, a, b):
    """Pairwise (row-wise) similarity of two equally long string lists as float32."""
    if hasattr(process, "cpdist"):  # rapidfuzz >= 3.9: parallel C++ implementation
        return process.cpdist(a, b, scorer=scorer, workers=-1, dtype=np.float32)
    return np.fromiter((scorer(x, y) for x, y in zip(a, b)), dtype=np.float32, count=len(a))


def string_features(df):
    """df has columns t_* and s_* with the record fields of both sides."""
    out = {}
    g = lambda c: df[c].fill_null("").to_list()  # noqa: E731
    tf, sf = g("t_n_full"), g("s_n_full")
    tc, sc = g("t_n_core"), g("s_n_core")
    out["nm_ratio"] = _sim(fuzz.ratio, tf, sf)
    out["nm_core_ratio"] = _sim(fuzz.ratio, tc, sc)
    out["nm_tsort"] = _sim(fuzz.token_sort_ratio, tc, sc)
    out["nm_tset"] = _sim(fuzz.token_set_ratio, tc, sc)
    out["nm_partial"] = _sim(fuzz.partial_ratio, tc, sc)
    out["nm_jw"] = _sim(JaroWinkler.normalized_similarity, tc, sc)
    # alias parts: best / last part of the Source 2/3 name against the Source 1 core name
    parts = df.select(
        pl.col("t_n_parts").fill_null("").str.split("|").alias("p")
    ).with_columns(pl.col("p").list.first().alias("p0"), pl.col("p").list.last().alias("p1"))
    p0 = parts["p0"].fill_null("").to_list()
    p1 = parts["p1"].fill_null("").to_list()
    a0 = _sim(fuzz.token_set_ratio, p0, sc)
    a1 = _sim(fuzz.token_set_ratio, p1, sc)
    out["nm_part_max"] = np.maximum(a0, a1)
    out["nm_part_min"] = np.minimum(a0, a1)
    # compact names (handles concatenated / domain style names)
    comp = df.select(
        pl.col("t_n_compact").fill_null("").str.split("|").list.first().fill_null("").alias("tc"),
        pl.col("s_n_compact").fill_null("").str.split("|").list.first().fill_null("").alias("sc"),
        pl.col("s_n_compact").fill_null("").str.split("|").list.last().fill_null("").alias("sf"),
        pl.col("t_n_domain").fill_null("").str.split("|").list.first().fill_null("").alias("td"),
    )
    tcc, scc, sfc, td = (comp[c].to_list() for c in ("tc", "sc", "sf", "td"))
    out["cmp_ratio"] = _sim(fuzz.ratio, tcc, scc)
    out["cmp_jw"] = _sim(JaroWinkler.normalized_similarity, tcc, scc)
    has_dom = np.array([len(x) > 0 for x in td])
    dpre = np.maximum(
        _sim(fuzz.ratio, td, [s[:len(d)] for s, d in zip(scc, td)]),
        _sim(fuzz.ratio, td, [s[:len(d)] for s, d in zip(sfc, td)]),
    )
    dfull = np.maximum(_sim(fuzz.ratio, td, scc), _sim(fuzz.ratio, td, sfc))
    out["dom_prefix"] = np.where(has_dom, dpre, -1).astype(np.float32)
    out["dom_full"] = np.where(has_dom, dfull, -1).astype(np.float32)
    # address
    ta, sa = g("t_a_norm"), g("s_a_norm")
    out["ad_ratio"] = _sim(fuzz.ratio, ta, sa)
    out["ad_tsort"] = _sim(fuzz.token_sort_ratio, ta, sa)
    out["ad_tset"] = _sim(fuzz.token_set_ratio, ta, sa)
    out["ad_partial"] = _sim(fuzz.partial_ratio, ta, sa)
    # first address component containing a digit (street line) vs the same on the other side
    street = df.select(
        pl.col("t_a_comp").fill_null("").str.split(",").list.eval(pl.element().filter(pl.element().str.contains(r"\d"))).list.first().fill_null("").alias("ts"),
        pl.col("s_a_comp").fill_null("").str.split(",").list.eval(pl.element().filter(pl.element().str.contains(r"\d"))).list.first().fill_null("").alias("ss"),
    )
    out["street_ratio"] = _sim(fuzz.ratio, street["ts"].to_list(), street["ss"].to_list())
    out["street_tset"] = _sim(fuzz.token_set_ratio, street["ts"].to_list(), street["ss"].to_list())
    return pl.DataFrame(out)


# --------------------------------------------------------------- house numbers / tokens
def _hn_relation(a, b):
    """Relation between two house numbers (leading zeros already stripped).

    -1 missing, 0 equal, 1 one is a suffix of the other (dropped leading digits),
    2 prefix (truncated), 3 same length with one substituted digit, 4 transposition, 5 other.
    True matches mostly corrupt a number by dropping/truncating digits, while decoy records
    of a *different* business sit a few doors away (small numeric offset).
    """
    if not a or not b:
        return -1
    if a == b:
        return 0
    if a.endswith(b) or b.endswith(a):
        return 1
    if a.startswith(b) or b.startswith(a):
        return 2
    if len(a) == len(b):
        if sum(x != y for x, y in zip(a, b)) == 1:
            return 3
        if sorted(a) == sorted(b):
            return 4
    return 5


def _tok_unmatched(xs, ys):
    """Number of tokens of xs with no fuzzy counterpart in ys (typo-tolerant set difference)."""
    n = 0
    for x in xs:
        if x in ys:
            continue
        if not any(fuzz.ratio(x, y) >= 80 or (len(x) >= 3 and len(y) >= 3 and (x.startswith(y) or y.startswith(x)))
                   for y in ys):
            n += 1
    return n


def _conflict_expr(t, s, name):
    """1 when both list columns are non-empty and share no element, 0 when both are non-empty and share
    one, -1 otherwise (vectorised version of the contradiction test)."""
    both = (t.list.len() > 0) & (s.list.len() > 0)
    return pl.when(both).then((t.list.set_intersection(s).list.len() == 0).cast(pl.Float32)).otherwise(-1.0).alias(name)


def conflict_features(df):
    """unit_conflict / pc_conflict / num_conflict from the normalised address components, the
    postal-like digit groups (house number excluded) and all digit groups."""
    # the number of each match: an optional "no" is skipped first so that "flatno14" gives "14", not "o14"
    unit = lambda c: pl.col(c).fill_null("").str.extract_all(UNIT_PATTERN).list.eval(pl.element().str.extract(r"(?:no\s*)?([a-z]?\d+[a-z]?)$"))  # noqa: E731
    pc = lambda c, hn: pl.col(c).fill_null("").str.extract_all(PC_PATTERN).list.set_difference(pl.concat_list(pl.col(hn).fill_null("")))  # noqa: E731
    num = lambda c: pl.col(c).fill_null("").str.split(" ").list.eval(pl.element().filter(pl.element() != ""))  # noqa: E731
    return df.select(
        _conflict_expr(unit("t_a_comp"), unit("s_a_comp"), "unit_conflict"),
        _conflict_expr(pc("t_a_num", "t_a_hn"), pc("s_a_num", "s_a_hn"), "pc_conflict"),
        _conflict_expr(num("t_a_num"), num("s_a_num"), "num_conflict"),
    )


def _py_features(args):
    t_hn, s_hn, t_num, s_num, t_core, s_core = args
    out = np.empty((len(t_hn), 9), dtype=np.float32)
    for i in range(len(t_hn)):
        a, b = t_hn[i] or "", s_hn[i] or ""
        rel = _hn_relation(a, b)
        # contradiction: house numbers that are neither equal nor a digit-corruption of each other
        out[i, 8] = -1 if rel < 0 else (1.0 if rel == 5 else 0.0)
        if rel >= 0:
            x, y = int(a[:9]), int(b[:9])
            d = abs(x - y)
            out[i, 1] = np.log1p(d)
            out[i, 2] = d / max(x, y, 1)
        else:
            out[i, 1] = out[i, 2] = -1
        tn = (t_num[i] or "").split()
        sn = (s_num[i] or "").split()
        out[i, 0] = rel
        out[i, 3] = (b in tn) if b and tn else -1
        out[i, 4] = (a in sn) if a and sn else -1
        out[i, 5] = len(a) - len(b) if a and b else -99
        tc = (t_core[i] or "").split()
        sc = (s_core[i] or "").split()
        out[i, 6] = _tok_unmatched(tc, sc)
        out[i, 7] = _tok_unmatched(sc, tc)
    return out


PY_FEATS = ["hn_rel", "hn_logdiff", "hn_reldiff", "hn_s_in_t", "hn_t_in_s", "hn_lendiff", "nm_xt", "nm_xs", "hn_conflict"]


def python_features(df, workers=None):
    cols = [df[c].to_list() for c in ("t_a_hn", "s_a_hn", "t_a_num", "s_a_num", "t_n_core", "s_n_core")]
    n = df.height
    step = 50_000
    jobs = [tuple(c[a:a + step] for c in cols) for a in range(0, n, step)]
    res = _POOL.map(_py_features, jobs) if _POOL is not None else [_py_features(j) for j in jobs]
    X = np.vstack(res) if res else np.empty((0, len(PY_FEATS)), dtype=np.float32)
    out = pl.DataFrame({name: X[:, k] for k, name in enumerate(PY_FEATS)})
    g = lambda c: df[c].fill_null("").to_list()  # noqa: E731
    ts, ss = g("t_a_street"), g("s_a_street")
    out = out.with_columns(
        pl.Series("st_ratio", _sim(fuzz.ratio, ts, ss)),
        pl.Series("st_eq", [(1.0 if a and a == b else (0.0 if a and b else -1.0)) for a, b in zip(ts, ss)], dtype=pl.Float32),
    )
    lg = df.select(
        pl.col("t_n_legal").fill_null("").str.split(" ").list.eval(pl.element().filter(pl.element() != "")).alias("tl"),
        pl.col("s_n_legal").fill_null("").str.split(" ").list.eval(pl.element().filter(pl.element() != "")).alias("sl"),
    ).select(
        pl.col("tl").list.len().alias("lg_t"),
        pl.col("sl").list.len().alias("lg_s"),
        pl.col("tl").list.set_intersection("sl").list.len().alias("lg_common"),
    ).with_columns(
        ((pl.col("lg_t") > 0) & (pl.col("lg_s") > 0) & (pl.col("lg_common") == 0)).cast(pl.Int8).alias("lg_conflict"),
    )
    return pl.concat([out, lg, conflict_features(df)], how="horizontal")


def name_idf(rec):
    """Source 1 document frequency of every core-name token -> (token, idf) table. Computed from the
    records of the split being processed (no labels): a token shared by two records is strong evidence
    when few Source 1 records carry it, and a token present on one side only is a strong contradiction
    when it is rare (a typo of a common token is rare too, which the typo-tolerant nm_xt/nm_xs cover).
    The idf is normalised by its maximum, log1p(n): a token seen once scores 1 whatever the corpus size
    (the raw log1p(n/df) shifts with the number of Source 1 records, which differs between the training
    split and any hold-out / test split)."""
    s1 = rec.filter(pl.col("src") == 1)
    n = max(s1.height, 1)
    df = (s1.lazy().select(pl.col("n_core").fill_null("").str.split(" ").alias("t")).explode("t")
          .filter(pl.col("t") != "").group_by("t").agg(pl.len().alias("df")).collect())
    top = float(np.log1p(n))
    return df.select("t", (np.log1p(n / pl.col("df")) / top).cast(pl.Float32).alias("idf")), 1.0


def idf_features(df, idf, idf_max):
    """xt_maxidf / xs_maxidf: rarest name token present only on the Source 2/3 / Source 1 side;
    common_maxidf: rarest shared token; *_sumidf: idf mass of the unmatched tokens. Unknown tokens
    (absent from Source 1) get the maximum idf; -1 when there is no such token."""
    tok = lambda c: pl.col(c).fill_null("").str.split(" ").list.eval(pl.element().filter(pl.element() != ""))  # noqa: E731
    x = df.select(tok("t_n_core").alias("tn"), tok("s_n_core").alias("sn")).with_row_index("i").with_columns(
        pl.col("tn").list.set_difference("sn").alias("xt"), pl.col("sn").list.set_difference("tn").alias("xs"),
        pl.col("tn").list.set_intersection("sn").alias("cm"))
    out = pl.DataFrame({"i": x["i"]})
    for col, name in (("xt", "xt"), ("xs", "xs"), ("cm", "common")):
        e = x.select("i", pl.col(col).alias("t")).explode("t").filter(pl.col("t").is_not_null()).join(idf, on="t", how="left").with_columns(
            pl.col("idf").fill_null(idf_max))
        agg = e.group_by("i").agg(pl.col("idf").max().alias(f"{name}_maxidf"), pl.col("idf").sum().alias(f"{name}_sumidf"))
        out = out.join(agg, on="i", how="left")
    return out.sort("i").drop("i").fill_null(-1.0)


def set_features(df):
    tok = lambda c: pl.col(c).fill_null("").str.split(" ").list.eval(pl.element().filter(pl.element() != ""))  # noqa: E731
    x = df.select(
        tok("t_n_core").alias("tn"), tok("s_n_core").alias("sn"),
        tok("t_a_norm").alias("ta"), tok("s_a_norm").alias("sa"),
        tok("t_a_num").alias("tu"), tok("s_a_num").alias("su"),
    )
    x = x.with_columns(
        pl.col("tn").list.len().alias("t_ntok"), pl.col("sn").list.len().alias("s_ntok"),
        pl.col("ta").list.len().alias("t_natok"), pl.col("sa").list.len().alias("s_natok"),
        pl.col("tu").list.len().alias("t_nnum"), pl.col("su").list.len().alias("s_nnum"),
        pl.col("tn").list.set_intersection("sn").list.len().alias("nm_common"),
        pl.col("ta").list.set_intersection("sa").list.len().alias("ad_common"),
        pl.col("tu").list.set_intersection("su").list.len().alias("num_common"),
        (pl.col("tu").list.first() == pl.col("su").list.first()).cast(pl.Int8).fill_null(-1).alias("num_first_eq"),
        pl.col("tn").list.first().alias("tn0"), pl.col("sn").list.first().alias("sn0"),
    )
    x = x.with_columns(
        (pl.col("nm_common") / (pl.col("t_ntok") + pl.col("s_ntok") - pl.col("nm_common")).clip(lower_bound=1)).alias("nm_jacc"),
        (pl.col("ad_common") / (pl.col("t_natok") + pl.col("s_natok") - pl.col("ad_common")).clip(lower_bound=1)).alias("ad_jacc"),
        (pl.col("num_common") / pl.max_horizontal("t_nnum", "s_nnum").clip(lower_bound=1)).alias("num_frac"),
        (pl.col("nm_common") / pl.col("s_ntok").clip(lower_bound=1)).alias("nm_cov_s"),
        (pl.col("nm_common") / pl.col("t_ntok").clip(lower_bound=1)).alias("nm_cov_t"),
    )
    first = _sim(fuzz.ratio, x["tn0"].fill_null("").to_list(), x["sn0"].fill_null("").to_list())
    return x.drop("tn", "sn", "ta", "sa", "tu", "su", "tn0", "sn0").with_columns(pl.Series("nm_first_ratio", first))


def record_features(df):
    return df.select(
        pl.col("t_src").cast(pl.Int8).alias("t_src"),
        pl.col("t_f_indic").cast(pl.Int8).alias("t_indic"),
        pl.col("t_f_alias").cast(pl.Int8).alias("t_alias"),
        (pl.col("t_n_domain").fill_null("") != "").cast(pl.Int8).alias("t_domain"),
        (pl.col("t_a_norm").fill_null("") == "").cast(pl.Int8).alias("t_noaddr"),
        (pl.col("s_a_norm").fill_null("") == "").cast(pl.Int8).alias("s_noaddr"),
        (pl.col("t_n_core").fill_null("") == "").cast(pl.Int8).alias("t_noname"),
        pl.col("t_n_full").fill_null("").str.len_chars().alias("t_nlen"),
        pl.col("s_n_full").fill_null("").str.len_chars().alias("s_nlen"),
        pl.col("s_a_mult").alias("s_addr_mult"),
        pl.col("s_a_smult").alias("s_street_mult"),
    )


def address_crowding(rec):
    """Adds a_mult / a_smult: how many Source 1 records of the same country share this record's
    exact street address (house number + street + locality) / its street (street + locality).
    Where many businesses share an address (common in some cities), an exact address match is
    weaker evidence of identity. Computed for Source 1 records (0 elsewhere)."""
    loc = pl.col("a_key").fill_null("").str.split(" ").list.set_difference(
        pl.col("a_street").fill_null("").str.split(" ")).list.sort().list.join(" ")
    k = rec.select(
        "rid", "src",
        pl.concat_str([pl.col("country").fill_null(""), pl.col("a_hn").fill_null(""), pl.col("a_street").fill_null(""), loc],
                      separator="|").alias("ka"),
        pl.concat_str([pl.col("country").fill_null(""), pl.col("a_street").fill_null(""), loc], separator="|").alias("ks"),
        ((pl.col("a_hn").fill_null("") != "") & (pl.col("a_street").fill_null("") != "")).alias("ok"),
    )
    s1 = pl.col("src") == 1
    k = k.with_columns(
        pl.when(s1 & pl.col("ok")).then(pl.len().over("ka", "src")).otherwise(0).cast(pl.Float32).alias("a_mult"),
        pl.when(s1 & pl.col("ok")).then(pl.len().over("ks", "src")).otherwise(0).cast(pl.Float32).alias("a_smult"),
    )
    return rec.with_columns(k["a_mult"], k["a_smult"])


def candidate_group_features(f):
    """Per Source 2/3 record, over its candidate list: how many candidates share its exact
    address, how many carry the same name (namesakes), and how many carry the same name at a
    contradicting address (t_n_namesake_contra). Plus the pair-level name/address contradiction:
    (near-)identical names whose house numbers disagree outright and whose streets differ."""
    exact = ((pl.col("hn_rel") == 0) & (pl.col("st_eq") == 1)).cast(pl.Float32)
    same = ((pl.col("nm_xt") == 0) & (pl.col("nm_xs") == 0)).cast(pl.Float32)
    contra = ((pl.col("nm_tset") >= 90) & (pl.col("hn_conflict") == 1) & (pl.col("st_eq") == 0)).cast(pl.Float32)
    return f.with_columns(
        exact.sum().over("t_rid").alias("t_n_addr_exact"),
        same.sum().over("t_rid").alias("t_n_namesake"),
        contra.alias("na_conflict"),
        (same * contra).sum().over("t_rid").alias("t_n_namesake_contra"),
    )


def chunk_bounds(t_rid, chunk):
    """Chunk boundaries that never split the candidate list of a Source 2/3 record."""
    n = len(t_rid)
    bounds, a = [], 0
    while a < n:
        b = min(a + chunk, n)
        while b < n and t_rid[b] == t_rid[b - 1]:
            b += 1
        bounds.append((a, b))
        a = b
    return bounds


def gather(rec, idx, prefix):
    return rec[idx].select(pl.all().name.prefix(prefix))


def build(cand, rec, chunk, out_dir, truth=None, token_idf=False):
    """Computes features chunk by chunk and writes <out_dir>/part_XXXX.parquet (float32 features)."""
    os.makedirs(out_dir, exist_ok=True)
    for f in os.listdir(out_dir):
        os.remove(os.path.join(out_dir, f))
    cand = cand.with_row_index("pid")
    idf, idf_max = name_idf(rec) if token_idf else (None, None)
    for i, (a, b) in enumerate(chunk_bounds(cand["t_rid"].to_numpy(), chunk)):
        c = cand.slice(a, b - a)
        t = gather(rec, c["t_rid"].to_numpy().astype(np.int64), "t_")
        s = gather(rec, c["s_rid"].to_numpy().astype(np.int64), "s_")
        df = pl.concat([t, s], how="horizontal")
        blocks = [c, string_features(df), set_features(df), record_features(df), python_features(df)]
        if token_idf:
            blocks.append(idf_features(df, idf, idf_max))
        f = pl.concat(blocks, how="horizontal")
        f = candidate_group_features(f)
        keep = {"pid", "t_rid", "s_rid"}
        f = f.with_columns(pl.col(x).cast(pl.Float32) for x in f.columns if x not in keep)
        if truth is not None:
            f = f.join(truth, on=["t_rid", "s_rid"], how="left").with_columns(pl.col("label").fill_null(0))
        f.write_parquet(os.path.join(out_dir, f"part_{i:04d}.parquet"))
        log(f"features {b}/{cand.height}")


def truth_pairs(rec, data_dir):
    gt = read_tsv(os.path.join(data_dir, "train", "train_ground_truth.tsv"))
    ids = rec.select("rid", "entity_id")
    return (
        gt.with_columns(pl.col("matched_entity_ids").str.split(","))
        .explode("matched_entity_ids")
        .drop_nulls("matched_entity_ids")
        .join(ids.rename({"rid": "s_rid", "entity_id": "source1_entity_id"}), on="source1_entity_id")
        .join(ids.rename({"rid": "t_rid", "entity_id": "matched_entity_ids"}), on="matched_entity_ids")
        .select(pl.col("t_rid").cast(pl.UInt32), pl.col("s_rid").cast(pl.UInt32), pl.lit(1, pl.Int8).alias("label"))
    )


def main():
    ap = base_args(__doc__)
    ap.add_argument("--split", required=True, choices=["train", "test"])
    ap.add_argument("--max-rank", type=int, default=None, help="experiments only: keep rank < this")
    ap.add_argument("--min-score", type=float, default=None, help="experiments only: keep score >= this")
    ap.add_argument("--chunk", type=int, default=2_000_000)
    ap.add_argument("--token-idf", action="store_true", help="add the Source 1 IDF token features (see module docstring: hurts under corpus shift)")
    args = ap.parse_args()
    with Stage(args.work_dir, "features_" + args.split):
        run(args)


def run(args):
    d = split_dir(args.work_dir, args.split)
    rec = address_crowding(pl.read_parquet(os.path.join(d, "records.parquet"), columns=REC_COLS))
    cand = select_candidates(pl.read_parquet(os.path.join(d, "candidates_raw.parquet")), args.max_rank, args.min_score)
    cand = context_features(cand).sort("t_rid", "rank")
    log(f"{args.split}: {cand.height} candidate pairs")
    truth = truth_pairs(rec, args.data_dir) if args.split == "train" else None
    global _POOL
    with mp.get_context("spawn").Pool(n_workers()) as _POOL:
        build(cand, rec, args.chunk, os.path.join(d, "pairs"), truth, args.token_idf)
    log("done")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/er/src/model.py
"""Matching model: two-stage LightGBM + many-to-one assignment with confidence-based rejection.

Stage 1 scores every candidate pair from its pairwise features.
Stage 2 re-scores each pair with the stage-1 probabilities of the competing
candidates around it (the other Source 1 records proposed for the same
Source 2/3 record, and the other Source 2/3 records proposed for the same
Source 1 record). Stage-2 training uses out-of-fold stage-1 predictions.

Assignment (many-to-one, no global one-to-one constraint): each Source 2/3 record belongs to
at most one Source 1 entity, while a Source 1 entity may receive any number of records. A record
is linked to its highest-probability candidate only when
  * that probability clears the global threshold (threshold_policy.py),
  * the margin over the record's second-best candidate is at least `margin`, and
  * the pair carries no strong contradiction (contradiction_flag), or its probability also clears
    the threshold raised by `contra_penalty` (penalty >= 1 is a hard veto).
Otherwise the record stays unmatched. The predictions are then aggregated per Source 1 entity
(predict.write_lists), which is what the metric scores.
"""
import glob
import os

import lightgbm as lgb
import numpy as np
import polars as pl

from common import left_join_ordered

NON_FEATURES = {"pid", "t_rid", "s_rid", "label", "fold"}
CONTRA_COLS = ["unit_conflict", "pc_conflict", "na_conflict", "hn_conflict", "st_eq"]

LGB_PARAMS = dict(
    objective="binary",
    learning_rate=0.1,
    num_leaves=127,
    min_data_in_leaf=200,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l2=1.0,
    max_bin=255,
    verbose=-1,
    num_threads=0,
)


def part_files(split_dir):
    return sorted(glob.glob(os.path.join(split_dir, "pairs", "part_*.parquet")))


def iter_parts(files, columns=None):
    for f in files:
        yield pl.read_parquet(f, columns=columns)


def stage1_features(files):
    cols = pl.read_parquet_schema(files[0]).keys()
    return [c for c in cols if c not in NON_FEATURES]


def house_numbers(split_dir):
    """rid -> house number (a_hn) of every record of a split."""
    return pl.read_parquet(os.path.join(split_dir, "records.parquet"), columns=["rid", "a_hn"]).select(
        pl.col("rid").cast(pl.UInt32), pl.col("a_hn").fill_null(""))


def contradiction_flag(df):
    """Strong contradiction of a candidate pair (bool per row of a pair-feature frame): different
    unit / flat numbers, different postal-like codes, identical names at contradicting addresses,
    or a house number that is neither equal nor a digit corruption of the other on the same street."""
    return (
        (df["unit_conflict"] == 1) | (df["pc_conflict"] == 1) | (df["na_conflict"] == 1)
        | ((df["hn_conflict"] == 1) & (df["st_eq"] == 1))
    ).to_numpy()


def stage2_context(meta, p1, hn):
    """Context features from stage-1 probabilities; meta is (t_rid, s_rid) aligned with p1.

    hn: (rid, a_hn) house numbers, used for the consensus features: among the *other* confident
    candidates (p1 >= 0.5) of the same Source 1 entity, how many share this record's house
    number, and how many share the entity's own house number.
    Candidate competition (per Source 2/3 record): best and second-best stage-1 score, the pair's
    margin over its strongest competitor (c_t_margin: p - best other candidate), rank, count.
    Returns (float32 matrix, column names), rows aligned with meta.
    """
    p = pl.col("p1")
    df = meta.select("t_rid", "s_rid").with_columns(pl.Series("p1", p1)).with_columns(
        p.max().over("t_rid").alias("c_t_pmax"),
        p.sum().over("t_rid").alias("c_t_psum"),
        p.rank("ordinal", descending=True).over("t_rid").alias("c_t_prank"),
        p.sort(descending=True).over("t_rid", mapping_strategy="join").list.get(1, null_on_oob=True).fill_null(0.0).alias("c_t_p2nd"),
        pl.len().over("t_rid").alias("c_t_n"),
        p.max().over("s_rid").alias("c_s_pmax"),
        p.sum().over("s_rid").alias("c_s_psum"),
        p.rank("ordinal", descending=True).over("s_rid").alias("c_s_prank"),
        (p > 0.5).sum().over("s_rid").alias("c_s_n05"),
        pl.len().over("s_rid").alias("c_s_n"),
    )
    df = df.with_columns(
        (pl.col("c_t_pmax") - p).alias("c_t_gap"),
        (2 * p - pl.col("c_t_psum")).alias("c_t_vs_rest"),
        (pl.col("c_s_psum") - p).alias("c_s_psum_other"),
        (pl.col("c_s_pmax") - p).alias("c_s_gap"),
        # margin over the strongest competitor: p - second best when this pair is the best, else p - best
        pl.when(pl.col("c_t_prank") == 1).then(p - pl.col("c_t_p2nd")).otherwise(p - pl.col("c_t_pmax")).alias("c_t_margin"),
    )
    df = left_join_ordered(df, hn.rename({"rid": "t_rid", "a_hn": "t_hn"}), "t_rid")
    df = left_join_ordered(df, hn.rename({"rid": "s_rid", "a_hn": "s_hn"}), "s_rid")
    strong = (p >= 0.5) & (pl.col("t_hn") != "")
    df = df.with_columns(
        (strong.cast(pl.Int32).sum().over("s_rid", "t_hn") - strong.cast(pl.Int32)).alias("c_hn_same_t"),
        ((strong & (pl.col("t_hn") == pl.col("s_hn"))).cast(pl.Int32).sum().over("s_rid")
         - (strong & (pl.col("t_hn") == pl.col("s_hn"))).cast(pl.Int32)).alias("c_hn_same_s"),
    ).with_columns(
        pl.when(pl.col("t_hn") == "").then(-1).otherwise(pl.col("c_hn_same_t")).alias("c_hn_same_t"),
        pl.when(pl.col("s_hn") == "").then(-1).otherwise(pl.col("c_hn_same_s")).alias("c_hn_same_s"),
    ).drop("t_rid", "s_rid", "p1", "t_hn", "s_hn")
    return df.to_numpy().astype(np.float32), df.columns


def train_lgb(X, y, rounds, params=None):
    params = dict(LGB_PARAMS, **(params or {}))
    return lgb.train(params, lgb.Dataset(X, label=y), num_boost_round=rounds)


def to_np(df, cols):
    return df.select(cols).to_numpy().astype(np.float32)


def best_candidates(meta, p, contra=None):
    """One row per Source 2/3 record: its best candidate (t_rid, s_rid, p), the second-best probability
    in its list (p2nd, 0 when there is none) and the best pair's contradiction flag (contra).
    Row order: the same as meta's first occurrence order is NOT kept; rows carry `i`, the row index
    of the best pair in meta."""
    df = meta.select("t_rid", "s_rid").with_row_index("i").with_columns(pl.Series("p", np.asarray(p, dtype=np.float32)))
    df = df.with_columns(pl.Series("contra", np.asarray(contra, dtype=bool)) if contra is not None else pl.lit(False).alias("contra"))
    df = df.sort("p", descending=True, maintain_order=True).with_columns(pl.int_range(pl.len()).over("t_rid").alias("j"))
    second = df.filter(pl.col("j") == 1).select("t_rid", pl.col("p").alias("p2nd"))
    best = df.filter(pl.col("j") == 0).drop("j").join(second, on="t_rid", how="left").with_columns(pl.col("p2nd").fill_null(0.0))
    return best


def accept(best, threshold, margin=0.0, contra_penalty=0.0):
    """Acceptance mask over the rows of best_candidates(): p >= thr AND p - p2nd >= margin AND
    (no contradiction OR p >= thr + contra_penalty). thr: scalar or one value per row."""
    p = best["p"].to_numpy().astype(np.float64)
    thr = np.asarray(threshold, dtype=np.float64)
    thr = thr if thr.ndim else np.full(len(p), float(thr))
    keep = (p >= thr) & (p - best["p2nd"].to_numpy() >= margin)
    if contra_penalty > 0:
        keep &= (~best["contra"].to_numpy()) | (p >= thr + contra_penalty)
    return keep


def assign(meta, p, threshold, margin=0.0, contra_penalty=0.0, contra=None):
    """Many-to-one assignment -> DataFrame (t_rid, s_rid, p): the best Source 1 candidate of every
    Source 2/3 record that passes `accept` (threshold, margin over the runner-up, contradiction rule).

    threshold: a scalar, or one threshold per row of meta (the pair is then accepted iff its own
    probability clears its own threshold).
    contra: optional bool per row of meta (model.contradiction_flag) used by contra_penalty."""
    best = best_candidates(meta, p, contra)
    thr = threshold if np.ndim(threshold) == 0 else np.asarray(threshold, dtype=np.float64)[best["i"].to_numpy()]
    keep = accept(best, thr, margin, contra_penalty)
    return best.filter(pl.Series(keep)).select("t_rid", "s_rid", "p")


In [ ]:
%%writefile /kaggle/working/er/src/thresholds.py
"""Vectorised threshold sweep for the challenge metric (macro F0.5 over Source 1 entities).

train.py used to call assign() + macro_f05() once per threshold, i.e. one full sort of every
candidate pair and several joins per grid point. The best candidate of each Source 2/3
record does not depend on the threshold, so it is computed once (`link_table`), after which
every grid point is a couple of np.bincount calls over one row per Source 2/3 record.

A decoy weight r > 1 reproduces select_threshold.py's density adjustment: each false link
caused by an unmatched ("decoy") record counts r times, which is what happens when the test
split has r times as many decoys per Source 1 entity as training.

Thresholds and decoy weights may be scalars or arrays aligned with the rows of `links`, which
keeps the machinery general: row i is accepted iff p_i >= thr_i.

Decision rule beyond the threshold (model.assign applies the same rule at prediction time):
`margin` requires p_i - p2nd_i >= margin, where p2nd is the probability of the record's runner-up
candidate (confidence-based rejection of close calls), and `contra_penalty` raises the threshold
of a link whose pair carries a strong contradiction (model.contradiction_flag); a penalty >= 1 is
a hard veto. Both are stored in the link table columns p2nd / contra, so every sweep stays a few
np.bincount calls per grid point.
"""
import numpy as np
import polars as pl

BETA2 = 0.25


def link_table(meta, p, truth, s1_rids, contra=None):
    """meta: (t_rid, s_rid) aligned with p; truth: (t_rid, s_rid) true pairs; s1_rids: rids of ALL
    Source 1 records (the evaluation set, singletons included); contra: optional bool per row of
    meta (strong contradiction of the pair).
    Returns (links DataFrame [t_rid, s_rid, p, p2nd, contra, s_idx, tp, decoy], nt array per s_idx)."""
    from model import best_candidates  # local import: model.py imports lightgbm
    s_index = pl.DataFrame({"s_rid": s1_rids.astype(np.uint32)}).with_row_index("s_idx")
    best = (
        best_candidates(meta, p, contra).drop("i")
        .join(truth.select("t_rid", pl.col("s_rid").alias("true_s")), on="t_rid", how="left")
        .join(s_index, on="s_rid", how="left")
        .with_columns(
            (pl.col("true_s") == pl.col("s_rid")).fill_null(False).alias("tp"),
            pl.col("true_s").is_null().alias("decoy"),
        )
        .drop("true_s")
    )
    nt = np.bincount(truth.join(s_index, on="s_rid", how="inner")["s_idx"].to_numpy(), minlength=len(s1_rids))
    return best, nt


def subset_links(links, nt, entity_mask):
    """Restricts a link table to the Source 1 entities where entity_mask (bool per s_idx) is True,
    re-indexing s_idx so that the result is a self-contained (links, nt) pair."""
    entity_mask = np.asarray(entity_mask, dtype=bool)
    idx_map = -np.ones(len(entity_mask), dtype=np.int64)
    idx_map[entity_mask] = np.arange(int(entity_mask.sum()))
    s_idx = links["s_idx"].to_numpy()
    keep = entity_mask[s_idx]
    lk = links.filter(pl.Series(keep)).with_columns(pl.Series("s_idx", idx_map[s_idx[keep]]).cast(pl.UInt32))
    return lk, nt[entity_mask]


def _as_rows(x, n):
    x = np.asarray(x, dtype=np.float64)
    return x if x.ndim else np.full(n, float(x))


def accept_mask(links, thr, margin=0.0, contra_penalty=0.0):
    """Acceptance of every link row under (threshold(s), margin, contradiction penalty)."""
    p = links["p"].to_numpy().astype(np.float64)
    t = _as_rows(thr, len(p))
    keep = p >= t
    if margin > 0:
        keep &= (p - links["p2nd"].to_numpy()) >= margin
    if contra_penalty > 0 and "contra" in links.columns:
        keep &= (~links["contra"].to_numpy()) | (p >= t + contra_penalty)
    return keep


def entity_scores(links, nt, thr, decoy_weight=1.0, keep=None, margin=0.0, contra_penalty=0.0):
    """Per-entity F0.5 (challenge definition, singletons included) at threshold(s) `thr`.

    thr / decoy_weight: scalar or one value per row of `links`. `keep` overrides the acceptance
    mask (bool per row) when the caller has already computed it; margin / contra_penalty are the
    decision-rule extras of accept_mask.
    Returns (f, tp, npred, keep) with f/tp/npred per Source 1 entity and keep per link row."""
    n_s = len(nt)
    p = links["p"].to_numpy()
    if keep is None:
        keep = accept_mask(links, thr, margin, contra_penalty)
    s_idx = links["s_idx"].to_numpy()[keep]
    tp_m = links["tp"].to_numpy()[keep]
    dec_m = links["decoy"].to_numpy()[keep]
    w = _as_rows(decoy_weight, len(p))[keep]
    tp = np.bincount(s_idx, weights=tp_m, minlength=n_s)
    fp = np.bincount(s_idx, weights=(~tp_m) & (~dec_m), minlength=n_s) + np.bincount(
        s_idx, weights=w * ((~tp_m) & dec_m), minlength=n_s)
    npred = tp + fp
    ntf = nt.astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        prec = np.where(npred > 0, tp / npred, 0.0)
        rec = np.where(ntf > 0, tp / ntf, 0.0)
        f = np.where(prec + rec > 0, (1 + BETA2) * prec * rec / (BETA2 * prec + rec), 0.0)
    f = np.where((ntf == 0) & (npred == 0), 1.0, f)
    f = np.where((ntf == 0) & (npred > 0), 0.0, f)
    return f, tp, npred, keep


def metrics_at(links, nt, thr, decoy_weight=1.0, entity_mask=None, link_mask=None, margin=0.0, contra_penalty=0.0):
    """Macro F0.5 and companions at one threshold (or one threshold per link row).

    entity_mask: restrict the entity-level numbers (macro F0.5, singleton accuracy) to these
    entities; link_mask: restrict the link-level numbers (micro precision / recall over the
    linked records, links, pair accuracy) to these link rows. Both default to everything, in
    which case the result is exactly the challenge metric over all Source 1 entities.
    Also reports the singleton false positives (singletons that received a link) and the
    false-positive links split into decoy records and wrong-entity links."""
    f, tp, npred, keep = entity_scores(links, nt, thr, decoy_weight, margin=margin, contra_penalty=contra_penalty)
    ntf = nt.astype(float)
    em = np.ones(len(nt), dtype=bool) if entity_mask is None else np.asarray(entity_mask, dtype=bool)
    lm = np.ones(len(keep), dtype=bool) if link_mask is None else np.asarray(link_mask, dtype=bool)
    tp_rows = links["tp"].to_numpy()
    # micro numbers over the link rows of interest: true positives among the accepted rows; the recall
    # denominator is every true record (retrieval misses included) when no link mask is given, else the
    # matched (non-decoy) records among the masked rows
    acc = keep & lm
    n_tp = float((acc & tp_rows).sum())
    n_true_total = float(ntf[em].sum()) if link_mask is None else float((lm & ~links["decoy"].to_numpy()).sum())
    singles = (ntf[em] == 0)
    dec_rows = links["decoy"].to_numpy()
    return {
        "threshold": float(thr) if np.ndim(thr) == 0 else None,
        "macro_f05": float(f[em].mean()) if em.any() else None,
        "micro_precision": float(n_tp / max(acc.sum(), 1)),
        "micro_recall": float(n_tp / max(n_true_total, 1)),
        "singleton_acc": float(((ntf[em] == 0) & (npred[em] == 0)).sum() / max(singles.sum(), 1)),
        "singleton_fp": int(((ntf[em] == 0) & (npred[em] > 0)).sum()),
        "singletons": int(singles.sum()),
        "links": int(acc.sum()),
        "fp_decoy": int((acc & ~tp_rows & dec_rows).sum()),
        "fp_wrong_entity": int((acc & ~tp_rows & ~dec_rows).sum()),
        "pair_accuracy": float((tp_rows[lm] == keep[lm]).mean()) if lm.any() else None,
        "n_entities": int(em.sum()),
    }


def sweep(links, nt, grid, decoy_weight=1.0, margin=0.0, contra_penalty=0.0):
    return [metrics_at(links, nt, float(t), decoy_weight, margin=margin, contra_penalty=contra_penalty) for t in grid]


def sweep_f05(links, nt, grid, decoy_weight=1.0, margin=0.0, contra_penalty=0.0):
    """Macro F0.5 only, at every grid threshold: the arrays are extracted once and the rows that the
    margin rejects are dropped up front; a contradiction penalty lowers the effective probability of
    the contradicted rows, so every grid point is three np.bincount calls (fit loops call this)."""
    n_s = len(nt)
    p = links["p"].to_numpy().astype(np.float64)
    ok = np.ones(len(p), dtype=bool)
    if margin > 0:
        ok &= (p - links["p2nd"].to_numpy()) >= margin
    if contra_penalty > 0 and "contra" in links.columns:
        p = np.where(links["contra"].to_numpy(), p - contra_penalty, p)
    s_idx = links["s_idx"].to_numpy()[ok]
    tp_m = links["tp"].to_numpy()[ok]
    dec_m = links["decoy"].to_numpy()[ok]
    w = _as_rows(decoy_weight, len(links["p"]))[ok]
    p = p[ok]
    fp_w = np.where(dec_m, w, 1.0) * (~tp_m)
    ntf = nt.astype(float)
    single = ntf == 0
    out = np.empty(len(grid))
    for i, t in enumerate(grid):
        keep = p >= t
        tp = np.bincount(s_idx[keep], weights=tp_m[keep], minlength=n_s)
        fp = np.bincount(s_idx[keep], weights=fp_w[keep], minlength=n_s)
        npred = tp + fp
        with np.errstate(divide="ignore", invalid="ignore"):
            prec = np.where(npred > 0, tp / npred, 0.0)
            rec = np.where(ntf > 0, tp / ntf, 0.0)
            f = np.where(prec + rec > 0, (1 + BETA2) * prec * rec / (BETA2 * prec + rec), 0.0)
        f = np.where(single & (npred == 0), 1.0, f)
        f = np.where(single & (npred > 0), 0.0, f)
        out[i] = f.mean()
    return out


def best_f05_threshold(links, nt, grid, decoy_weight=1.0, margin=0.0, contra_penalty=0.0):
    """(threshold, macro F0.5) maximising macro F0.5 over the grid (first maximum, like best_threshold)."""
    f = sweep_f05(links, nt, grid, decoy_weight, margin, contra_penalty)
    i = int(np.argmax(f))
    return float(grid[i]), float(f[i])


DEFAULT_MARGINS = (0.0, 0.1, 0.3, 0.5, 0.7)
DEFAULT_PENALTIES = (0.0, 0.1, 0.3)  # a hard veto (penalty >= 1) lost by 0.02 wherever it was tried; pass it explicitly to re-check


def nested_rule_f05(links, nt, s_fold, grid, margin=0.0, contra_penalty=0.0, decoy_weight=1.0):
    """Leak-free macro F0.5 of a decision rule: the threshold is fit on the entities of K-1 folds and
    applied to the held-out fold; the concatenated held-out links give one number. Returns
    (nested macro F0.5, per-fold thresholds)."""
    s_fold = np.asarray(s_fold)
    link_fold = s_fold[links["s_idx"].to_numpy()]
    thr = np.full(links.height, np.nan)
    fold_thr = []
    dw = np.asarray(decoy_weight, dtype=np.float64)
    for k in np.unique(s_fold):
        lk, nt_k = subset_links(links, nt, s_fold != k)
        dw_k = dw[link_fold != k] if dw.ndim else float(dw)
        t, _ = best_f05_threshold(lk, nt_k, grid, dw_k, margin, contra_penalty)
        thr[link_fold == k] = t
        fold_thr.append(float(t))
    return metrics_at(links, nt, thr, decoy_weight, margin=margin, contra_penalty=contra_penalty)["macro_f05"], fold_thr


def select_rule(links, nt, s_fold, grid, margins=DEFAULT_MARGINS, penalties=DEFAULT_PENALTIES, decoy_weight=1.0, min_gain=1e-4):
    """Chooses (margin, contra_penalty) by nested macro F0.5; the plain threshold rule is kept unless
    a richer rule beats it by at least min_gain. Returns (rule dict, table of all rules)."""
    rows = []
    for m in margins:
        for c in penalties:
            f, ft = nested_rule_f05(links, nt, s_fold, grid, m, c, decoy_weight)
            rows.append({"margin": float(m), "contra_penalty": float(c), "nested_macro_f05": float(f), "fold_thresholds": ft})
    base = next(r for r in rows if r["margin"] == 0.0 and r["contra_penalty"] == 0.0)
    best = max(rows, key=lambda r: r["nested_macro_f05"])
    chosen = best if best["nested_macro_f05"] - base["nested_macro_f05"] >= min_gain else base
    rule = {"margin": chosen["margin"], "contra_penalty": chosen["contra_penalty"], "nested_macro_f05": chosen["nested_macro_f05"],
            "nested_macro_f05_threshold_only": base["nested_macro_f05"], "gain": chosen["nested_macro_f05"] - base["nested_macro_f05"],
            "min_gain": min_gain}
    return rule, rows


def best_threshold(rows):
    return max(rows, key=lambda r: r["macro_f05"])


def default_grid():
    return np.round(np.arange(0.05, 0.99, 0.01), 2)


In [ ]:
%%writefile /kaggle/working/er/src/calibrate.py
"""Probability calibration of the matcher output, fit on out-of-fold predictions only.

Why: LightGBM scores are ranked well but are not probabilities. Two consumers need real
probabilities: (1) the expected-F0.5 decoder (decode.py) and (2) any threshold transfer to a
split whose positive/negative mix differs from training (the test split has ~2x the decoy
density). Only the OOF predictions (train.py) are used to fit anything here; the model
that produced a prediction never saw the record.

Methods
  platt    : p' = sigmoid(a * logit(p) + b), two parameters fit by Newton iterations on the
             log-loss (the standard Platt scaling, with logit(p) as the score).
  isotonic : monotone step function (PAV); more flexible, may overfit small folds.
  prior    : prior-shift correction of a calibrated probability when the positive rate moves
             from pi_train to pi_test:  logit(p') = logit(p) + log(odds_test / odds_train).
             It is applied on top of platt/isotonic at prediction time (see shift_odds).

Metrics: log loss, Brier score, expected calibration error (ECE, 15 equal-width bins) and
a reliability table. `nested_calibration_report` fits on K-1 folds and evaluates on the
held-out fold, so the reported numbers are themselves out-of-sample.
"""
import json

import numpy as np

EPS = 1e-6
LOGIT_CLIP = 12.0  # |logit| cap for the Platt input: raw scores at 1e-6 are not more informative than at 1e-5


def logit(p):
    p = np.clip(np.asarray(p, dtype=np.float64), EPS, 1 - EPS)
    return np.log(p / (1 - p))


def _log_loss(z, y):
    # numerically stable mean log-loss of logits z
    return float(np.mean(np.logaddexp(0.0, -z) * y + np.logaddexp(0.0, z) * (1 - y)))


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


# ------------------------------------------------------------------ Platt
def fit_platt(p, y, iters=100, l2=1e-3):
    """Damped Newton's method on the 2-parameter logistic regression y ~ sigmoid(a*logit(p)+b).

    Plain Newton steps overshoot on (near-)separable data - the score of a good matcher is
    close to separable - and diverge to |a| ~ 1e10. Each step is therefore halved until the
    (L2-regularised, towards a=1 / b=0) log-loss decreases, and the identity calibration is
    returned if the fit does not beat it. Input logits are clipped to +-LOGIT_CLIP."""
    x = np.clip(logit(p), -LOGIT_CLIP, LOGIT_CLIP)
    y = np.asarray(y, dtype=np.float64)
    n = len(y)

    def objective(a, b):
        return _log_loss(a * x + b, y) + 0.5 * l2 * ((a - 1) ** 2 + b ** 2) / n

    a, b = 1.0, 0.0
    cur = objective(a, b)
    for _ in range(iters):
        q = sigmoid(a * x + b)
        w = q * (1 - q) + 1e-12
        g = np.array([np.sum((q - y) * x) + l2 * (a - 1), np.sum(q - y) + l2 * b]) / n
        h = np.array([[np.sum(w * x * x) + l2, np.sum(w * x)], [np.sum(w * x), np.sum(w) + l2]]) / n
        step = np.linalg.solve(h, g)
        t = 1.0
        while t > 1e-6:  # backtracking line search
            na, nb = a - t * step[0], b - t * step[1]
            new = objective(na, nb)
            if new < cur:
                break
            t *= 0.5
        else:
            break
        moved = max(abs(na - a), abs(nb - b))
        a, b, cur = na, nb, new
        if moved < 1e-9:
            break
    cal = {"method": "platt", "a": float(a), "b": float(b), "log_loss": cur, "identity_log_loss": objective(1.0, 0.0)}
    if not np.isfinite(cur) or cur > cal["identity_log_loss"] or not (1e-3 < abs(a) < 1e3):
        cal.update({"a": 1.0, "b": 0.0, "fallback": "identity"})
    return cal


# ------------------------------------------------------------------ isotonic (PAV)
def fit_isotonic(p, y, n_bins=2000):
    """Pool-adjacent-violators on score-quantile bins (bins keep it O(n) in memory)."""
    p = np.asarray(p, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    order = np.argsort(p)
    p, y = p[order], y[order]
    edges = np.linspace(0, len(p), n_bins + 1).astype(int)
    xs, ys, ws = [], [], []
    for a, b in zip(edges[:-1], edges[1:]):
        if b > a:
            xs.append(p[a:b].mean())
            ys.append(y[a:b].mean())
            ws.append(b - a)
    # PAV
    vals, wts, lo = [], [], []
    for x, v, w in zip(xs, ys, ws):
        vals.append(v)
        wts.append(w)
        lo.append(x)
        while len(vals) > 1 and vals[-2] > vals[-1]:
            v2 = (vals[-2] * wts[-2] + vals[-1] * wts[-1]) / (wts[-2] + wts[-1])
            vals[-2:] = [v2]
            wts[-2:] = [wts[-2] + wts[-1]]
            lo[-2:] = [lo[-2]]
    return {"method": "isotonic", "x": [float(v) for v in lo], "y": [float(v) for v in vals]}


def apply(cal, p):
    p = np.asarray(p, dtype=np.float64)
    if cal is None or cal.get("method") == "identity":
        return p
    if cal["method"] == "platt":
        return sigmoid(cal["a"] * np.clip(logit(p), -LOGIT_CLIP, LOGIT_CLIP) + cal["b"])
    if cal["method"] == "isotonic":
        x, y = np.asarray(cal["x"]), np.asarray(cal["y"])
        idx = np.clip(np.searchsorted(x, p, side="right") - 1, 0, len(y) - 1)
        return y[idx]
    raise ValueError(cal["method"])


def shift_odds(p, odds_ratio):
    """Prior-shift correction: multiply the odds by odds_ratio = (pi_te/(1-pi_te)) / (pi_tr/(1-pi_tr))."""
    return sigmoid(logit(p) + np.log(odds_ratio))


# ------------------------------------------------------------------ metrics
def calibration_metrics(p, y, n_bins=15):
    p = np.clip(np.asarray(p, dtype=np.float64), EPS, 1 - EPS)
    y = np.asarray(y, dtype=np.float64)
    ll = float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
    brier = float(np.mean((p - y) ** 2))
    bins = np.minimum((p * n_bins).astype(int), n_bins - 1)
    table, ece = [], 0.0
    for b in range(n_bins):
        m = bins == b
        if m.any():
            conf, acc, n = float(p[m].mean()), float(y[m].mean()), int(m.sum())
            ece += abs(conf - acc) * n / len(p)
            table.append({"bin": b, "n": n, "mean_pred": round(conf, 4), "frac_pos": round(acc, 4)})
    return {"log_loss": ll, "brier": brier, "ece": float(ece), "reliability": table}


def nested_calibration_report(p, y, folds, methods=("platt", "isotonic")):
    """Fit each method on K-1 folds of the OOF predictions, evaluate on the held-out fold."""
    out = {"uncalibrated": calibration_metrics(p, y)}
    for m in methods:
        q = np.empty_like(p, dtype=np.float64)
        for k in np.unique(folds):
            tr, te = folds != k, folds == k
            cal = fit_platt(p[tr], y[tr]) if m == "platt" else fit_isotonic(p[tr], y[tr])
            q[te] = apply(cal, p[te])
        out[m] = calibration_metrics(q, y)
    for k, v in out.items():
        v.pop("reliability", None) if k != "uncalibrated" and k != "platt" else None
    return out


def save(cal, path):
    with open(path, "w") as f:
        json.dump(cal, f)


def load(path):
    with open(path) as f:
        return json.load(f)


In [ ]:
%%writefile /kaggle/working/er/src/checkpoint.py
"""Training checkpoints: everything one trained model needs to be re-evaluated later.

Layout: <work>/checkpoints/<name>/
    manifest.json        what was trained, on which data, with which arguments (+ git commit, timing)
    stage1_fold<k>.txt   cross-fitted fold models (written as soon as each one is trained: a crashed
    stage2_fold<k>.txt   run resumes from them with train.py --resume)
    oof_stage1.npy       out-of-fold stage-1 probabilities (resume point for stage 2)
    stage1.txt           final models (refit on the whole sample)
    stage2.txt
    calibration.json     Platt calibrator fit on the OOF predictions
    model_meta.json      feature lists, the OOF-optimal global threshold, feature importances
    oof.parquet          OOF predictions of every training pair (p1, p2, p2_cal, folds, label)
    validation_report.json
    thresholds/          threshold policies of this checkpoint (train.py, select_threshold.py; selected.json is applied)
    experiments.jsonl    one line per threshold-tuning / hold-out evaluation run on this checkpoint

<work>/checkpoints/LATEST names the most recent checkpoint. The top-level files that older
tools read (<work>/stage1.txt, model_meta.json, calibration.json, train/oof.parquet, ...) are
links to the latest checkpoint, so every existing script keeps working.
"""
import datetime as dt
import json
import os
import shutil
import subprocess

FINAL_FILES = ["stage1.txt", "stage2.txt", "calibration.json", "model_meta.json", "validation_report.json"]


def _git_commit():
    try:
        root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
        out = subprocess.run(["git", "-C", root, "rev-parse", "--short", "HEAD"], capture_output=True, text=True, timeout=5)
        dirty = subprocess.run(["git", "-C", root, "status", "--porcelain", "--untracked-files=no"], capture_output=True, text=True, timeout=5)
        return (out.stdout.strip() + ("+dirty" if dirty.stdout.strip() else "")) if out.returncode == 0 else None
    except (OSError, subprocess.SubprocessError):
        return None


def data_fingerprint(data_dir):
    """Sizes + mtimes of the challenge files: enough to notice that a checkpoint was trained on other data."""
    fp = {}
    for split in ("train", "test"):
        d = os.path.join(data_dir, split)
        if os.path.isdir(d):
            for f in sorted(os.listdir(d)):
                if f.endswith(".tsv"):
                    st = os.stat(os.path.join(d, f))
                    fp[f"{split}/{f}"] = {"bytes": st.st_size, "mtime": int(st.st_mtime)}
    return fp


def link_or_copy(src, dst):
    """Relative symlink dst -> src (copy where symlinks are unavailable); replaces whatever was at dst."""
    if os.path.lexists(dst):
        os.remove(dst)
    os.makedirs(os.path.dirname(os.path.abspath(dst)), exist_ok=True)
    try:
        os.symlink(os.path.relpath(src, os.path.dirname(os.path.abspath(dst))), dst)
    except (OSError, NotImplementedError):
        shutil.copy2(src, dst)


class Checkpoint:
    def __init__(self, work_dir, name):
        self.work_dir, self.name = work_dir, name
        self.dir = os.path.join(work_dir, "checkpoints", name)

    # ---- lookup
    @staticmethod
    def default_name():
        return dt.datetime.now().strftime("ckpt_%Y%m%d_%H%M%S")

    @classmethod
    def resolve(cls, work_dir, name=None):
        """name: a checkpoint name, a path to a checkpoint directory, or None for LATEST."""
        if name and os.path.isdir(name) and os.path.exists(os.path.join(name, "manifest.json")):
            return cls(os.path.dirname(os.path.dirname(os.path.abspath(name))), os.path.basename(os.path.normpath(name)))
        if name is None:
            latest = os.path.join(work_dir, "checkpoints", "LATEST")
            if not os.path.exists(latest):
                return None
            with open(latest) as f:
                name = f.read().strip()
        return cls(work_dir, name)

    @classmethod
    def list(cls, work_dir):
        root = os.path.join(work_dir, "checkpoints")
        if not os.path.isdir(root):
            return []
        out = []
        for n in sorted(os.listdir(root)):
            c = cls(work_dir, n)
            if c.exists():
                out.append(c)
        return out

    def exists(self):
        return os.path.exists(self.path("manifest.json"))

    def path(self, fname):
        return os.path.join(self.dir, fname)

    def has(self, fname):
        return os.path.exists(self.path(fname))

    # ---- manifest
    def manifest(self):
        if not self.exists():
            return {}
        with open(self.path("manifest.json")) as f:
            return json.load(f)

    def update_manifest(self, **fields):
        os.makedirs(self.dir, exist_ok=True)
        m = self.manifest()
        if not m:
            m = {"name": self.name, "created": dt.datetime.now().isoformat(timespec="seconds"), "git_commit": _git_commit(), "stages_done": []}
        m.update(fields)
        m["updated"] = dt.datetime.now().isoformat(timespec="seconds")
        m["files"] = {f: os.path.getsize(self.path(f)) for f in sorted(os.listdir(self.dir)) if os.path.isfile(self.path(f))}
        tmp = self.path("manifest.json.tmp")
        with open(tmp, "w") as f:
            json.dump(m, f, indent=1)
        os.replace(tmp, self.path("manifest.json"))
        return m

    def mark_stage(self, stage):
        m = self.manifest()
        done = list(m.get("stages_done", []))
        if stage not in done:
            done.append(stage)
        self.update_manifest(stages_done=done)

    def stage_done(self, stage):
        return stage in self.manifest().get("stages_done", [])

    # ---- models
    def save_model(self, booster, fname):
        os.makedirs(self.dir, exist_ok=True)
        tmp = self.path(fname + ".tmp")
        booster.save_model(tmp)
        os.replace(tmp, self.path(fname))

    def load_model(self, fname):
        import lightgbm as lgb
        return lgb.Booster(model_file=self.path(fname))

    # ---- experiments log
    def log_experiment(self, record):
        os.makedirs(self.dir, exist_ok=True)
        record = {"time": dt.datetime.now().isoformat(timespec="seconds"), "checkpoint": self.name, **record}
        with open(self.path("experiments.jsonl"), "a") as f:
            f.write(json.dumps(record) + "\n")

    def experiments(self):
        if not self.has("experiments.jsonl"):
            return []
        with open(self.path("experiments.jsonl")) as f:
            return [json.loads(line) for line in f if line.strip()]

    # ---- publish as the current model
    def set_latest(self):
        """Marks this checkpoint as LATEST and points the legacy top-level files at it."""
        os.makedirs(os.path.join(self.work_dir, "checkpoints"), exist_ok=True)
        with open(os.path.join(self.work_dir, "checkpoints", "LATEST"), "w") as f:
            f.write(self.name + "\n")
        for fname in FINAL_FILES:
            if self.has(fname):
                link_or_copy(self.path(fname), os.path.join(self.work_dir, fname))
        if self.has("oof.parquet"):
            link_or_copy(self.path("oof.parquet"), os.path.join(self.work_dir, "train", "oof.parquet"))
        if self.has(os.path.join("thresholds", "selected.json")):
            link_or_copy(self.path(os.path.join("thresholds", "selected.json")), os.path.join(self.work_dir, "threshold_policy.json"))

    def summary(self):
        m = self.manifest()
        return {"name": self.name, "created": m.get("created"), "git_commit": m.get("git_commit"), "stages_done": m.get("stages_done"),
                "train_args": m.get("train_args"), "oof_macro_f05": m.get("oof_macro_f05"), "global_threshold": m.get("global_threshold")}


In [ ]:
%%writefile /kaggle/working/er/src/threshold_policy.py
"""The decision policy of the matcher: one global acceptance threshold on the calibrated probability,
plus the decision-rule extras shared by every link (margin over the runner-up candidate and the
contradiction penalty, see thresholds.accept_mask / model.assign).

    policy = ThresholdPolicy(0.76, margin=0.0, contra_penalty=0.0)
    policy.save(path); policy = ThresholdPolicy.load(path)
    links = assign_with_policy(meta, p, policy, contra)

Per-class thresholds (one threshold per country / source / script ...) were evaluated in
docs/REPORT.md section 13 and never beat the global threshold outside the fold-to-fold noise, so
the machinery was removed (v5.1); the global, decoy-density-adjusted threshold of
select_threshold.py is the policy every run ships with.
"""
import json
import os

import numpy as np

POLICY_VERSION = 3  # 3: global threshold + decision rule only


class ThresholdPolicy:
    def __init__(self, default, scale="calibrated", name=None, fit=None, margin=0.0, contra_penalty=0.0, **_ignored):
        self.default = float(default)
        self.scale = scale
        self.name = name or "global"
        self.fit = fit or {}
        self.margin = float(margin or 0.0)
        self.contra_penalty = float(contra_penalty or 0.0)

    @property
    def threshold(self):
        return self.default

    @property
    def rule(self):
        return {"margin": self.margin, "contra_penalty": self.contra_penalty}

    def to_dict(self):
        return {"version": POLICY_VERSION, "name": self.name, "scheme": "global", "scale": self.scale, "default": self.default,
                "margin": self.margin, "contra_penalty": self.contra_penalty, "fit": self.fit}

    @classmethod
    def from_dict(cls, d):
        return cls(d["default"], d.get("scale", "calibrated"), d.get("name"), d.get("fit"), d.get("margin", 0.0), d.get("contra_penalty", 0.0))

    def save(self, path):
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        with open(path, "w") as f:
            json.dump(self.to_dict(), f, indent=1)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            return cls.from_dict(json.load(f))

    def describe(self):
        extra = f" (margin {self.margin:.2f}, contradiction penalty {self.contra_penalty:.2f})" if (self.margin or self.contra_penalty) else ""
        return f"{self.name}: global threshold {self.default:.3f}{extra}"


def assign_with_policy(meta, p, policy, contra=None):
    """Links (t_rid, s_rid, p) accepted under a policy (the same rule as model.assign)."""
    from model import assign  # local import: model.py imports lightgbm, which the tools do not need
    return assign(meta, np.asarray(p), policy.default, policy.margin, policy.contra_penalty, contra)


In [ ]:
%%writefile /kaggle/working/er/src/train.py
"""Step 4: train the two-stage matcher on the training split, with leak-free validation.

Folds come from folds.py (records.parquet carries `fold` / `group`): all namesakes of a
Source 1 entity share its fold, matched Source 2/3 records inherit it, and unmatched
("decoy") records are put in the fold of their best blocking candidate so that nearly every
candidate pair is inside one fold.

* The evaluation fold of a candidate pair is the fold of its Source 1 entity, so every
  entity's whole candidate list is scored by one model.
* The model for fold k is trained on pairs whose Source 1 entity AND Source 2/3 record are
  both outside fold k (strict exclusion; the remaining "cross-fold" pairs are counted and
  reported in validation_report.json).
* Stage 1 and stage 2 are cross-fitted; stage 2 uses out-of-fold stage-1 probabilities.
* The stage-2 OOF probabilities are calibrated (Platt, calibrate.py) - fit on OOF only -
  and the acceptance threshold is chosen on the calibrated scale by maximising macro F0.5
  over ALL training Source 1 entities (singletons included). The sweep is vectorised
  (thresholds.py) instead of re-sorting every pair per grid point.
* Decision rule (thresholds.select_rule, --rule-search): besides the threshold, a margin over the
  record's runner-up candidate and a threshold penalty for contradicted pairs can be considered; the
  rule is chosen by *nested* macro F0.5 (threshold fit on K-1 folds, scored on the held-out fold) and
  only kept when it beats the plain threshold by --rule-min-gain. Off by default: on the 5 % subset no
  rule beat the plain threshold by more than the fold noise and the search cost ~2 min per run. The
  report always carries the nested macro F0.5 of the plain threshold (a few seconds), so the headline
  number is one the thresholds were not tuned on.
* Reporting: macro F0.5, singleton false positives, false positives by kind, per-source
  (Source 2 vs 3) and per-country metrics, and the candidate-generation audit of blocking.py.
* Per-fold metrics, threshold stability across folds, calibration metrics and the
  cross-fold audit go to <work>/validation_report.json; OOF predictions to
  <work>/train/oof.parquet; final models (refit on all sampled rows) to <work>/stage*.txt.
* Checkpointing (checkpoint.py): every artefact of the run is written to
  <work>/checkpoints/<name>/ - each cross-fitted fold model as soon as it is trained, the
  stage-1 OOF probabilities, the final models, calibration, OOF predictions, report and a
  manifest (arguments, data fingerprint, git commit). `--resume` continues an interrupted run
  from the last saved fold model. The top-level <work>/ files are links to the latest
  checkpoint, so one checkpoint can be evaluated under any number of threshold policies
  (select_threshold.py / predict.py --checkpoint --policy) without retraining.
"""
import json
import os
import shutil

import numpy as np
import polars as pl

import calibrate
from checkpoint import Checkpoint, data_fingerprint
from common import Stage, base_args, left_join_ordered, log, split_dir
from model import CONTRA_COLS, contradiction_flag, house_numbers, iter_parts, part_files, stage1_features, stage2_context, to_np, train_lgb
from pair_features import truth_pairs
from threshold_policy import ThresholdPolicy
from thresholds import best_threshold, default_grid, link_table, metrics_at, nested_rule_f05, select_rule, subset_links, sweep

TRAIN_ARGS = ("rounds1", "rounds2", "sample_rows", "seed", "drop_features", "no_stage2", "tag")
NO_RULE = {"margin": 0.0, "contra_penalty": 0.0}


def oof_predict(files, feats_fn, models, eval_fold, n):
    """Streams the parts once; each row is scored by the model that did not see its fold."""
    out = np.zeros(n, dtype=np.float32)
    for part in files:
        df = pl.read_parquet(part)
        pid = df["pid"].to_numpy()
        X = feats_fn(df, pid)
        fk = eval_fold[pid]
        for k, m in models.items():
            sel = fk == k
            if sel.any():
                out[pid[sel]] = m.predict(X[sel])
    return out


def assign_folds(meta, rec):
    """Adds s_fold (eval fold), t_fold and the strict train mask helpers to meta."""
    f = rec.select(pl.col("rid").cast(pl.UInt32), "fold")
    meta = left_join_ordered(meta, f.rename({"rid": "s_rid", "fold": "s_fold"}), "s_rid")
    meta = left_join_ordered(meta, f.rename({"rid": "t_rid", "fold": "t_fold"}), "t_rid")
    # decoys (t_fold == -1): fold of the best-scoring candidate of the record
    best_s_fold = (
        meta.filter(pl.col("t_fold") < 0)
        .sort("score", descending=True)
        .unique("t_rid", keep="first")
        .select("t_rid", pl.col("s_fold").alias("decoy_fold"))
    )
    meta = left_join_ordered(meta, best_s_fold, "t_rid").with_columns(
        pl.when(pl.col("t_fold") < 0).then(pl.col("decoy_fold")).otherwise(pl.col("t_fold")).cast(pl.Int8).alias("t_fold")
    ).drop("decoy_fold")
    return meta


def fold_report(links, nt, s_fold_of_s1, thr, grid, rule=NO_RULE):
    """Per-fold macro F0.5 at the global threshold, plus each fold's own best threshold."""
    rows = []
    for k in np.unique(s_fold_of_s1):
        sel_s = s_fold_of_s1 == k
        lk, nt_k = subset_links(links, nt, sel_s)
        at = metrics_at(lk, nt_k, thr, **rule)
        own = best_threshold(sweep(lk, nt_k, grid, **rule))
        rows.append({"fold": int(k), "n_entities": int(sel_s.sum()), **{f"{a}_at_global_thr": b for a, b in at.items() if a != "threshold"},
                     "best_threshold": own["threshold"], "macro_f05_at_own_thr": own["macro_f05"]})
    f = np.array([r["macro_f05_at_global_thr"] for r in rows])
    t = np.array([r["best_threshold"] for r in rows])
    return {"folds": rows, "macro_f05_mean": float(f.mean()), "macro_f05_std": float(f.std(ddof=0)),
            "macro_f05_min": float(f.min()), "best_threshold_std": float(t.std(ddof=0)), "best_threshold_range": [float(t.min()), float(t.max())]}


def per_source_report(links, nt, thr, rec, rule=NO_RULE):
    """Link-level precision / recall and false positives per Source 2/3 record source (entity-level
    macro F0.5 cannot be split by source: one entity receives records of both sources)."""
    src = links.select("t_rid").join(rec.select(pl.col("rid").cast(pl.UInt32).alias("t_rid"), "src"), on="t_rid", how="left")["src"].to_numpy()
    out = {}
    for s in sorted(set(src.tolist())):
        lm = src == s
        r = metrics_at(links, nt, thr, link_mask=lm, **rule)
        out[f"source{int(s)}"] = {k: r[k] for k in ("micro_precision", "micro_recall", "links", "fp_decoy", "fp_wrong_entity", "pair_accuracy")}
        out[f"source{int(s)}"]["link_rows"] = int(lm.sum())
    return out


def main():
    ap = base_args(__doc__)
    ap.add_argument("--rounds1", type=int, default=300)
    ap.add_argument("--rounds2", type=int, default=200)
    ap.add_argument("--sample-rows", type=int, default=5_000_000)
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--drop-features", default="", help="comma separated feature names to exclude (ablations)")
    ap.add_argument("--no-stage2", action="store_true", help="ablation: use stage-1 probabilities directly")
    ap.add_argument("--rule-search", action="store_true", help="also search margin / contradiction-penalty rules (nested); default: plain threshold")
    ap.add_argument("--rule-min-gain", type=float, default=1e-4, help="nested macro F0.5 gain a margin / contradiction rule needs")
    ap.add_argument("--tag", default="", help="suffix for the report / oof file names (ablations)")
    ap.add_argument("--checkpoint-name", default=None,
                    help="name of the checkpoint directory under <work>/checkpoints (default: ckpt_<timestamp>[_<tag>])")
    ap.add_argument("--resume", action="store_true",
                    help="reuse the fold models / stage-1 OOF already saved in the checkpoint (same arguments required)")
    ap.add_argument("--overwrite", action="store_true", help="replace an existing checkpoint of the same name")
    args = ap.parse_args()
    with Stage(args.work_dir, "train" + (f"_{args.tag}" if args.tag else "")):
        run(args)


def open_checkpoint(args):
    """Creates (or, with --resume, reopens) the checkpoint of this run and records its arguments."""
    name = args.checkpoint_name or (Checkpoint.default_name() + (f"_{args.tag}" if args.tag else ""))
    ckpt = Checkpoint(args.work_dir, name)
    train_args = {k: getattr(args, k) for k in TRAIN_ARGS}
    if ckpt.exists() and args.overwrite and not args.resume:
        log(f"overwriting checkpoint {ckpt.dir}")
        shutil.rmtree(ckpt.dir)
    if ckpt.exists():
        prev = ckpt.manifest().get("train_args")
        if not args.resume:
            raise SystemExit(f"checkpoint {ckpt.dir} exists; --resume continues it, --overwrite replaces it, or choose another --checkpoint-name")
        if prev != train_args:
            raise SystemExit(f"cannot resume {ckpt.dir}: it was trained with {prev}, now {train_args}")
        log(f"resuming checkpoint {ckpt.dir}: stages done {ckpt.manifest().get('stages_done')}")
    ckpt.update_manifest(train_args=train_args, data_dir=os.path.abspath(args.data_dir), data=data_fingerprint(args.data_dir),
                         resumed=bool(args.resume and ckpt.exists()))
    return ckpt


def load_or_train(ckpt, fname, resume, fn):
    """Fold model from the checkpoint when resuming, else trained by fn() and saved immediately."""
    if resume and ckpt.has(fname):
        log(f"  {fname}: loaded from checkpoint")
        return ckpt.load_model(fname)
    m = fn()
    ckpt.save_model(m, fname)
    return m


def run(args):
    d = split_dir(args.work_dir, "train")
    ckpt = open_checkpoint(args)
    files = part_files(d)
    rec = pl.read_parquet(os.path.join(d, "records.parquet"), columns=["rid", "entity_id", "src", "fold", "country"])
    truth = truth_pairs(rec, args.data_dir).drop("label")
    s1 = rec.filter(pl.col("src") == 1)
    s1_rids = s1["rid"].to_numpy().astype(np.uint32)
    s1_fold = s1["fold"].to_numpy()

    meta = pl.concat(list(iter_parts(files, ["pid", "t_rid", "s_rid", "label", "score", *CONTRA_COLS]))).sort("pid")
    n = meta.height
    assert meta["pid"][-1] == n - 1
    contra = contradiction_flag(meta)
    meta = assign_folds(meta, rec).drop(CONTRA_COLS)
    s_fold = meta["s_fold"].to_numpy()
    t_fold = meta["t_fold"].to_numpy()
    y_all = meta["label"].to_numpy()
    fold_ids = sorted(int(k) for k in np.unique(s_fold))
    cross = s_fold != t_fold
    log(f"pairs {n}, positives {int(y_all.sum())}, eval-fold sizes {np.bincount(s_fold)}, "
        f"cross-fold pairs {cross.mean():.4f} (positives among them {int(y_all[cross].sum())})")

    drop = {x for x in args.drop_features.split(",") if x}
    f1 = [c for c in stage1_features(files) if c not in drop]
    log("stage-1 features", len(f1), "dropped", sorted(drop))
    rng = np.random.default_rng(args.seed)
    in_sample = np.zeros(n, dtype=bool)
    in_sample[rng.choice(n, min(args.sample_rows, n), replace=False)] = True
    parts = []
    for df in iter_parts(files, ["pid"] + f1):
        parts.append(df.filter(pl.Series(in_sample[df["pid"].to_numpy()])))
    sample = pl.concat(parts).sort("pid")
    del parts
    s_pid = sample["pid"].to_numpy()
    X1 = to_np(sample, f1)
    del sample
    ys, sf, tf = y_all[s_pid], s_fold[s_pid], t_fold[s_pid]
    log("sample", X1.shape, "positive rate", ys.mean())

    def train_mask(k):
        return (sf != k) & (tf != k)  # strict: neither side of the pair is in the held-out fold

    # ---------------- stage 1 (OOF)
    m1s = {}
    for k in fold_ids:
        m = train_mask(k)
        m1s[k] = load_or_train(ckpt, f"stage1_fold{k}.txt", args.resume, lambda: train_lgb(X1[m], ys[m], args.rounds1))
        log(f"stage-1 fold {k} ready ({int(m.sum())} training pairs)")
    if args.resume and ckpt.has("oof_stage1.npy") and ckpt.stage_done("stage1"):
        p1 = np.load(ckpt.path("oof_stage1.npy"))
        assert len(p1) == n, "checkpointed stage-1 OOF does not match the pair table"
        log("stage-1 OOF loaded from checkpoint")
    else:
        p1 = oof_predict(files, lambda df, pid: to_np(df, f1), m1s, s_fold, n)
        np.save(ckpt.path("oof_stage1.npy"), p1)
    ckpt.mark_stage("stage1")
    links1, nt = link_table(meta, p1, truth, s1_rids, contra)
    s1_best = best_threshold(sweep(links1, nt, default_grid()))
    log("stage-1 OOF", {k: s1_best[k] for k in ("threshold", "macro_f05", "micro_precision", "micro_recall")})

    # ---------------- stage 2 (OOF) on stage-1 OOF context
    if args.no_stage2:
        p2, f2, C = p1, f1, None
        m2s = None
    else:
        C, cnames = stage2_context(meta, p1, house_numbers(d))
        keep_c = [i for i, c in enumerate(cnames) if c not in drop]  # --drop-features also applies to the context features
        C, cnames = np.ascontiguousarray(C[:, keep_c]), [cnames[i] for i in keep_c]
        f2 = f1 + ["p1"] + cnames
        X2 = np.hstack([X1, p1[s_pid, None], C[s_pid]])
        m2s = {}
        for k in fold_ids:
            m = train_mask(k)
            m2s[k] = load_or_train(ckpt, f"stage2_fold{k}.txt", args.resume, lambda: train_lgb(X2[m], ys[m], args.rounds2))
            log(f"stage-2 fold {k} ready")
        p2 = oof_predict(files, lambda df, pid: np.hstack([to_np(df, f1), p1[pid, None], C[pid]]), m2s, s_fold, n)
    ckpt.mark_stage("stage2")

    # ---------------- calibration (fit on OOF only) + threshold on the calibrated scale
    cal = calibrate.fit_platt(p2, y_all)
    cal_report = calibrate.nested_calibration_report(p2, y_all, s_fold)
    p2c = calibrate.apply(cal, p2).astype(np.float32)
    log(f"Platt a={cal['a']:.3f} b={cal['b']:.3f}; nested calibration: " +
        ", ".join(f"{k}: ll={v['log_loss']:.4f} brier={v['brier']:.5f} ece={v['ece']:.4f}" for k, v in cal_report.items()))
    grid = default_grid()
    links, nt = link_table(meta, p2c, truth, s1_rids, contra)
    # ---------------- decision rule (margin over the runner-up, contradiction penalty): nested selection
    if not args.rule_search:
        nested_f, fold_thr = nested_rule_f05(links, nt, s1_fold, grid)
        rule_sel, rule_rows = dict(NO_RULE, nested_macro_f05=nested_f, nested_macro_f05_threshold_only=nested_f, fold_thresholds=fold_thr), []
    else:
        rule_sel, rule_rows = select_rule(links, nt, s1_fold, grid, min_gain=args.rule_min_gain)
        log("decision rules (nested macro F0.5):", [(r["margin"], r["contra_penalty"], round(r["nested_macro_f05"], 5)) for r in rule_rows])
    rule = {"margin": rule_sel["margin"], "contra_penalty": rule_sel["contra_penalty"]}
    log(f"decision rule: {rule} (nested macro F0.5 {rule_sel['nested_macro_f05']:.5f} vs threshold-only {rule_sel['nested_macro_f05_threshold_only']:.5f})")
    rows = sweep(links, nt, grid, **rule)
    best = best_threshold(rows)
    best_plain = best_threshold(sweep(links, nt, grid)) if rule != NO_RULE else best
    log("stage-2 OOF best", {k: best[k] for k in ("threshold", "macro_f05", "micro_precision", "micro_recall", "singleton_fp", "fp_decoy", "fp_wrong_entity")})
    folds_rep = fold_report(links, nt, s1_fold, best["threshold"], grid, rule)
    log("per-fold macro F0.5 at global thr", [round(r["macro_f05_at_global_thr"], 4) for r in folds_rep["folds"]],
        "std", round(folds_rep["macro_f05_std"], 5), "fold-best thresholds", [r["best_threshold"] for r in folds_rep["folds"]])
    # metrics on the entities whose candidate lists are entirely within their fold (no cross-fold pair)
    within_s = np.ones(len(s1_rids), dtype=bool)
    s_index = pl.DataFrame({"s_rid": s1_rids}).with_row_index("s_idx")
    cross_s = meta.filter(pl.Series(cross)).select("s_rid").unique().join(s_index, on="s_rid")["s_idx"].to_numpy()
    within_s[cross_s] = False
    audit = {"cross_fold_pair_share": float(cross.mean()), "entities_with_cross_fold_pair": float((~within_s).mean())}
    if within_s.any() and (~within_s).any():
        for name, sel in (("within_fold_only", within_s), ("with_cross_fold_pairs", ~within_s)):
            audit[name] = metrics_at(*subset_links(links, nt, sel), best["threshold"], **rule)
    # per-country (entity level) and per-source (link level)
    country = s1["country"].fill_null("").to_numpy()
    per_country = {}
    for c in sorted(set(country.tolist())):
        per_country[c] = metrics_at(*subset_links(links, nt, country == c), best["threshold"], **rule)
    per_source = per_source_report(links, nt, best["threshold"], rec, rule)
    # pair-level metrics: why "accuracy" looks great while F0.5 does not
    pred_pos = p2c >= best["threshold"]
    pair = {"accuracy": float((pred_pos == (y_all == 1)).mean()), "positive_rate": float(y_all.mean()),
            "precision": float((pred_pos & (y_all == 1)).sum() / max(pred_pos.sum(), 1)),
            "recall": float((pred_pos & (y_all == 1)).sum() / max((y_all == 1).sum(), 1))}
    blocking_recall = None
    br = os.path.join(d, "blocking_recall.json")
    if os.path.exists(br):
        with open(br) as fh:
            blocking_recall = json.load(fh)
    report = {
        "checkpoint": ckpt.name,
        "n_pairs": int(n), "n_positive_pairs": int(y_all.sum()), "n_s1": int(len(s1_rids)), "folds": fold_ids,
        "stage1_features": len(f1), "dropped_features": sorted(drop), "stage2": not args.no_stage2,
        "threshold": best["threshold"], "oof_at_threshold": best, "threshold_sweep": rows,
        "decision_rule": {**rule_sel, "candidates": rule_rows, "oof_at_threshold_only": best_plain},
        "stage1_oof_at_threshold": s1_best,
        "per_fold": folds_rep, "per_country": per_country, "per_source": per_source, "pair_level": pair,
        "calibration": {"platt": cal, "nested": cal_report}, "leakage_audit": audit,
        "blocking_recall": blocking_recall, "candidate_audit": blocking_recall,
    }
    tag = f"_{args.tag}" if args.tag else ""
    with open(ckpt.path("validation_report.json"), "w") as fh:
        json.dump(report, fh, indent=1)
    meta.select("pid", "t_rid", "s_rid", "label", "s_fold", "t_fold").with_columns(
        pl.Series("p1", p1), pl.Series("p2", p2), pl.Series("p2_cal", p2c), pl.Series("contra", contra)).write_parquet(ckpt.path("oof.parquet"))
    calibrate.save(cal, ckpt.path("calibration.json"))
    # the OOF-optimal global threshold (+ the selected decision rule) as a policy file: the baseline every
    # per-class policy is compared with
    ThresholdPolicy(best["threshold"], name="global_train", fit={"source": "train.py OOF sweep", "objective": "macro_f05",
                    "oof_macro_f05": best["macro_f05"], "nested_macro_f05": rule_sel["nested_macro_f05"]}, **rule).save(
        ckpt.path(os.path.join("thresholds", "global_train.json")))
    ckpt.update_manifest(oof_macro_f05=best["macro_f05"], nested_macro_f05=rule_sel["nested_macro_f05"], global_threshold=best["threshold"],
                         decision=rule, n_pairs=int(n), n_s1=int(len(s1_rids)), folds=fold_ids, calibration=cal, stage2=not args.no_stage2)
    ckpt.mark_stage("oof")
    if args.tag:
        # ablation run: report + OOF only (no final models); keep the legacy tagged copies at the top level
        with open(os.path.join(args.work_dir, f"validation_report{tag}.json"), "w") as fh:
            json.dump(report, fh, indent=1)
        pl.read_parquet(ckpt.path("oof.parquet")).write_parquet(os.path.join(d, f"oof{tag}.parquet"))
        log(f"ablation checkpoint {ckpt.dir}")
        return

    # ---------------- final models on the whole sample
    m1 = load_or_train(ckpt, "stage1.txt", args.resume, lambda: train_lgb(X1, ys, args.rounds1))
    if not args.no_stage2:
        m2 = load_or_train(ckpt, "stage2.txt", args.resume, lambda: train_lgb(X2, ys, args.rounds2))
        imp = sorted(zip(f2, m2.feature_importance("gain")), key=lambda x: -x[1])
    else:
        imp = sorted(zip(f1, m1.feature_importance("gain")), key=lambda x: -x[1])
    with open(ckpt.path("model_meta.json"), "w") as fh:
        json.dump({"checkpoint": ckpt.name, "f1": f1, "f2": f2, "stage2": not args.no_stage2, "threshold": best["threshold"],
                   "threshold_scale": "calibrated", "oof_macro_f05": best["macro_f05"], "nested_macro_f05": rule_sel["nested_macro_f05"],
                   "decision": rule, "feature_importance_gain": [(nm, float(g)) for nm, g in imp]}, fh, indent=1)
    ckpt.mark_stage("final")
    ckpt.set_latest()
    log(f"checkpoint {ckpt.dir} complete; <work>/ files now point at it")
    log("top features", [(nm, round(float(g))) for nm, g in imp[:25]])


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/er/src/leakage_check.py
"""Automated leakage audit of a finished training run (exit code 1 on any violation).

Checks
  1. fold isolation      : every Source 1 name group lives in exactly one fold; every matched
                           Source 2/3 record carries the fold of its Source 1 entity; every
                           positive candidate pair is within one fold.
  2. lexicon isolation   : an Indic token whose only alignment evidence comes from fold k is
                           absent from the lexicon used to normalise fold k ("fold_k").
  3. train/eval overlap  : the OOF file scores each pair exactly once, with the model of its
                           own fold, and no pid is duplicated.
  4. label-shuffle canary: a small stage-1 model trained on shuffled labels with the same
                           strict folds must have OOF AUC ~ 0.5. Anything clearly above means
                           rows (or their duplicates) are shared between training and
                           evaluation folds.
  5. single-feature AUC  : no stage-1 feature separates the classes perfectly on its own
                           (a feature derived from the label would).
  6. test split hygiene  : test records / pairs carry no label or fold column, and the model's
                           feature list contains none of the bookkeeping columns.
Writes <work>/leakage_check.json.
"""
import collections
import json
import os
import sys

import numpy as np
import polars as pl

from build_lexicon import count_alignments
from common import Stage, base_args, log, read_tsv, source_path, split_dir
from model import NON_FEATURES, part_files, stage1_features, train_lgb, to_np
from textnorm import INDIC_RE


def auc(score, y):
    order = np.argsort(score)
    ranks = np.empty(len(score), dtype=np.float64)
    ranks[order] = np.arange(1, len(score) + 1)
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    return float((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def main():
    ap = base_args(__doc__)
    ap.add_argument("--canary-rows", type=int, default=1_000_000)
    ap.add_argument("--canary-rounds", type=int, default=50)
    args = ap.parse_args()
    with Stage(args.work_dir, "leakage_check"):
        ok = run(args)
    sys.exit(0 if ok else 1)


def run(args):
    dtr = split_dir(args.work_dir, "train")
    res, problems = {}, []

    # ---- 1. fold isolation
    rec = pl.read_parquet(os.path.join(dtr, "records.parquet"), columns=["rid", "entity_id", "src", "fold", "group", "country"])
    s1 = rec.filter(pl.col("src") == 1)
    g = s1.group_by("group").agg(pl.col("fold").n_unique().alias("nf"))
    multi = int((g["nf"] > 1).sum())
    res["name_groups"] = {"n": g.height, "split_across_folds": multi}
    if multi:
        problems.append(f"{multi} Source 1 name groups appear in more than one fold")
    gt = read_tsv(os.path.join(args.data_dir, "train", "train_ground_truth.tsv"))
    ex = (gt.with_columns(pl.col("matched_entity_ids").str.split(",")).explode("matched_entity_ids").drop_nulls("matched_entity_ids"))
    f = rec.select("entity_id", "fold")
    ex = ex.join(f.rename({"entity_id": "source1_entity_id", "fold": "fs"}), on="source1_entity_id").join(
        f.rename({"entity_id": "matched_entity_ids", "fold": "ft"}), on="matched_entity_ids")
    bad = int((ex["fs"] != ex["ft"]).sum())
    res["matched_records_fold_mismatch"] = bad
    if bad:
        problems.append(f"{bad} matched Source 2/3 records are not in their Source 1 entity's fold")

    # ---- 3. OOF overlap + positive pairs within fold
    oof = pl.read_parquet(os.path.join(dtr, "oof.parquet"))
    dup = oof.height - oof["pid"].n_unique()
    res["oof"] = {"pairs": oof.height, "duplicate_pids": dup,
                  "positive_pairs_cross_fold": int(oof.filter((pl.col("label") == 1) & (pl.col("s_fold") != pl.col("t_fold"))).height),
                  "cross_fold_pair_share": float((oof["s_fold"] != oof["t_fold"]).mean())}
    if dup:
        problems.append(f"{dup} duplicated pids in oof.parquet")
    if res["oof"]["positive_pairs_cross_fold"]:
        problems.append("positive pairs with mismatching folds")

    # ---- 2. lexicon isolation
    with open(os.path.join(args.work_dir, "lexicon.json"), encoding="utf-8") as fh:
        lexicons = json.load(fh)
    s1raw = read_tsv(source_path(args.data_dir, "train", "source1"))
    others = pl.concat([read_tsv(source_path(args.data_dir, "train", s)) for s in ("source2", "source3")])
    pairs = ex.select(pl.col("source1_entity_id").alias("s1"), pl.col("matched_entity_ids").alias("t"), pl.col("fs").alias("fold"))
    indic = others.filter(pl.col("business_name").str.contains(INDIC_RE.pattern)
                          | pl.col("business_address").fill_null("").str.contains(INDIC_RE.pattern))
    j = indic.join(pairs, left_on="entity_id", right_on="t").join(
        s1raw.select(pl.col("entity_id").alias("s1"), pl.col("business_name").alias("n1"), pl.col("business_address").alias("a1")), on="s1")
    per_fold = {k: count_alignments(j.filter(pl.col("fold") == k)) for k in sorted(j["fold"].unique().to_list())}
    lex_viol, lex_checked = 0, 0
    for k, (nc, ac) in per_fold.items():
        lex = lexicons.get(f"fold_{k}")
        if lex is None:
            continue
        others_tokens = collections.Counter()
        for kk, (nc2, ac2) in per_fold.items():
            if kk != k:
                others_tokens.update(nc2.keys())
                others_tokens.update(ac2.keys())
        for tok in list(nc.keys()) + list(ac.keys()):
            if tok not in others_tokens:  # evidence only in fold k
                lex_checked += 1
                if tok in lex["name"] or tok in lex["addr"]:
                    lex_viol += 1
    res["lexicon"] = {"fold_only_tokens_checked": lex_checked, "leaked_into_own_fold_lexicon": lex_viol,
                      "lexicons": {k: {"name": len(v["name"]), "addr": len(v["addr"])} for k, v in lexicons.items()}}
    if lex_viol:
        problems.append(f"{lex_viol} fold-specific Indic tokens present in their own fold's lexicon")

    # ---- 4./5. label-shuffle canary and single-feature AUC on a row sample
    files = part_files(dtr)
    feats = stage1_features(files)
    rng = np.random.default_rng(0)
    # sample while streaming the parts: reading every feature file in full is ~10 GB on the full data
    n_total = sum(pl.read_parquet(fpath, columns=["pid"]).height for fpath in files)
    frac = min(1.0, args.canary_rows / max(n_total, 1))
    frames = []
    for fpath in files:
        df = pl.read_parquet(fpath, columns=["pid", "label"] + feats)
        if frac < 1.0:
            df = df.filter(pl.Series(rng.random(df.height) < frac))
        frames.append(df)
    df = pl.concat(frames)
    df = df.join(oof.select("pid", "s_fold", "t_fold"), on="pid", how="left")
    X = to_np(df, feats)
    y = df["label"].to_numpy().astype(np.int8)
    sf, tf = df["s_fold"].to_numpy(), df["t_fold"].to_numpy()
    y_sh = y.copy()
    for k in np.unique(sf):  # shuffle within fold: keeps each fold's positive rate
        idx = np.where(sf == k)[0]
        y_sh[idx] = y[rng.permutation(idx)]
    p = np.zeros(len(y), dtype=np.float32)
    for k in np.unique(sf):
        tr = (sf != k) & (tf != k)
        m = train_lgb(X[tr], y_sh[tr], args.canary_rounds, {"num_leaves": 31, "min_data_in_leaf": 50})
        p[sf == k] = m.predict(X[sf == k])
    canary = auc(p, y_sh)
    res["label_shuffle_canary_auc"] = canary
    if not (canary < 0.55):
        problems.append(f"label-shuffle canary OOF AUC {canary:.3f} (expected ~0.5): rows are shared between folds")
    single = sorted(((c, auc(X[:, i], y)) for i, c in enumerate(feats)), key=lambda x: -abs(x[1] - 0.5))[:8]
    res["top_single_feature_auc"] = [(c, round(a, 4)) for c, a in single]
    perfect = [c for c, a in single if a > 0.999 or a < 0.001]
    if perfect:
        problems.append(f"features separating the classes perfectly: {perfect}")

    # ---- 6. test split hygiene
    dte = split_dir(args.work_dir, "test")
    test_ok = True
    if os.path.exists(os.path.join(dte, "records.parquet")):
        cols = set(pl.read_parquet_schema(os.path.join(dte, "records.parquet")).keys())
        tfiles = part_files(dte)
        pcols = set(pl.read_parquet_schema(tfiles[0]).keys()) if tfiles else set()
        test_ok = "label" not in pcols and "fold" not in pcols and "label" not in cols
        res["test_split"] = {"records_have_fold_or_label": bool({"fold", "label"} & cols), "pairs_have_label": "label" in pcols}
    with open(os.path.join(args.work_dir, "model_meta.json")) as fh:
        meta = json.load(fh)
    leaky_feats = sorted(set(meta["f1"]) & NON_FEATURES)
    res["model_features_bookkeeping"] = leaky_feats
    if leaky_feats or not test_ok:
        problems.append(f"bookkeeping columns used as features / present in test: {leaky_feats}")

    res["problems"] = problems
    res["ok"] = not problems
    with open(os.path.join(args.work_dir, "leakage_check.json"), "w") as fh:
        json.dump(res, fh, indent=1)
    log("leakage check", "OK" if not problems else "FAILED", json.dumps({k: v for k, v in res.items() if k != "lexicon"}, default=str)[:1500])
    for p_ in problems:
        log("  PROBLEM:", p_)
    return not problems


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/er/src/select_threshold.py
"""Step 4b: pick the acceptance threshold for the test split under the expected prior shift.

The test split has more Source 2/3 records per Source 1 entity than training (5.75 vs
4.68), i.e. more unmatched "decoy" records per entity. Two ways to transfer the threshold
tuned on the training OOF predictions are compared here, both using record counts only
(no test labels):

  density  : re-weight every false link caused by a decoy by r = decoys_per_S1(test) /
             decoys_per_S1(train) in the OOF macro F0.5 and maximise (the previous method).
  prior    : treat the calibrated probability as P(match | pair) under the training prior and
             shift its odds by the change in the positive/negative pair ratio between
             train and test candidate pairs, then keep the training threshold
             (calibrate.shift_odds). Equivalent to moving the threshold on the training
             scale; reported as such.

Both are computed on the calibrated OOF probabilities (train.py writes p2_cal); the chosen
threshold (on the calibrated scale) and the full analysis go to <work>/model_meta.json and
<work>/threshold_analysis.json. --method chooses which one is written as `threshold`. The chosen
threshold (with the decision rule selected by train.py) is the policy predict.py applies: it is
written to <checkpoint>/thresholds/selected.json and linked as <work>/threshold_policy.json.
"""
import json
import os

import numpy as np
import polars as pl

import calibrate
from checkpoint import Checkpoint
from common import base_args, log, split_dir
from pair_features import truth_pairs
from threshold_policy import ThresholdPolicy
from thresholds import best_threshold, default_grid, link_table, metrics_at, sweep


def decoy_ratio(src_tr, src_te, n_true):
    """Decoy-density ratio test/train from record counts per source ([n_s1, n_s2, n_s3] per split)
    and the number of true training pairs. Returns (m, decoys_per_s1_train, decoys_per_s1_test, r)."""
    m = n_true / src_tr[0]
    dec_tr = (src_tr[1] + src_tr[2]) / src_tr[0] - m
    dec_te = (src_te[1] + src_te[2]) / src_te[0] - m
    r = max(1.0, dec_te / dec_tr) if dec_tr > 0 else 1.0
    return m, dec_tr, dec_te, r


def source_counts(work_dir):
    """[n_s1, n_s2, n_s3] of the train and test split, from records.parquet."""
    out = []
    for split in ("train", "test"):
        path = os.path.join(split_dir(work_dir, split), "records.parquet")
        if not os.path.exists(path):
            out.append(None)
            continue
        vc = pl.read_parquet(path, columns=["src"])["src"].value_counts().sort("src")
        out.append(vc["count"].to_list())
    return out


def main():
    ap = base_args(__doc__)
    ap.add_argument("--ratio", type=float, default=None, help="override the estimated decoy-density ratio")
    ap.add_argument("--method", choices=["density", "prior", "train"], default="density")
    args = ap.parse_args()
    dtr = split_dir(args.work_dir, "train")
    rec = pl.read_parquet(os.path.join(dtr, "records.parquet"), columns=["rid", "entity_id", "src"])
    truth = truth_pairs(rec, args.data_dir).drop("label")
    src_tr, src_te = source_counts(args.work_dir)
    m, dec_tr, dec_te, r = decoy_ratio(src_tr, src_te, truth.height)
    r = args.ratio if args.ratio is not None else r
    log(f"true matches/S1 (train) {m:.3f}; decoys/S1 train {dec_tr:.3f} test {dec_te:.3f}; ratio r={r:.3f}")

    oof = pl.read_parquet(os.path.join(dtr, "oof.parquet"))
    contra = oof["contra"].to_numpy() if "contra" in oof.columns else None
    s1_rids = rec.filter(pl.col("src") == 1)["rid"].to_numpy().astype(np.uint32)
    links, nt = link_table(oof.select("t_rid", "s_rid"), oof["p2_cal"].to_numpy(), truth, s1_rids, contra)
    grid = default_grid()
    meta_path = os.path.join(args.work_dir, "model_meta.json")
    with open(meta_path) as f:
        meta = json.load(f)
    rule = dict(meta.get("decision") or {"margin": 0.0, "contra_penalty": 0.0})  # decision rule selected by train.py
    plain = sweep(links, nt, grid, **rule)
    density = sweep(links, nt, grid, decoy_weight=r, **rule)
    # prior shift: positive pairs / negative pairs among the candidates. Under the density model
    # the negatives grow by the decoy share; the odds ratio is (neg_train / neg_test_expected).
    y = oof["label"].to_numpy()
    n_pos, n_neg = float(y.sum()), float((1 - y).sum())
    # negative pairs that come from decoy records scale with r; the rest (wrong candidates of
    # matched records) do not
    matched_t = truth.select("t_rid").unique()
    n_decoy_pairs = float(oof.join(matched_t, on="t_rid", how="anti").height)
    neg_te = n_neg + (r - 1.0) * n_decoy_pairs
    odds_ratio = (n_pos / neg_te) / (n_pos / n_neg)
    shifted = calibrate.shift_odds(links["p"].to_numpy(), odds_ratio)
    links_shift = links.with_columns(pl.Series("p", shifted))
    prior_rows = sweep(links_shift, nt, grid, decoy_weight=r, **rule)
    thr_train = best_threshold(plain)["threshold"]
    prior_equiv = float(calibrate.sigmoid(calibrate.logit(thr_train) - np.log(odds_ratio)))  # same rule on the unshifted scale
    choice = {"train": thr_train, "density": best_threshold(density)["threshold"], "prior": round(prior_equiv, 4)}
    thr = choice[args.method]
    analysis = {
        "decoy_ratio": r, "odds_ratio_prior_shift": odds_ratio, "chosen_method": args.method, "threshold": thr,
        "candidates": choice,
        "decision_rule": rule,
        "oof_plain_at": {k: metrics_at(links, nt, v, **rule) for k, v in choice.items()},
        "oof_density_adjusted_at": {k: metrics_at(links, nt, v, decoy_weight=r, **rule) for k, v in choice.items()},
        "sweep_plain": plain, "sweep_density": density, "sweep_prior_shifted": prior_rows,
    }
    for k, v in choice.items():
        log(f"{k:8s} thr={v:.3f}  OOF macro F0.5={analysis['oof_plain_at'][k]['macro_f05']:.5f}  "
            f"density-adjusted={analysis['oof_density_adjusted_at'][k]['macro_f05']:.5f}")
    with open(os.path.join(args.work_dir, "threshold_analysis.json"), "w") as f:
        json.dump(analysis, f, indent=1)
    meta.update({"threshold": thr, "threshold_method": args.method, "decoy_ratio": r, "threshold_candidates": choice,
                 "oof_macro_f05_at_threshold": analysis["oof_plain_at"][args.method]["macro_f05"],
                 "adjusted_macro_f05_at_threshold": analysis["oof_density_adjusted_at"][args.method]["macro_f05"]})
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=1)
    ckpt = Checkpoint.resolve(args.work_dir, None)
    if ckpt is not None and ckpt.exists():
        # every candidate as a policy file (predict.py --policy scores any of them without re-running the model)
        for k, v in choice.items():
            ThresholdPolicy(v, name=f"global_{k}", fit={"source": "select_threshold.py", "method": k, "decoy_ratio": r,
                            "oof_macro_f05": analysis["oof_plain_at"][k]["macro_f05"]}, **rule).save(ckpt.path(os.path.join("thresholds", f"global_{k}.json")))
        ThresholdPolicy(thr, name=f"global_{args.method}", fit={"source": "select_threshold.py", "method": args.method, "decoy_ratio": r,
                        "oof_macro_f05": analysis["oof_plain_at"][args.method]["macro_f05"]}, **rule).save(ckpt.path(os.path.join("thresholds", "selected.json")))
        ckpt.set_latest()  # refreshes <work>/threshold_policy.json
        ckpt.log_experiment({"kind": "select_threshold", "method": args.method, "threshold": thr, "decoy_ratio": r, "candidates": choice})
    log(f"selected threshold {thr} ({args.method})")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/er/src/decode.py
"""Expected-F0.5 decoding.

Each Source 2/3 record t is offered only to its best Source 1 candidate s (one-to-one
structure), with probability q = p2(t, s). For every Source 1 entity the offered records are
sorted by q and we choose how many of them (k = 0..K) to link so as to maximise the expected
per-entity F0.5, treating the offers as independent Bernoulli(q) events:

    E[F_k] = sum_a sum_b P(TP_k = a) P(FN_k = b) * F(a, b, k)

where TP_k is the number of true records among the k linked ones and FN_k among the rest
(both Poisson-binomial, computed exactly by dynamic programming). k = 0 scores 1 only when
the entity has no true record at all (singleton). `miss` adds the expected number of true
records that blocking never proposed (they lower recall whatever we choose).
"""
import numpy as np
import polars as pl

BETA2 = 0.25


def _pb_dist(q):
    """Poisson-binomial pmf for each row of q (N, m) -> (N, m+1)."""
    N, m = q.shape
    d = np.zeros((N, m + 1))
    d[:, 0] = 1.0
    for j in range(m):
        p = q[:, j:j + 1]
        d[:, 1:] = d[:, 1:] * (1 - p) + d[:, :-1] * p
        d[:, 0] = d[:, 0] * (1 - p[:, 0])
    return d


def best_k(Q, miss=0.0):
    """Q: (N, K) offers sorted descending per row (0-padded). Returns chosen k per row."""
    N, K = Q.shape
    best = np.zeros(N, dtype=np.int64)
    best_val = np.full(N, -1.0)
    a = np.arange(K + 1)[:, None]
    b = np.arange(K + 1)[None, :]
    for k in range(K + 1):
        tp = _pb_dist(Q[:, :k]) if k > 0 else np.ones((N, 1))
        fn = _pb_dist(Q[:, k:]) if k < K else np.ones((N, 1))
        A = a[: tp.shape[1]]
        B = b[:, : fn.shape[1]]
        nt = A + B + miss
        if k == 0:
            # empty prediction scores 1 only for a singleton; with miss > 0 the entity may still
            # have unproposed true records, approximated as Poisson(miss): P(none) = exp(-miss)
            f = np.where(A + B == 0, np.exp(-miss), 0.0).astype(float)
        else:
            f = (1 + BETA2) * A / (BETA2 * nt + k)
        val = np.einsum("na,nb,ab->n", tp, fn, f)
        upd = val > best_val
        best[upd] = k
        best_val[upd] = val[upd]
    return best


def decode(meta, p, max_offers=12, miss=0.0, min_q=0.02):
    """meta (t_rid, s_rid) aligned with p -> links DataFrame (t_rid, s_rid, p)."""
    offers = (
        meta.select("t_rid", "s_rid").with_columns(pl.Series("p", p))
        .sort("p", descending=True)
        .unique("t_rid", keep="first")
        .filter(pl.col("p") >= min_q)
        .sort(["s_rid", "p"], descending=[False, True])
        .with_columns(pl.int_range(pl.len()).over("s_rid").alias("j"))
    )
    top = offers.filter(pl.col("j") < max_offers)
    s_ids, s_idx = np.unique(top["s_rid"].to_numpy(), return_inverse=True)
    Q = np.zeros((len(s_ids), max_offers))
    Q[s_idx, top["j"].to_numpy()] = top["p"].to_numpy()
    k = best_k(Q, miss=miss)
    kk = pl.DataFrame({"s_rid": s_ids, "k": k})
    return top.join(kk, on="s_rid").filter(pl.col("j") < pl.col("k")).select("t_rid", "s_rid", "p")


In [ ]:
%%writefile /kaggle/working/er/src/predict.py
"""Step 5: score the test candidate pairs and write the two submission files.

output/matching_results.tsv : source1_entity_id, matched_entity_ids
output/candidate_pairs.tsv  : source1_entity_id, candidate_entity_ids (the exact pairs scored)
Every test Source 1 entity gets exactly one row in both files (empty list if nothing).

Model and decision rule are decoupled:
  --checkpoint  which trained checkpoint to score with (default: <work>/checkpoints/LATEST, falling
                back to the legacy <work>/stage*.txt files)
  --policy      threshold policy JSON (threshold_policy.py): the global threshold + decision rule.
                Default: the checkpoint's thresholds/selected.json (written by select_threshold.py),
                else <work>/threshold_policy.json, else the checkpoint's OOF-optimal global threshold.
  --threshold   a plain global threshold, overriding the policy
  --reuse-scores  skip model inference when <work>/test/scores_<checkpoint>.parquet exists (written
                on the first run): evaluating one checkpoint under many policies then costs seconds.
output/prediction_meta.json records which checkpoint and policy produced the files.
"""
import json
import os

import lightgbm as lgb
import numpy as np
import polars as pl

import calibrate
from checkpoint import Checkpoint
from common import Stage, base_args, left_join_ordered, log, split_dir
from decode import decode
from model import CONTRA_COLS, assign, best_candidates, contradiction_flag, house_numbers, iter_parts, part_files, stage2_context, to_np
from threshold_policy import ThresholdPolicy


def write_lists(s1_ids, links, id_col, path):
    """links: (source1_entity_id, entity_id). Writes one row per Source 1 id, tab separated."""
    agg = links.group_by("source1_entity_id").agg(pl.col("entity_id").unique().sort().str.join(",").alias(id_col))
    out = left_join_ordered(s1_ids, agg, "source1_entity_id").with_columns(pl.col(id_col).fill_null(""))
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(f"source1_entity_id\t{id_col}\n")
        for sid, ids in out.iter_rows():
            f.write(f"{sid}\t{ids}\n")
    return out


def add_args(ap):
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--checkpoint", default=None, help="checkpoint name or directory (default: LATEST)")
    ap.add_argument("--policy", default=None, help="threshold policy JSON (default: the checkpoint's selected policy)")
    ap.add_argument("--threshold", type=float, default=None, help="override: one global threshold")
    ap.add_argument("--decoder", choices=["threshold", "expected_f"], default=None,
                    help="override the decision rule (default: threshold policy)")
    ap.add_argument("--reuse-scores", action="store_true", help="reuse cached test scores of this checkpoint if present")
    return ap


def main():
    ap = add_args(base_args(__doc__))
    args = ap.parse_args()
    with Stage(args.work_dir, "predict"):
        run(args)


def resolve_model(work_dir, checkpoint):
    """(checkpoint or None, path resolver) - legacy work dirs without checkpoints still work."""
    ckpt = Checkpoint.resolve(work_dir, checkpoint)
    if ckpt is not None and ckpt.exists():
        return ckpt, ckpt.path
    if checkpoint:
        raise SystemExit(f"checkpoint {checkpoint} not found under {work_dir}/checkpoints")
    return None, lambda f: os.path.join(work_dir, f)


def resolve_policy(args, ckpt, meta_json):
    if args.threshold is not None:
        return ThresholdPolicy(args.threshold, name=f"cli_{args.threshold}", scale=meta_json.get("threshold_scale", "calibrated"),
                               **meta_json.get("decision", {}))
    if args.policy:
        if not os.path.exists(args.policy):
            raise SystemExit(f"policy file {args.policy} not found")
        return ThresholdPolicy.load(args.policy)
    for path in ([ckpt.path(os.path.join("thresholds", "selected.json"))] if ckpt else []) + [os.path.join(args.work_dir, "threshold_policy.json")]:
        if os.path.exists(path):
            return ThresholdPolicy.load(path)
    return ThresholdPolicy(meta_json["threshold"], name="model_meta_global", scale=meta_json.get("threshold_scale", "calibrated"))


def score_test(d, files, meta, n, m1, m2, f1, use_stage2, cal, calibrated, f2=None):
    p1 = np.zeros(n, dtype=np.float32)
    for df in iter_parts(files):
        pid = df["pid"].to_numpy()
        p1[pid] = m1.predict(to_np(df, f1))
    if use_stage2:
        C, cnames = stage2_context(meta, p1, house_numbers(d))
        if f2 is not None:  # the context columns the stage-2 model was trained with (train.py --drop-features)
            C = np.ascontiguousarray(C[:, [cnames.index(c) for c in f2 if c in cnames]])
        p2 = np.zeros(n, dtype=np.float32)
        for df in iter_parts(files):
            pid = df["pid"].to_numpy()
            p2[pid] = m2.predict(np.hstack([to_np(df, f1), p1[pid, None], C[pid]]))
    else:
        p2 = p1
    # the thresholds are tuned on the calibrated scale (train.py); apply the same calibrator here
    p2 = calibrate.apply(cal, p2).astype(np.float32) if calibrated else p2
    return p1, p2


def run(args):
    d = split_dir(args.work_dir, "test")
    ckpt, path = resolve_model(args.work_dir, args.checkpoint)
    with open(path("model_meta.json")) as f:
        meta_json = json.load(f)
    policy = resolve_policy(args, ckpt, meta_json)
    if policy.scale != meta_json.get("threshold_scale", "calibrated"):
        raise SystemExit(f"policy scale {policy.scale} does not match the model's {meta_json.get('threshold_scale')}")
    ckpt_name = ckpt.name if ckpt else "legacy"
    files = part_files(d)
    meta = pl.concat(list(iter_parts(files, ["pid", "t_rid", "s_rid", *CONTRA_COLS]))).sort("pid")
    contra = contradiction_flag(meta)
    meta = meta.drop(CONTRA_COLS)
    n = meta.height
    log(f"test pairs {n}; checkpoint {ckpt_name}; {policy.describe()}")

    cache = os.path.join(d, f"scores_{ckpt_name}.parquet")
    if args.reuse_scores and os.path.exists(cache):
        sc = pl.read_parquet(cache)
        assert sc.height == n and (sc["pid"].to_numpy() == meta["pid"].to_numpy()).all(), "cached scores do not match the pair table"
        p1, p2 = sc["p1"].to_numpy(), sc["p2"].to_numpy()
        log(f"reused cached scores {cache}")
    else:
        f1 = meta_json["f1"]
        m1 = lgb.Booster(model_file=path("stage1.txt"))
        use_stage2 = meta_json.get("stage2", True)
        m2 = lgb.Booster(model_file=path("stage2.txt")) if use_stage2 else None
        cal = calibrate.load(path("calibration.json")) if os.path.exists(path("calibration.json")) else None
        p1, p2 = score_test(d, files, meta, n, m1, m2, f1, use_stage2, cal, meta_json.get("threshold_scale") == "calibrated", meta_json.get("f2"))
        meta.with_columns(pl.Series("p1", p1), pl.Series("p2", p2)).write_parquet(cache)
        meta.with_columns(pl.Series("p1", p1), pl.Series("p2", p2)).write_parquet(os.path.join(d, "test_scores.parquet"))

    decoder = args.decoder or meta_json.get("decoder", "threshold")
    rec = pl.read_parquet(os.path.join(d, "records.parquet"), columns=["rid", "entity_id", "src"])
    if decoder == "expected_f":
        links = decode(meta, p2)
    else:
        links = assign(meta, p2, policy.default, policy.margin, policy.contra_penalty, contra)
    log(f"decoder {decoder}: {links.height} links")

    ids = rec.select(pl.col("rid").cast(pl.UInt32), "entity_id")
    s1_ids = rec.filter(pl.col("src") == 1).select(pl.col("entity_id").alias("source1_entity_id"))

    def named(df):
        return (
            df.select("t_rid", "s_rid")
            .join(ids.rename({"rid": "s_rid", "entity_id": "source1_entity_id"}), on="s_rid")
            .join(ids.rename({"rid": "t_rid"}), on="t_rid")
            .select("source1_entity_id", "entity_id")
        )

    os.makedirs(args.out_dir, exist_ok=True)
    m = write_lists(s1_ids, named(links), "matched_entity_ids", os.path.join(args.out_dir, "matching_results.tsv"))
    c = write_lists(s1_ids, named(meta), "candidate_entity_ids", os.path.join(args.out_dir, "candidate_pairs.tsv"))
    log(f"wrote {m.height} rows; {(m['matched_entity_ids'] != '').sum()} Source 1 entities matched; "
        f"{(c['candidate_entity_ids'] != '').sum()} with candidates")
    best = best_candidates(meta, p2, contra)
    rejected = {"records_with_candidates": int(best.height), "linked": int(links.height)}
    if decoder != "expected_f":
        pb = best["p"].to_numpy()
        above = pb >= policy.default
        rejected.update({"below_threshold": int((~above).sum()),
                         "rejected_by_margin": int((above & (pb - best["p2nd"].to_numpy() < policy.margin)).sum()),
                         "rejected_by_contradiction": int((above & (pb - best["p2nd"].to_numpy() >= policy.margin)
                                                           & best["contra"].to_numpy() & (pb < policy.default + policy.contra_penalty)).sum())
                         if policy.contra_penalty > 0 else 0})
    log(f"rejection: {rejected}")
    info = {"checkpoint": ckpt_name, "checkpoint_dir": ckpt.dir if ckpt else None, "policy": policy.to_dict(), "decoder": decoder,
            "test_pairs": int(n), "links": int(links.height), "entities_matched": int((m["matched_entity_ids"] != "").sum()),
            "rejection": rejected, "out_dir": os.path.abspath(args.out_dir)}
    with open(os.path.join(args.out_dir, "prediction_meta.json"), "w") as f:
        json.dump(info, f, indent=1)
    if ckpt:
        ckpt.log_experiment({"kind": "predict", "policy": policy.name, "threshold": policy.default, "rule": policy.rule,
                             "decoder": decoder, "links": int(links.height), "out_dir": os.path.abspath(args.out_dir)})
    return info


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/er/src/evaluate.py
"""Scores a matching_results.tsv against a ground-truth file with the challenge metric.

    python src/evaluate.py --pred output/matching_results.tsv --truth subset/test/subset_ground_truth.tsv \
        [--source1 subset/test/test_source1.tsv]   # for per-country breakdown

Reports macro F0.5 (singletons included, exactly as scored), micro precision / recall,
per-entity mean precision / recall, singleton accuracy, and the same per country when the
Source 1 file is given. Writes JSON to --out when given.
"""
import argparse
import json

import polars as pl

from common import read_tsv
from metrics import macro_f05


def id_lists(df, col):
    return df.select(
        pl.col("source1_entity_id").alias("s"),
        pl.col(col).fill_null("").str.split(",").list.eval(pl.element().filter(pl.element() != "")).alias("m"),
    )


def per_entity(pred, truth):
    """Per-entity precision / recall / F0.5 as a frame (s, nt, np, tp, p, r, f)."""
    j = truth.rename({"m": "true"}).join(pred.rename({"m": "pred"}), on="s", how="left").with_columns(
        pl.col("pred").fill_null(pl.lit([], dtype=pl.List(pl.String))))
    j = j.with_columns(
        pl.col("true").list.len().alias("nt"), pl.col("pred").list.len().alias("np"),
        pl.col("true").list.set_intersection("pred").list.len().alias("tp"))
    j = j.with_columns(
        pl.when(pl.col("np") > 0).then(pl.col("tp") / pl.col("np")).otherwise(0.0).alias("p"),
        pl.when(pl.col("nt") > 0).then(pl.col("tp") / pl.col("nt")).otherwise(0.0).alias("r"))
    f = pl.when((pl.col("nt") == 0) & (pl.col("np") == 0)).then(1.0).when(pl.col("nt") == 0).then(0.0).when(
        pl.col("p") + pl.col("r") > 0).then(1.25 * pl.col("p") * pl.col("r") / (0.25 * pl.col("p") + pl.col("r"))).otherwise(0.0)
    return j.with_columns(f.alias("f")).drop("true", "pred")


def summary(pe):
    """The numbers people quote, side by side, so the discrepancies between them are explicit."""
    non_single = pe.filter(pl.col("nt") > 0)
    return {
        "n_entities": pe.height,
        "macro_f05": float(pe["f"].mean()),
        "macro_f05_non_singletons": float(non_single["f"].mean()) if non_single.height else None,
        "singleton_share": float((pe["nt"] == 0).mean()),
        "singleton_acc": float(((pe["nt"] == 0) & (pe["np"] == 0)).sum() / max((pe["nt"] == 0).sum(), 1)),
        "micro_precision": float(pe["tp"].sum() / max(pe["np"].sum(), 1)),
        "micro_recall": float(pe["tp"].sum() / max(pe["nt"].sum(), 1)),
        "mean_entity_precision": float(non_single["p"].mean()) if non_single.height else None,
        "mean_entity_recall": float(non_single["r"].mean()) if non_single.height else None,
        "entities_perfect": float((pe["f"] == 1.0).mean()),
        "entities_zero": float((pe["f"] == 0.0).mean()),
    }


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--pred", required=True)
    ap.add_argument("--truth", required=True)
    ap.add_argument("--source1", default=None)
    ap.add_argument("--out", default=None)
    args = ap.parse_args()
    pred = id_lists(read_tsv(args.pred), "matched_entity_ids")
    truth = id_lists(read_tsv(args.truth), "matched_entity_ids")
    f, stats = macro_f05(pred, truth)
    pe = per_entity(pred, truth)
    res = {"overall": summary(pe)}
    assert abs(res["overall"]["macro_f05"] - f) < 1e-9
    if args.source1:
        s1 = read_tsv(args.source1).select(pl.col("entity_id").alias("s"), pl.col("country").fill_null(""))
        pe_c = pe.join(s1, on="s", how="left")
        res["per_country"] = {c: summary(pe_c.filter(pl.col("country") == c)) for c in sorted(pe_c["country"].unique().to_list())}
    print(json.dumps(res, indent=1))
    if args.out:
        with open(args.out, "w") as fh:
            json.dump(res, fh, indent=1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/er/src/validate_submission.py
#!/usr/bin/env python3
"""
ML Challenge 2026 — Submission Validator

Run this BEFORE submitting. It checks your output files against every formatting
rule the scorer enforces, so you can catch a rejection locally instead of burning
a submission. It reads only your output files and the test source files (to learn
which S1 entities are required and which S2/S3 IDs exist); it never needs the
ground truth and never computes your score.

It validates two files:

* ``matching_results.tsv`` (required) — your final matches, the file scored on the
  leaderboard.
* ``candidate_pairs.tsv`` (optional) — the candidate set from your blocking stage.
  When present, the validator also checks that your final matches are a subset of
  your candidates and *warns* (never fails) otherwise. When absent it is skipped
  with a warning; it is still expected in your final submission zip.

Stdlib only, Python 3.8+. Run from the ``student_resource/`` directory::

    python3 utils/validate_submission.py \
        --matching output/matching_results.tsv \
        --candidate output/candidate_pairs.tsv \
        --test-dir dataset/test

Exit code 0 means the files are safe to submit; 1 means fix the listed issues
(warnings never fail the run).

ID-existence check (off by default). By default the validator does NOT check that
every matched/candidate ID actually exists in the test set: that check loads all
Source-2/3 IDs into memory, which on the full ~1.7M-entity test set costs a few GB
(more when ``candidate_pairs.tsv`` is included). The default run therefore stays fast
and light and verifies every other rule; it prints a warning noting the check was
skipped. Pass ``--check-ids`` to turn it on (it reads ``test_source2.tsv`` /
``test_source3.tsv`` from ``--test-dir``); a missing/garbage matched ID only lowers
your score rather than being rejected by the scorer, so this check is a diagnostic,
not a gate. If ``--check-ids`` runs out of memory, drop ``--candidate`` (the candidate
cross-check is the biggest memory user, and the matching file is the only one scored).
"""

import argparse
import os
import sys

DELIM = "\t"
MAX_EXAMPLES = 5  # how many offending IDs to show per issue
MATCHING_HEADER = ["source1_entity_id", "matched_entity_ids"]
CANDIDATE_HEADER = ["source1_entity_id", "candidate_entity_ids"]


def read_ids(path):
    """Return the set of first-column entity IDs from a source TSV.

    The header row is skipped and blank lines are ignored.
    """
    with open(path, encoding="utf-8") as f:
        next(f, None)  # skip header
        return {line.split(DELIM, 1)[0].strip() for line in f if line.strip()}


def examples(items):
    """Return a short, human-readable sample of ``items`` for an error message."""
    items = sorted(items)
    shown = ", ".join(items[:MAX_EXAMPLES])
    if len(items) > MAX_EXAMPLES:
        return f"{len(items)} total, e.g. {shown}, ..."
    return shown


def load_match_targets(test_dir, warnings):
    """Return the set of valid S2/S3 match IDs, or ``None`` if unavailable.

    Only called when ``--check-ids`` is on. When ``test_source2.tsv`` or
    ``test_source3.tsv`` is missing we cannot check that matched IDs exist, so we
    record a warning and return ``None`` to signal that the existence check should be
    skipped.
    """
    targets = set()
    for name in ("test_source2.tsv", "test_source3.tsv"):
        path = os.path.join(test_dir, name)
        if not os.path.isfile(path):
            warnings.append(
                f"{path} not found — skipping the (optional) check that matched "
                f"IDs exist in the test set. Every other rule is still checked. "
                f"This is the lighter-memory mode; provide test_source2/3.tsv to "
                f"enable the ID-existence check."
            )
            return None
        targets |= read_ids(path)
    return targets


def validate_id_list_file(path, expected_header, col_label, required, valid_ids, errors):
    """Validate one results-style TSV (matching or candidate).

    Applies the shared formatting rules and appends any problems to ``errors``.
    Returns a ``{source1_id: set(matched/candidate ids)}`` mapping, or ``None`` on a
    fatal problem (missing file, empty file, or a broken header) that stops parsing.
    """
    if not os.path.isfile(path):
        errors.append(f"File not found: {path}")
        return None

    name = os.path.basename(path)
    mapping = {}
    seen, dup_rows, intra_dupes = set(), set(), set()
    self_matches, wrong_prefix, unknown = set(), set(), set()
    n_rows = empties = 0

    with open(path, encoding="utf-8") as f:
        header = f.readline()
        if not header:
            errors.append(f"{name} is empty.")
            return None
        if DELIM not in header and "," in header:  # the #1 mistake: a CSV
            errors.append(
                f"{name}: header has no TAB but contains commas — the file looks "
                "COMMA-separated. Submissions must be TAB-separated (.tsv); "
                "write it with df.to_csv(sep='\\t', index=False)."
            )
            return None
        cols = [c.strip().lower() for c in header.rstrip("\n").split(DELIM)]
        if cols != expected_header:
            errors.append(
                f"{name}: unexpected header {cols}. "
                f"Expected exactly {expected_header} (tab-separated)."
            )
            return None

        for line_num, line in enumerate(f, start=2):
            s1, tab, rest = line.partition(DELIM)
            if not tab:
                if s1.strip():
                    errors.append(
                        f"{name}: malformed row (no tab) at line {line_num}: "
                        f"{line.rstrip()!r}"
                    )
                continue

            n_rows += 1
            if s1 in seen:
                dup_rows.add(s1)
            seen.add(s1)

            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            if not ids:
                empties += 1
                mapping[s1] = set()
                continue
            if len(ids) != len(set(ids)):
                intra_dupes.add(s1)
            id_set = set(ids)
            mapping[s1] = id_set
            for mid in id_set:
                if mid.startswith("S1-"):
                    self_matches.add(mid)
                elif not mid.startswith(("S2-", "S3-")):
                    wrong_prefix.add(mid)
                elif valid_ids is not None and mid not in valid_ids:
                    unknown.add(mid)

    # Aggregate the per-category findings. Each entry is (offenders, message);
    # only non-empty categories become errors.
    findings = [
        (
            dup_rows,
            "{name}: duplicate source1_entity_id row(s): {ex}. "
            "Each S1 entity may appear on only one row.",
        ),
        (
            intra_dupes,
            "{name}: repeated ID inside a {col} list for: {ex}. "
            "No duplicate IDs are allowed within a list.",
        ),
        (
            self_matches,
            "{name}: {col} contains Source-1 IDs (self-matches): {ex}. "
            "Only S2-/S3- IDs are allowed.",
        ),
        (
            wrong_prefix,
            "{name}: {col} contains IDs without an S2-/S3- prefix: {ex}.",
        ),
        (
            unknown,
            "{name}: {col} references IDs not in the test "
            "Source-2/3 files: {ex}.",
        ),
        (
            required - seen,
            "{name}: required S1 entity(ies) missing: {ex}. "
            "Every entity in test_source1.tsv needs a row (empty = no match).",
        ),
        (
            seen - required,
            "{name}: row(s) using an S1 ID that is not in the test set: {ex}.",
        ),
    ]
    for offenders, message in findings:
        if offenders:
            errors.append(message.format(name=name, ex=examples(offenders), col=col_label))

    print(f"  {name}: {n_rows} rows ({empties} empty, {n_rows - empties} non-empty).")
    return mapping


def validate(matching_path, candidate_path, test_dir, check_ids=False):
    """Validate the submission output(s); return ``(errors, warnings)`` lists.

    ``check_ids`` (``--check-ids``) turns on the optional, memory-heavy check that
    every matched/candidate ID exists in the test Source-2/3 files. It is off by
    default so the common run stays fast and light.
    """
    errors, warnings = [], []

    source1 = os.path.join(test_dir, "test_source1.tsv")
    if not os.path.isfile(source1):
        errors.append(f"Test source1 file not found: {source1} (check --test-dir).")
        return errors, warnings
    required = read_ids(source1)
    print(f"  required S1 entities: {len(required)}")

    if check_ids:
        valid_ids = load_match_targets(test_dir, warnings)
        if valid_ids is not None:
            print(f"  valid S2/S3 match IDs: {len(valid_ids)}")
    else:
        valid_ids = None
        warnings.append(
            "ID-existence check is OFF (the default) — not checking that matched/"
            "candidate IDs exist in the test set. Every other rule is still checked. "
            "Re-run with --check-ids to enable it (needs test_source2/3.tsv; uses "
            "more memory). A nonexistent ID only lowers your score, never rejects "
            "your submission."
        )

    matched = validate_id_list_file(
        matching_path, MATCHING_HEADER, "matched_entity_ids", required, valid_ids, errors
    )

    # candidate_pairs.tsv is optional: if it's absent we skip its checks with a
    # warning (it's still expected in your final submission zip). A missing
    # candidate file never fails this run on its own.
    candidate = None
    if candidate_path and os.path.isfile(candidate_path):
        candidate = validate_id_list_file(
            candidate_path, CANDIDATE_HEADER, "candidate_entity_ids",
            required, valid_ids, errors,
        )
    elif candidate_path:
        warnings.append(
            f"{candidate_path} not found — skipping candidate_pairs.tsv checks. "
            "It is optional here, but your final submission zip must include "
            "output/candidate_pairs.tsv."
        )

    # Soft check: your final matches should come from your blocking candidates.
    # A matched ID absent from candidate_pairs.tsv usually means a pipeline bug,
    # so we warn but never fail on it.
    if matched is not None and candidate is not None:
        offenders = {
            s1 for s1, mids in matched.items() if mids - candidate.get(s1, set())
        }
        if offenders:
            warnings.append(
                f"{len(offenders)} S1 entity(ies) have matched IDs not present in "
                f"candidate_pairs.tsv, e.g. {examples(offenders)}. Final matches "
                "normally come from your blocking candidates — double-check these."
            )

    return errors, warnings


def main():
    parser = argparse.ArgumentParser(
        description="Validate ML Challenge 2026 submission output files before submitting."
    )
    parser.add_argument(
        "--matching",
        "-m",
        default="output/matching_results.tsv",
        help="Path to matching_results.tsv (default: %(default)s)",
    )
    parser.add_argument(
        "--candidate",
        "-c",
        default=None,
        help="Path to candidate_pairs.tsv "
        "(default: output/candidate_pairs.tsv if it exists).",
    )
    parser.add_argument(
        "--test-dir",
        "-t",
        default="dataset/test",
        help="Folder with test_source1/2/3.tsv (default: %(default)s). "
        "test_source2/3.tsv are only read when --check-ids is given.",
    )
    parser.add_argument(
        "--check-ids",
        action="store_true",
        help="Also check that every matched/candidate ID exists in the test "
        "Source-2/3 files. Off by default (loads all S2/S3 IDs into memory — a few "
        "GB on the full test set). A nonexistent ID only lowers your score, so this "
        "is a diagnostic, not a submission gate.",
    )
    args = parser.parse_args()

    # candidate_pairs.tsv is optional; default to the conventional path and let
    # validate() skip (with a warning) if the file isn't there.
    candidate_path = args.candidate or "output/candidate_pairs.tsv"

    print("ML Challenge 2026 — submission validator")
    print(f"  test dir: {args.test_dir}")
    try:
        errors, warnings = validate(
            args.matching, candidate_path, args.test_dir, check_ids=args.check_ids
        )
    except UnicodeDecodeError:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(
            f"  1. A file is not valid UTF-8 text (most likely {args.matching} or "
            f"{candidate_path}). Re-save it as a plain UTF-8, tab-separated .tsv — "
            "not cp1252/Latin-1, and not a compressed or binary file (.gz/.xlsx/"
            ".parquet) renamed to .tsv. In pandas: "
            "df.to_csv(path, sep='\\t', index=False, encoding='utf-8')."
        )
        return 1
    except OSError as exc:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(f"  1. Could not read a file: {exc}.")
        return 1

    print()
    for warning in warnings:
        print(f"WARNING: {warning}")
    if errors:
        print(f"FAIL — {len(errors)} issue(s) to fix before submitting:")
        for i, error in enumerate(errors, 1):
            print(f"  {i}. {error}")
        return 1
    print("PASS — no blocking issues found. Safe to submit.")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /kaggle/working/er/tools/make_subset.py
"""Builds a small dataset directory with the challenge layout for fast experiments.

Sampling is done by *name group* (lower-cased business name + country of the Source 1
record), not by individual entity: 38 % of Source 1 records share their exact name with
another Source 1 record of the same country (chains, franchises, namesakes). Sampling
whole groups keeps those hard-to-separate namesakes together, so the subset stays about as
hard as the full data instead of becoming artificially easy.

For every sampled Source 1 entity all of its true Source 2/3 records are kept. Unmatched
("decoy") Source 2/3 records are sampled independently with rate = --frac * --decoy-mult,
so --decoy-mult 2 produces a subset with roughly the test split's higher decoy density.

Usage: python tools/make_subset.py --data-dir dataset --out-dir subset --frac 0.05 [--decoy-mult 1]
The output has train/ and test/ folders: test/ is a *second, disjoint* sample of the training
data (rate --test-frac, decoy multiplier --test-decoy-mult) with its own ground truth kept in
test/subset_ground_truth.tsv, so a full run of the pipeline can be scored locally.
"""
import argparse
import os
import sys

import polars as pl

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from common import read_tsv  # noqa: E402


def write_tsv(df, path):
    df.write_csv(path, separator="\t", quote_style="never", null_value="")


def sample(s1, s23, gt, ex, lo, hi, decoy_lo, decoy_hi, seed):
    """Keep S1 name groups whose hash lands in [lo, hi) (fractions of 2**64) and decoys in [decoy_lo, decoy_hi)."""
    key = (pl.col("business_name").str.to_lowercase().fill_null("") + "|" + pl.col("country").fill_null("")).hash(seed=seed)
    u = (key.cast(pl.Float64) / float(2**64)).alias("u")
    s1k = s1.with_columns(u).filter((pl.col("u") >= lo) & (pl.col("u") < hi)).drop("u")
    keep_s1 = s1k.select(pl.col("entity_id").alias("source1_entity_id"))
    matched = ex.join(keep_s1, on="source1_entity_id").select(pl.col("matched_entity_ids").alias("entity_id"))
    is_matched = s23.select("entity_id").join(ex.select(pl.col("matched_entity_ids").alias("entity_id")), on="entity_id", how="semi")
    du = (pl.col("entity_id").hash(seed=seed + 1).cast(pl.Float64) / float(2**64)).alias("u")
    decoys = s23.join(is_matched, on="entity_id", how="anti").with_columns(du).filter(
        (pl.col("u") >= decoy_lo) & (pl.col("u") < decoy_hi)).drop("u").select("entity_id")
    keep_23 = pl.concat([matched, decoys]).unique()
    s23k = s23.join(keep_23, on="entity_id", how="semi")
    gtk = gt.join(keep_s1, on="source1_entity_id", how="semi")
    return s1k, s23k, gtk


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data-dir", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--frac", type=float, default=0.05)
    ap.add_argument("--decoy-mult", type=float, default=1.0)
    ap.add_argument("--test-frac", type=float, default=0.02)
    ap.add_argument("--test-decoy-mult", type=float, default=2.0)
    ap.add_argument("--seed", type=int, default=2026)
    args = ap.parse_args()

    s1 = read_tsv(os.path.join(args.data_dir, "train", "train_source1.tsv"))
    s2 = read_tsv(os.path.join(args.data_dir, "train", "train_source2.tsv"))
    s3 = read_tsv(os.path.join(args.data_dir, "train", "train_source3.tsv"))
    gt = read_tsv(os.path.join(args.data_dir, "train", "train_ground_truth.tsv"))
    ex = (gt.with_columns(pl.col("matched_entity_ids").str.split(",")).explode("matched_entity_ids")
          .drop_nulls("matched_entity_ids"))
    s23 = pl.concat([s2, s3])

    plan = [
        ("train", 0.0, args.frac, 0.0, args.frac * args.decoy_mult),
        ("test", args.frac, args.frac + args.test_frac, args.frac * args.decoy_mult,
         args.frac * args.decoy_mult + args.test_frac * args.test_decoy_mult),
    ]
    for split, lo, hi, dlo, dhi in plan:
        d = os.path.join(args.out_dir, split)
        os.makedirs(d, exist_ok=True)
        s1k, s23k, gtk = sample(s1, s23, gt, ex, lo, hi, dlo, dhi, args.seed)
        write_tsv(s1k, os.path.join(d, f"{split}_source1.tsv"))
        write_tsv(s23k.filter(pl.col("entity_id").str.starts_with("S2-")), os.path.join(d, f"{split}_source2.tsv"))
        write_tsv(s23k.filter(pl.col("entity_id").str.starts_with("S3-")), os.path.join(d, f"{split}_source3.tsv"))
        gt_name = "train_ground_truth.tsv" if split == "train" else "subset_ground_truth.tsv"
        write_tsv(gtk, os.path.join(d, gt_name))
        n_m = ex.join(gtk.select("source1_entity_id"), on="source1_entity_id", how="semi").height
        print(f"{split}: S1={s1k.height} S2/3={s23k.height} matched={n_m} decoys={s23k.height - n_m} "
              f"(S2/3 per S1 = {s23k.height / max(s1k.height, 1):.3f}, decoys/S1 = {(s23k.height - n_m) / max(s1k.height, 1):.3f})")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/er/tools/summarize_reports.py
"""Prints the diagnostics of a work directory as markdown tables.

    python tools/summarize_reports.py --work-dir work [--baseline-profile other/profile.json]
"""
import argparse
import json
import os


def load(path):
    try:
        with open(path) as f:
            return json.load(f)
    except (OSError, ValueError):
        return None


def fmt(x, nd=5):
    return f"{x:.{nd}f}" if isinstance(x, float) else str(x)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--work-dir", required=True)
    args = ap.parse_args()
    w = args.work_dir
    rep = load(os.path.join(w, "validation_report.json"))
    if rep:
        at = rep["oof_at_threshold"]
        print("## Out-of-fold validation\n")
        print(f"pairs {rep['n_pairs']}, positive pairs {rep['n_positive_pairs']}, Source 1 entities {rep['n_s1']}, "
              f"stage-1 features {rep['stage1_features']}, folds {rep['folds']}\n")
        print("| metric | value |\n|---|---|")
        for k in ("threshold", "macro_f05", "micro_precision", "micro_recall", "singleton_acc", "singleton_fp", "fp_decoy", "fp_wrong_entity", "links"):
            if k in at:
                print(f"| {k} | {fmt(at[k])} |")
        dr = rep.get("decision_rule")
        if dr:
            print(f"| decision rule | margin {dr['margin']:.2f}, contradiction penalty {dr['contra_penalty']:.2f} |")
            print(f"| nested macro F0.5 (threshold fit on K-1 folds) | {fmt(dr['nested_macro_f05'])} (threshold-only rule: {fmt(dr['nested_macro_f05_threshold_only'])}) |")
        pl_ = rep["pair_level"]
        print(f"| pair-level accuracy | {fmt(pl_['accuracy'])} |\n| pair-level positive rate | {fmt(pl_['positive_rate'])} |"
              f"\n| pair-level precision | {fmt(pl_['precision'])} |\n| pair-level recall | {fmt(pl_['recall'])} |")
        ca = rep.get("candidate_audit") or rep.get("blocking_recall")
        if ca and ca.get("recall_at_k"):
            r = ca["recall_at_k"]
            print("| retrieval recall@1 / @3 / @5 / @10 | " + " / ".join(fmt(r[str(k)], 4) for k in (1, 3, 5, 10) if str(k) in r) + " |")
        if ca and "pair_recall" in ca:
            print(f"| candidate pair recall / complete-entity coverage / oracle macro F0.5 | {fmt(ca['pair_recall'], 4)} / {fmt(ca['entity_complete_coverage'], 4)} / {fmt(ca['oracle_macro_f05'], 4)} |")
            print(f"| candidates per record / channels | {fmt(ca['candidates_per_record'], 2)} / {ca['channels']} |")
            if ca.get("misses"):
                m = ca["misses"]
                print(f"| missed true pairs: ambiguous (address-less namesake) / recoverable | {m['missed_ambiguous']} / {m['missed_recoverable']} (recall on recoverable {fmt(m['recall_on_recoverable'], 4)}) |")
        if rep.get("per_source"):
            print("\n### Per source (link level)\n")
            print("| source | link rows | links | precision | recall | FP decoy | FP wrong entity |\n|---|---|---|---|---|---|---|")
            for k, v in rep["per_source"].items():
                print(f"| {k} | {v['link_rows']} | {v['links']} | {fmt(v['micro_precision'])} | {fmt(v['micro_recall'])} | {v['fp_decoy']} | {v['fp_wrong_entity']} |")
        print("\n### Per fold (at the global threshold)\n")
        print("| fold | entities | macro F0.5 | precision | recall | singleton acc | own best thr | F0.5 @ own thr |\n|---|---|---|---|---|---|---|---|")
        for r in rep["per_fold"]["folds"]:
            print(f"| {r['fold']} | {r['n_entities']} | {fmt(r['macro_f05_at_global_thr'])} | {fmt(r['micro_precision_at_global_thr'])} | "
                  f"{fmt(r['micro_recall_at_global_thr'])} | {fmt(r['singleton_acc_at_global_thr'], 4)} | {r['best_threshold']:.2f} | {fmt(r['macro_f05_at_own_thr'])} |")
        pf = rep["per_fold"]
        print(f"\nmean {fmt(pf['macro_f05_mean'])}, std {fmt(pf['macro_f05_std'])}, min {fmt(pf['macro_f05_min'])}; "
              f"fold-optimal thresholds {pf['best_threshold_range']} (std {fmt(pf['best_threshold_std'], 4)})\n")
        print("### Per country\n")
        print("| country | entities | links | macro F0.5 | precision | recall | singleton FP |\n|---|---|---|---|---|---|---|")
        for c, r in rep["per_country"].items():
            print(f"| {c or '(none)'} | {r['n_entities']} | {r['links']} | {fmt(r['macro_f05'])} | {fmt(r['micro_precision'])} | {fmt(r['micro_recall'])} | {r.get('singleton_fp', '-')} |")
        print("\n### Calibration (nested: fit on K-1 folds, scored on the held-out fold)\n")
        print("| probabilities | log loss | Brier | ECE |\n|---|---|---|---|")
        for k, v in rep["calibration"]["nested"].items():
            print(f"| {k} | {fmt(v['log_loss'])} | {fmt(v['brier'])} | {fmt(v['ece'])} |")
        print(f"\nPlatt: a={rep['calibration']['platt']['a']:.3f}, b={rep['calibration']['platt']['b']:.3f}\n")
        au = rep["leakage_audit"]
        print("### Cross-fold audit\n")
        print(f"cross-fold pair share {fmt(au['cross_fold_pair_share'], 4)}, entities with a cross-fold pair {fmt(au['entities_with_cross_fold_pair'], 4)}\n")
        if "within_fold_only" in au:
            print("| entity set | macro F0.5 | precision | recall |\n|---|---|---|---|")
            for k in ("within_fold_only", "with_cross_fold_pairs"):
                print(f"| {k} | {fmt(au[k]['macro_f05'])} | {fmt(au[k]['micro_precision'])} | {fmt(au[k]['micro_recall'])} |")
        print("\n### Threshold sweep (calibrated scale, OOF)\n")
        print("| thr | macro F0.5 | precision | recall | links |\n|---|---|---|---|---|")
        for r in rep["threshold_sweep"]:
            if round(r["threshold"] * 100) % 10 == 0 or abs(r["threshold"] - rep["threshold"]) < 1e-9:
                print(f"| {r['threshold']:.2f} | {fmt(r['macro_f05'])} | {fmt(r['micro_precision'])} | {fmt(r['micro_recall'])} | {r['links']} |")
    lk = load(os.path.join(w, "leakage_check.json"))
    if lk:
        print("\n## Leakage check\n")
        print(f"result: **{'OK' if lk['ok'] else 'FAILED'}**; problems: {lk['problems'] or 'none'}\n")
        print("| check | value |\n|---|---|")
        print(f"| name groups split across folds | {lk['name_groups']['split_across_folds']} / {lk['name_groups']['n']} |")
        print(f"| matched records outside their entity's fold | {lk['matched_records_fold_mismatch']} |")
        print(f"| duplicated pids in OOF | {lk['oof']['duplicate_pids']} |")
        print(f"| positive pairs across folds | {lk['oof']['positive_pairs_cross_fold']} |")
        print(f"| fold-only Indic tokens leaked into own-fold lexicon | {lk['lexicon']['leaked_into_own_fold_lexicon']} / {lk['lexicon']['fold_only_tokens_checked']} checked |")
        print(f"| label-shuffle canary OOF AUC | {fmt(lk['label_shuffle_canary_auc'], 4)} |")
        print(f"| strongest single features (AUC) | {lk['top_single_feature_auc'][:5]} |")
        print(f"| bookkeeping columns among model features | {lk['model_features_bookkeeping'] or 'none'} |")
    ta = load(os.path.join(w, "threshold_analysis.json"))
    if ta:
        print("\n## Threshold transfer to the test split\n")
        print(f"decoy-density ratio r = {ta['decoy_ratio']:.3f}; pair-level odds ratio for the prior shift = {ta['odds_ratio_prior_shift']:.3f}; chosen: {ta['chosen_method']}\n")
        print("| method | threshold | OOF macro F0.5 (train mix) | OOF macro F0.5 (test-like mix) | precision | recall |\n|---|---|---|---|---|---|")
        for k, v in ta["candidates"].items():
            a, b = ta["oof_plain_at"][k], ta["oof_density_adjusted_at"][k]
            print(f"| {k} | {v:.3f} | {fmt(a['macro_f05'])} | {fmt(b['macro_f05'])} | {fmt(b['micro_precision'])} | {fmt(b['micro_recall'])} |")
    ck_root = os.path.join(w, "checkpoints")
    if os.path.isdir(ck_root):
        print("\n## Checkpoints\n")
        latest = open(os.path.join(ck_root, "LATEST")).read().strip() if os.path.exists(os.path.join(ck_root, "LATEST")) else None
        print("| checkpoint | created | git | stages | rounds | OOF macro F0.5 | global thr | policies |\n|---|---|---|---|---|---|---|---|")
        for name in sorted(os.listdir(ck_root)):
            m = load(os.path.join(ck_root, name, "manifest.json"))
            if not m:
                continue
            ta = m.get("train_args") or {}
            pols = sorted(f[:-5] for f in os.listdir(os.path.join(ck_root, name, "thresholds"))) if os.path.isdir(os.path.join(ck_root, name, "thresholds")) else []
            print(f"| {name}{' (LATEST)' if name == latest else ''} | {m.get('created')} | {m.get('git_commit')} | {','.join(m.get('stages_done', []))} | "
                  f"{ta.get('rounds1')}/{ta.get('rounds2')} | {fmt(m.get('oof_macro_f05')) if m.get('oof_macro_f05') is not None else '-'} | "
                  f"{m.get('global_threshold', '-')} | {', '.join(pols)} |")
    ea = load(os.path.join(w, "error_analysis.json"))
    if ea and "categories" in ea:
        print("\n## Entity-level error analysis (OOF, selected policy; tools/error_analysis.py)\n")
        print(f"macro F0.5 {fmt(ea['macro_f05'])}; points lost {fmt(ea['total_points_lost'])}; missing multiple matches: {ea['missing_multi']['entities']} entities "
              f"({fmt(ea['missing_multi']['points_lost'])} points)\n")
        print("| category (fix priority) | entities | macro F0.5 points lost | share of loss | no address | S1 namesakes |\n|---|---|---|---|---|---|")
        for c in ea["priority"]:
            v = ea["categories"][c]
            print(f"| {c} | {v['entities']} | {fmt(v['macro_f05_points_lost'])} | {v['share_of_total_loss']:.3f} | {fmt(v['records']['no_address'], 3) if v['records']['no_address'] is not None else '-'} | "
                  f"{fmt(v['records']['s1_has_namesakes'], 3) if v['records']['s1_has_namesakes'] is not None else '-'} |")
    prof = load(os.path.join(w, "profile.json"))
    if prof:
        print("\n## Runtime profile\n")
        print("| stage | wall s | cpu s | cpu/wall | workers |\n|---|---|---|---|---|")
        tot = 0.0
        for k, v in prof.items():
            tot += v["wall_s"]
            print(f"| {k} | {v['wall_s']:.1f} | {v['cpu_s']:.1f} | {v['cpu_per_wall']:.2f} | {v['workers']} |")
        print(f"| **total** | {tot:.1f} | | | |")
    ab = load(os.path.join(w, "ablation.json"))
    if ab:
        base = ab["results"].get("baseline")
        print("\n## Feature ablation (OOF, one group removed at a time)\n")
        print("| removed group | #feats | macro F0.5 | delta | recall | delta recall | precision | thr | fold std |\n|---|---|---|---|---|---|---|---|---|")
        for g, r in ab["results"].items():
            d = r["macro_f05"] - base["macro_f05"] if base else 0
            dr = r["micro_recall"] - base["micro_recall"] if base else 0
            print(f"| {g} | {len(r['dropped'])} | {fmt(r['macro_f05'])} | {d:+.5f} | {fmt(r['micro_recall'])} | {dr:+.5f} | {fmt(r['micro_precision'])} | {r['threshold']:.2f} | {fmt(r['fold_std'])} |")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/er/tools/error_analysis.py
"""Entity-level error analysis of the out-of-fold predictions under the selected policy.

The metric is macro F0.5 over Source 1 entities, so errors are attributed per *entity* and weighted
by the F0.5 each one loses (1 - F0.5 of the entity, divided by the number of entities = the macro
F0.5 points the entity costs). Every entity with F0.5 < 1 is assigned to the first matching
category of

  singleton_fp     : a singleton (no true record) received a link             -> F0.5 = 0
  fp_wrong_entity  : it received a record that belongs to another entity
  fp_decoy         : it received an unmatched ("decoy") record
  blocking_miss    : one of its true records was never proposed as a candidate
  wrong_top        : a true record was proposed but another entity outscored it
  below_threshold  : a true record was the best candidate but rejected (by the threshold, the margin
                     over the runner-up, or the contradiction penalty - reported separately)

and the flag `missing_multi` (entity with >= 2 true records, some but not all found) is counted on
top. For every category: entities, macro F0.5 points lost, the share of the total loss, and the
composition of the affected records (Indic script, alias, domain-style name, no address, Source 1
with namesakes, Source 3 share) plus a per-country split; then the worst examples with raw strings.
Categories are printed in the order of their loss, which is the priority order for fixes.

    python tools/error_analysis.py --data-dir DATA --work-dir WORK [--policy FILE] [--examples 15]

Writes <work>/error_analysis.json and prints markdown.
"""
import argparse
import json
import os
import sys

import numpy as np
import polars as pl

SRC = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "src")
sys.path.insert(0, SRC)
from checkpoint import Checkpoint  # noqa: E402
from model import best_candidates  # noqa: E402
from pair_features import truth_pairs  # noqa: E402
from threshold_policy import ThresholdPolicy  # noqa: E402

CATEGORIES = ("singleton_fp", "fp_wrong_entity", "fp_decoy", "blocking_miss", "wrong_top", "below_threshold")


def share(df, col):
    return float(df[col].cast(pl.Float64).mean()) if df.height else None


def load_policy(work_dir, path):
    if path:
        return ThresholdPolicy.load(path)
    ckpt = Checkpoint.resolve(work_dir, None)
    for cand in ([ckpt.path(os.path.join("thresholds", "selected.json")), ckpt.path(os.path.join("thresholds", "global_train.json"))] if ckpt else []) \
            + [os.path.join(work_dir, "threshold_policy.json")]:
        if os.path.exists(cand):
            return ThresholdPolicy.load(cand)
    with open(os.path.join(work_dir, "model_meta.json")) as f:
        meta = json.load(f)
    return ThresholdPolicy(meta["threshold"], name="model_meta_global", **meta.get("decision", {}))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data-dir", required=True)
    ap.add_argument("--work-dir", required=True)
    ap.add_argument("--policy", default=None, help="threshold policy JSON (default: the selected policy of the latest checkpoint)")
    ap.add_argument("--examples", type=int, default=15)
    ap.add_argument("--out", default=None, help="output JSON (default <work>/error_analysis.json)")
    args = ap.parse_args()
    d = os.path.join(args.work_dir, "train")
    policy = load_policy(args.work_dir, args.policy)
    cols = ["rid", "entity_id", "src", "country", "business_name", "business_address", "n_core", "f_indic", "f_alias", "n_domain", "a_norm", "group"]
    rec = pl.read_parquet(os.path.join(d, "records.parquet"), columns=cols)
    rec = rec.with_columns(pl.col("rid").cast(pl.UInt32), (pl.col("n_domain").fill_null("") != "").alias("f_domain"),
                           (pl.col("a_norm").fill_null("") == "").alias("f_noaddr"))
    s1 = rec.filter(pl.col("src") == 1)
    namesakes = s1.group_by("group").agg(pl.len().alias("n_namesakes"))
    s1 = s1.join(namesakes, on="group")
    truth = truth_pairs(rec, args.data_dir).drop("label")
    oof = pl.read_parquet(os.path.join(d, "oof.parquet"))
    contra = oof["contra"].to_numpy() if "contra" in oof.columns else None
    p = oof["p2_cal"].to_numpy()
    best = best_candidates(oof.select("t_rid", "s_rid"), p, contra)
    thr = np.full(best.height, policy.default)
    pb, p2 = best["p"].to_numpy().astype(np.float64), best["p2nd"].to_numpy()
    ok_thr = pb >= thr
    ok_margin = (pb - p2) >= policy.margin
    ok_contra = (~best["contra"].to_numpy()) | (pb >= thr + policy.contra_penalty) if policy.contra_penalty > 0 else np.ones(len(pb), dtype=bool)
    accepted = ok_thr & ok_margin & ok_contra
    best = best.with_columns(pl.Series("thr", thr), pl.Series("accepted", accepted), pl.Series("ok_thr", ok_thr),
                             pl.Series("ok_margin", ok_margin), pl.Series("ok_contra", ok_contra)).drop("i")
    links = best.filter(pl.col("accepted"))

    # ---- per true record: outcome
    tp_pairs = truth.join(oof.select("t_rid", "s_rid", pl.col("p2_cal").alias("p_true")), on=["t_rid", "s_rid"], how="left")
    tp_pairs = tp_pairs.join(best.select("t_rid", pl.col("s_rid").alias("best_s"), pl.col("p").alias("p_best"), "accepted", "ok_thr", "ok_margin", "ok_contra"), on="t_rid", how="left")
    outcome = (
        pl.when(pl.col("p_true").is_null()).then(pl.lit("blocking_miss"))
        .when(pl.col("best_s") != pl.col("s_rid")).then(pl.lit("wrong_top"))
        .when(~pl.col("accepted")).then(pl.lit("below_threshold"))
        .otherwise(pl.lit("found"))
    )
    reject_kind = (
        pl.when(pl.col("accepted") | (pl.col("best_s") != pl.col("s_rid")) | pl.col("p_true").is_null()).then(pl.lit(None))
        .when(~pl.col("ok_thr")).then(pl.lit("threshold"))
        .when(~pl.col("ok_margin")).then(pl.lit("margin"))
        .otherwise(pl.lit("contradiction"))
    )
    tp_pairs = tp_pairs.with_columns(outcome.alias("outcome"), reject_kind.alias("reject_kind"))

    # ---- per accepted link: kind
    lk = links.join(truth.select("t_rid", pl.col("s_rid").alias("true_s")), on="t_rid", how="left").with_columns(
        pl.when(pl.col("true_s") == pl.col("s_rid")).then(pl.lit("tp"))
        .when(pl.col("true_s").is_null()).then(pl.lit("fp_decoy")).otherwise(pl.lit("fp_wrong_entity")).alias("kind"))

    # ---- per entity: F0.5 and flags
    s_ids = s1.select("rid").rename({"rid": "s_rid"})
    nt = truth.group_by("s_rid").len().rename({"len": "nt"})
    npred = lk.group_by("s_rid").agg(pl.len().alias("np"), (pl.col("kind") == "tp").sum().alias("tp"),
                                     (pl.col("kind") == "fp_decoy").sum().alias("n_fp_decoy"), (pl.col("kind") == "fp_wrong_entity").sum().alias("n_fp_wrong"))
    per_true = tp_pairs.group_by("s_rid").agg(*[(pl.col("outcome") == oc).sum().alias("n_" + oc) for oc in ("blocking_miss", "wrong_top", "below_threshold")],
                                              *[(pl.col("reject_kind") == rk).sum().alias("n_rej_" + rk) for rk in ("threshold", "margin", "contradiction")])
    e = s_ids.join(nt, on="s_rid", how="left").join(npred, on="s_rid", how="left").join(per_true, on="s_rid", how="left").fill_null(0)
    e = e.with_columns(
        pl.when(pl.col("np") > 0).then(pl.col("tp") / pl.col("np")).otherwise(0.0).alias("prec"),
        pl.when(pl.col("nt") > 0).then(pl.col("tp") / pl.col("nt")).otherwise(0.0).alias("rec"))
    f = (pl.when((pl.col("nt") == 0) & (pl.col("np") == 0)).then(1.0).when(pl.col("nt") == 0).then(0.0)
         .when(pl.col("prec") + pl.col("rec") > 0).then(1.25 * pl.col("prec") * pl.col("rec") / (0.25 * pl.col("prec") + pl.col("rec"))).otherwise(0.0))
    e = e.with_columns(f.alias("f")).with_columns((1.0 - pl.col("f")).alias("loss"))
    cat = (pl.when(pl.col("f") >= 1.0).then(pl.lit("perfect"))
           .when((pl.col("nt") == 0) & (pl.col("np") > 0)).then(pl.lit("singleton_fp"))
           .when(pl.col("n_fp_wrong") > 0).then(pl.lit("fp_wrong_entity"))
           .when(pl.col("n_fp_decoy") > 0).then(pl.lit("fp_decoy"))
           .when(pl.col("n_blocking_miss") > 0).then(pl.lit("blocking_miss"))
           .when(pl.col("n_wrong_top") > 0).then(pl.lit("wrong_top"))
           .when(pl.col("n_below_threshold") > 0).then(pl.lit("below_threshold")).otherwise(pl.lit("other")))
    e = e.with_columns(cat.alias("category"), ((pl.col("nt") >= 2) & (pl.col("tp") > 0) & (pl.col("tp") < pl.col("nt"))).alias("missing_multi"))
    e = e.join(s1.select("rid", "country", "n_namesakes", pl.col("f_noaddr").alias("s_noaddr")).rename({"rid": "s_rid"}), on="s_rid")
    n_ent = e.height
    macro = float(e["f"].mean())
    total_loss = float(e["loss"].sum())

    t_side = rec.select(pl.col("rid").alias("t_rid"), pl.col("src").alias("t_src"), "f_indic", "f_alias", "f_domain", "f_noaddr",
                        pl.col("business_name").alias("t_name"), pl.col("business_address").alias("t_addr"))
    s_side = s1.select(pl.col("rid").alias("s_rid"), "country", "n_namesakes", pl.col("business_name").alias("s_name"), pl.col("business_address").alias("s_addr"))
    fn = tp_pairs.join(t_side, on="t_rid").join(s_side, on="s_rid")
    fp = lk.filter(pl.col("kind") != "tp").join(t_side, on="t_rid").join(s_side, on="s_rid")

    def composition(sub):
        return {"records": sub.height, "indic": share(sub, "f_indic"), "alias": share(sub, "f_alias"), "domain": share(sub, "f_domain"),
                "no_address": share(sub, "f_noaddr"), "s1_has_namesakes": float((sub["n_namesakes"] > 1).mean()) if sub.height else None,
                "source3_share": float((sub["t_src"] == 3).mean()) if sub.height else None}

    cats = {}
    for c in CATEGORIES:
        sub = e.filter(pl.col("category") == c)
        loss = float(sub["loss"].sum())
        rec_sub = {"singleton_fp": fp.join(sub.select("s_rid"), on="s_rid"), "fp_wrong_entity": fp.filter(pl.col("kind") == "fp_wrong_entity"),
                   "fp_decoy": fp.filter(pl.col("kind") == "fp_decoy"), "blocking_miss": fn.filter(pl.col("outcome") == "blocking_miss"),
                   "wrong_top": fn.filter(pl.col("outcome") == "wrong_top"), "below_threshold": fn.filter(pl.col("outcome") == "below_threshold")}[c]
        cats[c] = {"entities": sub.height, "entity_share": sub.height / n_ent, "macro_f05_points_lost": loss / n_ent,
                   "share_of_total_loss": loss / total_loss if total_loss else 0.0, "mean_entity_f05": float(sub["f"].mean()) if sub.height else None,
                   "by_country": {k: {"entities": int(v), "points_lost": float(sub.filter(pl.col("country") == k)["loss"].sum() / n_ent)}
                                  for k, v in sub["country"].value_counts().iter_rows()},
                   "missing_multi_entities": int(sub["missing_multi"].sum()),
                   "records": composition(rec_sub)}
        if c == "below_threshold":
            cats[c]["rejected_by"] = {k: int(fn.filter(pl.col("reject_kind") == k).height) for k in ("threshold", "margin", "contradiction")}
    order = sorted(CATEGORIES, key=lambda c: -cats[c]["macro_f05_points_lost"])
    out = {
        "policy": policy.to_dict(), "entities": n_ent, "macro_f05": macro, "total_points_lost": total_loss / n_ent,
        "perfect_entities": float((e["f"] >= 1.0).mean()), "links": int(links.height),
        "true_pairs": int(truth.height), "records_found": int(fn.filter(pl.col("outcome") == "found").height),
        "missing_multi": {"entities": int(e["missing_multi"].sum()), "points_lost": float(e.filter(pl.col("missing_multi"))["loss"].sum() / n_ent)},
        "priority": order, "categories": {c: cats[c] for c in order},
        "base_rates": {"true_pairs": composition(fn), "s1_has_namesakes": float((fn["n_namesakes"] > 1).mean())},
    }
    ex = {}
    for c in ("blocking_miss", "wrong_top", "below_threshold"):
        sub = fn.filter(pl.col("outcome") == c).sort("p_true", descending=True, nulls_last=True).head(args.examples)
        ex[c] = [{"s1": r[0], "s1_addr": r[1], "record": r[2], "record_addr": r[3], "p_true": r[4], "p_best": r[5], "n_namesakes": r[6]}
                 for r in sub.select("s_name", "s_addr", "t_name", "t_addr", "p_true", "p_best", "n_namesakes").iter_rows()]
    for c in ("fp_decoy", "fp_wrong_entity"):
        sub = fp.filter(pl.col("kind") == c).sort("p", descending=True).head(args.examples)
        ex[c] = [{"s1": r[0], "s1_addr": r[1], "record": r[2], "record_addr": r[3], "p": r[4], "p2nd": r[5], "n_namesakes": r[6]}
                 for r in sub.select("s_name", "s_addr", "t_name", "t_addr", "p", "p2nd", "n_namesakes").iter_rows()]
    sub = fp.join(e.filter(pl.col("category") == "singleton_fp").select("s_rid"), on="s_rid").sort("p", descending=True).head(args.examples)
    ex["singleton_fp"] = [{"s1": r[0], "s1_addr": r[1], "record": r[2], "record_addr": r[3], "p": r[4]}
                          for r in sub.select("s_name", "s_addr", "t_name", "t_addr", "p").iter_rows()]
    out["examples"] = ex
    with open(args.out or os.path.join(args.work_dir, "error_analysis.json"), "w") as f:
        json.dump(out, f, indent=1, ensure_ascii=False)

    f_ = lambda x, nd=3: "-" if x is None else f"{x:.{nd}f}"  # noqa: E731
    print(f"policy {policy.describe()}\nentities {n_ent}; OOF macro F0.5 {macro:.5f}; perfect entities {out['perfect_entities']:.4f}; "
          f"links {links.height}; true records found {out['records_found']}/{truth.height}")
    print("\n| category (fix priority) | entities | share | macro F0.5 points lost | share of loss | records | indic | alias | domain | no address | S1 namesakes | S3 share |")
    print("|---|---|---|---|---|---|---|---|---|---|---|---|")
    for c in order:
        v, r = cats[c], cats[c]["records"]
        print(f"| {c} | {v['entities']} | {v['entity_share']:.4f} | {v['macro_f05_points_lost']:.5f} | {v['share_of_total_loss']:.3f} | {r['records']} | "
              f"{f_(r['indic'])} | {f_(r['alias'])} | {f_(r['domain'])} | {f_(r['no_address'])} | {f_(r['s1_has_namesakes'])} | {f_(r['source3_share'])} |")
    print(f"\nmissing multiple matches (entity with >= 2 true records, some found): {out['missing_multi']['entities']} entities, "
          f"{out['missing_multi']['points_lost']:.5f} points; below-threshold rejections by rule: {cats['below_threshold'].get('rejected_by')}")
    print(f"base rates over all true pairs: {json.dumps(out['base_rates'])}")
    print("\n| category | " + " | ".join(f"{c} entities / points" for c in sorted({k for v in cats.values() for k in v['by_country']})) + " |")
    countries = sorted({k for v in cats.values() for k in v["by_country"]})
    print("|---|" + "---|" * len(countries))
    for c in order:
        print(f"| {c} | " + " | ".join(f"{cats[c]['by_country'].get(k, {}).get('entities', 0)} / {cats[c]['by_country'].get(k, {}).get('points_lost', 0.0):.5f}" for k in countries) + " |")
    for k, rows in ex.items():
        print(f"\n### {k}")
        for r in rows[:6]:
            print("  ", json.dumps(r, ensure_ascii=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/er/tests/test_textnorm.py
"""Edge cases of the text normalisation (run: python -m pytest tests -q).

Every case here is a token that the previous, more aggressive cleaning destroyed or
mangled: country names that are part of the business name, articles that carry identity,
ambiguous two-letter legal forms, initials, abbreviations, and address state codes.
"""
import os
import sys

import pytest

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
import textnorm  # noqa: E402
from textnorm import norm_addr, norm_name, normalize_record  # noqa: E402


def core(name, country=None):
    return norm_name(name, country)["n_core"]


def full(name, country=None):
    return norm_name(name, country)["n_full"]


# ------------------------------------------------------------------ country tokens
@pytest.mark.parametrize("name,country,expected_core", [
    ("Air India Ltd", "India", "air india"),
    ("Reliance India", "India", "reliance india"),
    ("India Cements Limited", "India", "india cements"),
    ("Air France SAS", "France", "air france"),
    ("Toys R Us", "US", "toys r us"),
    ("Us Engineering Private Limited", "US", "us engineering"),
])
def test_country_token_kept_when_part_of_name(name, country, expected_core):
    assert core(name, country) == expected_core


def test_parenthesised_country_tag_removed_everywhere():
    r = norm_name("Gulabchand Joy (India) Limited", "India")
    assert r["n_core"] == "gulabchand joy"
    assert "india" not in r["n_full"]


def test_ocr_corrupted_country_tag_removed():
    # "(lndia)" is the corrupted "(India)" seen in Source 2/3 records
    assert core("Limited Gulabchand Joy (lndia)", "India") == "gulabchand joy"


def test_parenthesised_non_country_content_kept():
    assert core("Elegance (Events) Private Limited", "India") == "elegance events"


def test_trailing_country_tag_dropped_only_from_long_names():
    assert core("Tata Consultancy Services India", "India") == "tata consultancy svcs"
    assert "india" in full("Tata Consultancy Services India", "India")
    # short names keep the trailing country token: dropping it would change the identity
    assert core("Air India", "India") == "air india"


# ------------------------------------------------------------------ articles / short tokens
@pytest.mark.parametrize("name,expected_core", [
    ("The Dent Diner", "dent diner"),
    ("The Gap Inc", "the gap"),
    ("The One", "the one"),
    ("La Poste", "la poste"),
    ("El Lincoln", "el lincoln"),
    ("Le Montessori School", "montessori school"),
])
def test_articles_kept_when_name_would_collapse(name, expected_core):
    assert core(name) == expected_core


def test_article_always_kept_in_full_name():
    assert full("The Dent Diner") == "the dent diner"


# ------------------------------------------------------------------ initials / abbreviations
@pytest.mark.parametrize("name,expected_core", [
    ("A & W Restaurants", "aw restaurants"),
    ("D & L Metro LLC", "dl metro"),
    ("J.P. Morgan Chase", "jp morgan chase"),
    ("L A Fitness", "la fitness"),
    ("U Q & Z Kimco", "uqz kimco"),
])
def test_initials_merged(name, expected_core):
    assert core(name) == expected_core


def test_initials_and_compact_form_agree():
    assert core("D&L Metro") == core("D & L Metro") == core("DL Metro")


def test_abbreviations_collapse_to_short_form_not_expanded():
    assert core("Tech Mahindra") == "tech mahindra"
    assert core("Bharat Jai Technologies") == "bharat jai tech"
    assert core("Bharat Jai Technology") == "bharat jai tech"
    assert core("Acme International") == core("Acme Intl")
    assert core("Acme Manufacturing Co") == core("Acme Mfg")


# ------------------------------------------------------------------ legal suffixes
@pytest.mark.parametrize("name,expected_core,expected_legal", [
    ("PC World Ltd", "pc world", "ltd"),
    ("Crystal Lending PC", "crystal lending", "pc"),
    ("Ag Supply Inc", "ag supply", "inc"),
    ("SA Toys", "sa toys", ""),
    ("Tiffany & Co", "tiffany", "co"),
    ("Namaskar & Co Company", "namaskar", "co"),
    ("PA Impex Private Limited", "pa impex", "ltd pvt"),
    ("Chaturthi Digital Pvt Ltd", "chaturthi digital", "ltd pvt"),
    ("Chaturthi Digital Private Limited", "chaturthi digital", "ltd pvt"),
])
def test_ambiguous_legal_tokens_are_contextual(name, expected_core, expected_legal):
    r = norm_name(name)
    assert r["n_core"] == expected_core
    assert r["n_legal"] == expected_legal


def test_strict_legal_token_removed_even_when_shuffled():
    # corrupted records shuffle tokens: the legal form can land in the middle
    assert core("Federal LLC Minerals Star") == "federal minerals star"
    assert core("Private Bhiwandi Refinery Limited") == "bhiwandi refinery"


def test_legal_conflict_information_preserved():
    assert norm_name("Acme LLC")["n_legal"] == "llc"
    assert norm_name("Acme Inc")["n_legal"] == "inc"


# ------------------------------------------------------------------ misc name handling
def test_id_tag_and_messrs_prefix_removed():
    assert core("M/s Ram Traders (ID: 123)") == "ram traders"


def test_square_brackets_are_not_removed_content():
    assert core("Total [Farms]") == "total farms"


def test_domain_name_extracted():
    r = norm_name("mérrittentertainment.com")
    assert r["n_domain"] == "merrittentertainment"


def test_alias_parts_split():
    r = norm_name("Acme Holdings DBA: Bob's Diner")
    assert r["f_alias"] is True
    assert r["n_parts"] == "acme holdings|bobs diner"


def test_possessive_apostrophe_removed():
    assert core("Orelee's Barbershop") == "orelees barbershop"


def test_empty_name():
    r = norm_name(None, "US")
    assert r["n_core"] == "" and r["n_full"] == ""


# ------------------------------------------------------------------ addresses
def test_state_code_components_kept():
    # "DE" and "LA" are Delaware / Louisiana here, not French articles
    assert "de" in norm_addr("123 Main St, Wilmington, DE", "US")["a_comp"].split(",")
    assert "la" in norm_addr("2571 Lark Street, Baton Rouge, LA", "US")["a_comp"].split(",")


def test_article_dropped_inside_component():
    assert norm_addr("12 rue de la Paix, Paris", "France")["a_comp"].startswith("12 rue paix")


def test_french_abbreviations_only_in_france():
    assert "rue" in norm_addr("12 r du Bac, Paris", "France")["a_norm"].split()
    assert "rue" not in norm_addr("R K Puram, New Delhi", "India")["a_norm"].split()
    assert "r" in norm_addr("R K Puram, New Delhi", "India")["a_norm"].split()


def test_full_state_name_canonicalised():
    a = norm_addr("713 Butternut Street, Syracuse, New York", "US")
    b = norm_addr("713 Butternut St, Syracuse, NY", "US")
    assert a["a_comp"] == b["a_comp"]


def test_house_number_leading_zero_stripped():
    assert norm_addr("011014 Island Dr, Town Of Gibraltar, Wisconsin", "US")["a_hn"] == "11014"


def test_normalize_record_contract():
    r = normalize_record("Acme Inc", "1 Main St, Springfield, IL", "US")
    for k in ("n_full", "n_core", "n_parts", "n_compact", "n_domain", "n_legal", "f_indic", "f_alias",
              "a_norm", "a_comp", "a_num", "a_hn", "a_street", "a_key", "feat_name", "feat_addr", "feat_cross", "feat_nchar"):
        assert k in r


def test_indic_lexicon_applied():
    textnorm.set_lexicon({"name": {textnorm.indic_key("राम"): "ram"}, "addr": {}})
    try:
        assert core("राम मार्केटिंग", "India").split()[0] == "ram"  # second token needs unidecode (optional)
    finally:
        textnorm.set_lexicon({"name": {}, "addr": {}})


In [ ]:
%%writefile /kaggle/working/er/tests/test_calibrate.py
"""Calibration must be stable on the near-separable scores a good matcher produces."""
import os
import sys

import numpy as np

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
import calibrate  # noqa: E402


def _data(n=20000, seed=0, sharp=1.0, shift=0.0):
    rng = np.random.default_rng(seed)
    y = (rng.random(n) < 0.25).astype(float)
    z = np.where(y == 1, 4.0, -4.0) + rng.normal(0, 2.0, n)
    p = calibrate.sigmoid(sharp * z + shift)  # over-confident (sharp > 1) / biased (shift) scores
    return p, y


def test_platt_recovers_temperature_and_bias():
    p, y = _data(sharp=2.0, shift=1.0)
    cal = calibrate.fit_platt(p, y)
    assert "fallback" not in cal
    assert cal["log_loss"] < cal["identity_log_loss"]
    assert cal["b"] < 0  # undoes the positive shift
    before = calibrate.calibration_metrics(p, y)["ece"]
    after = calibrate.calibration_metrics(calibrate.apply(cal, p), y)["ece"]
    assert after < before / 2


def test_platt_does_not_diverge_on_separable_scores():
    rng = np.random.default_rng(1)
    y = (rng.random(50000) < 0.25).astype(float)
    p = np.where(y == 1, 1 - 1e-7, 1e-7)  # perfectly separable, extreme scores
    p[rng.random(len(p)) < 0.01] = 0.5
    cal = calibrate.fit_platt(p, y)
    assert np.isfinite(cal["a"]) and np.isfinite(cal["b"])
    assert abs(cal["a"]) < 1e3
    q = calibrate.apply(cal, p)
    assert calibrate.calibration_metrics(q, y)["log_loss"] <= calibrate.calibration_metrics(p, y)["log_loss"] + 1e-9


def test_platt_falls_back_to_identity_when_it_cannot_help():
    p, y = _data()
    cal = calibrate.fit_platt(calibrate.apply(calibrate.fit_platt(p, y), p), y)
    q = calibrate.apply(cal, p)
    assert np.allclose(np.sort(q), np.sort(calibrate.apply({"method": "platt", "a": cal["a"], "b": cal["b"]}, p)))


def test_isotonic_monotone_and_apply():
    p, y = _data(sharp=3.0)
    cal = calibrate.fit_isotonic(p, y)
    assert all(a <= b + 1e-12 for a, b in zip(cal["y"], cal["y"][1:]))
    q = calibrate.apply(cal, p)
    assert q.min() >= 0 and q.max() <= 1


def test_prior_shift_moves_odds():
    p = np.array([0.5, 0.9])
    q = calibrate.shift_odds(p, 0.5)  # half the positive odds
    assert np.allclose(q, [1 / 3, 0.9 * 0.5 / (0.9 * 0.5 + 0.1)])


def test_nested_report_keys():
    p, y = _data()
    folds = np.arange(len(y)) % 3
    rep = calibrate.nested_calibration_report(p, y, folds)
    assert set(rep) == {"uncalibrated", "platt", "isotonic"}
    assert rep["platt"]["ece"] <= rep["uncalibrated"]["ece"] + 1e-3


In [ ]:
%%writefile /kaggle/working/er/tests/test_checkpoint.py
"""Checkpoint directory: manifest, stages, LATEST pointer and the legacy links."""
import json
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from checkpoint import Checkpoint, link_or_copy  # noqa: E402


def test_manifest_stages_and_latest(tmp_path):
    work = str(tmp_path / "work")
    c = Checkpoint(work, "ckpt_a")
    assert not c.exists() and Checkpoint.resolve(work) is None
    c.update_manifest(train_args={"rounds1": 3})
    assert c.exists() and c.manifest()["train_args"] == {"rounds1": 3}
    c.mark_stage("stage1")
    c.mark_stage("stage1")
    assert c.stage_done("stage1") and not c.stage_done("stage2")
    assert c.manifest()["stages_done"] == ["stage1"]
    for f in ("stage1.txt", "model_meta.json", "calibration.json"):
        with open(c.path(f), "w") as fh:
            fh.write("{}")
    os.makedirs(c.path("thresholds"))
    with open(c.path("thresholds/selected.json"), "w") as fh:
        json.dump({"default": 0.5}, fh)
    c.set_latest()
    assert Checkpoint.resolve(work).name == "ckpt_a"
    assert Checkpoint.resolve(work, c.dir).name == "ckpt_a"
    assert os.path.exists(os.path.join(work, "model_meta.json"))
    assert json.load(open(os.path.join(work, "threshold_policy.json")))["default"] == 0.5
    assert [x.name for x in Checkpoint.list(work)] == ["ckpt_a"]
    c.log_experiment({"kind": "test", "x": 1})
    assert c.experiments()[0]["x"] == 1 and c.experiments()[0]["checkpoint"] == "ckpt_a"
    assert "files" in c.manifest() and "manifest.json" in c.manifest()["files"]


def test_link_or_copy_replaces(tmp_path):
    a, b = tmp_path / "a.txt", tmp_path / "sub" / "b.txt"
    a.write_text("1")
    link_or_copy(str(a), str(b))
    assert b.read_text() == "1"
    a2 = tmp_path / "a2.txt"
    a2.write_text("2")
    link_or_copy(str(a2), str(b))
    assert b.read_text() == "2"


In [ ]:
%%writefile /kaggle/working/er/tests/test_decision.py
"""Many-to-one assignment with confidence-based rejection, the vectorised link table and the
nested decision-rule selection."""
import os
import sys

import numpy as np
import polars as pl

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from model import accept, assign, best_candidates, contradiction_flag  # noqa: E402
from thresholds import accept_mask, link_table, metrics_at, select_rule  # noqa: E402


def _meta():
    # record 10: candidates 1 (0.9) and 2 (0.8) -> close call; record 11: 1 (0.95), 3 (0.1);
    # record 12: single candidate 2 (0.6); record 13: 3 (0.85, contradicted) and 1 (0.2)
    meta = pl.DataFrame({"t_rid": np.array([10, 10, 11, 11, 12, 13, 13], dtype=np.uint32),
                         "s_rid": np.array([1, 2, 1, 3, 2, 3, 1], dtype=np.uint32)})
    p = np.array([0.9, 0.8, 0.95, 0.1, 0.6, 0.85, 0.2], dtype=np.float32)
    contra = np.array([False, False, False, False, False, True, False])
    return meta, p, contra


def test_best_candidates_runner_up_and_flags():
    meta, p, contra = _meta()
    b = best_candidates(meta, p, contra).sort("t_rid")
    assert b["t_rid"].to_list() == [10, 11, 12, 13]
    assert b["s_rid"].to_list() == [1, 1, 2, 3]
    assert np.allclose(b["p2nd"].to_list(), [0.8, 0.1, 0.0, 0.2])
    assert b["contra"].to_list() == [False, False, False, True]
    assert b["i"].to_list() == [0, 2, 4, 5]  # row index of the best pair in meta


def test_assign_is_many_to_one_not_one_to_one():
    meta, p, contra = _meta()
    links = assign(meta, p, 0.5)
    # Source 1 entity 1 receives records 10 and 11 (many-to-one); nobody receives two links per record
    assert links.filter(pl.col("s_rid") == 1).height == 2
    assert links["t_rid"].n_unique() == links.height
    assert set(links["t_rid"].to_list()) == {10, 11, 12, 13}


def test_margin_rejects_close_calls():
    meta, p, contra = _meta()
    links = assign(meta, p, 0.5, margin=0.2)
    assert 10 not in links["t_rid"].to_list()  # 0.9 - 0.8 < 0.2
    assert set(links["t_rid"].to_list()) == {11, 12, 13}


def test_contradiction_penalty_and_hard_veto():
    meta, p, contra = _meta()
    assert 13 in assign(meta, p, 0.5, contra_penalty=0.3, contra=contra)["t_rid"].to_list()   # 0.85 >= 0.8
    assert 13 not in assign(meta, p, 0.5, contra_penalty=0.4, contra=contra)["t_rid"].to_list()
    assert 13 not in assign(meta, p, 0.5, contra_penalty=1.0, contra=contra)["t_rid"].to_list()  # hard veto
    assert 13 in assign(meta, p, 0.5, contra_penalty=1.0, contra=None)["t_rid"].to_list()     # no flags: no veto


def test_per_row_thresholds_follow_meta_rows():
    meta, p, contra = _meta()
    thr = np.full(meta.height, 0.5)
    thr[4] = 0.7  # the pair (12, 2) gets a stricter threshold
    links = assign(meta, p, thr)
    assert 12 not in links["t_rid"].to_list()


def test_accept_mask_matches_model_accept():
    meta, p, contra = _meta()
    truth = pl.DataFrame({"t_rid": np.array([10, 11], dtype=np.uint32), "s_rid": np.array([1, 1], dtype=np.uint32)})
    links, nt = link_table(meta, p, truth, np.array([1, 2, 3], dtype=np.uint32), contra)
    b = best_candidates(meta, p, contra)
    for rule in ({}, {"margin": 0.2}, {"contra_penalty": 0.4}, {"margin": 0.2, "contra_penalty": 1.0}):
        m1 = accept_mask(links, 0.5, **rule)
        m2 = accept(b, 0.5, **rule)
        # both tables are per record; compare as {t_rid: accepted}
        d1 = dict(zip(links["t_rid"].to_list(), m1.tolist()))
        d2 = dict(zip(b["t_rid"].to_list(), m2.tolist()))
        assert d1 == d2
    assert nt.tolist() == [2, 0, 0]
    m = metrics_at(links, nt, 0.5)
    assert m["singleton_fp"] == 2 and m["fp_decoy"] == 2 and m["links"] == 4  # records 12, 13 are decoys linked to singletons 2, 3
    assert abs(m["macro_f05"] - 1 / 3) < 1e-9  # entity 1 perfect, singletons 2 and 3 wrong


def test_contradiction_flag_definition():
    df = pl.DataFrame({"unit_conflict": [1, -1, -1, -1, -1], "pc_conflict": [-1, 1, -1, -1, -1], "na_conflict": [0, 0, 1, 0, 0],
                       "hn_conflict": [0, 0, 0, 1, 1], "st_eq": [1, 1, 1, 1, 0]})
    assert contradiction_flag(df).tolist() == [True, True, True, True, False]


def test_select_rule_prefers_margin_when_it_helps():
    """Close calls (runner-up within 0.1) are wrong 70 % of the time, so a margin rule must win the
    nested comparison; with no such structure the plain threshold must be kept."""
    rng = np.random.default_rng(3)
    n_ent = 3000
    t_rid, s_rid, p, truth_t, truth_s = [], [], [], [], []
    t = 100_000
    for s in range(n_ent):
        # one true record per entity, scored high
        truth_t.append(t); truth_s.append(s)
        t_rid.append(t); s_rid.append(s); p.append(rng.uniform(0.85, 0.99))
        t += 1
        # a close-call record: best candidate = this entity, runner-up almost as high; true only 30 % of the time
        other = (s + 1) % n_ent
        hi = rng.uniform(0.7, 0.9)
        t_rid += [t, t]; s_rid += [s, other]; p += [hi, hi - 0.05]
        if rng.random() < 0.3:
            truth_t.append(t); truth_s.append(s)
        t += 1
    meta = pl.DataFrame({"t_rid": np.array(t_rid, dtype=np.uint32), "s_rid": np.array(s_rid, dtype=np.uint32)})
    truth = pl.DataFrame({"t_rid": np.array(truth_t, dtype=np.uint32), "s_rid": np.array(truth_s, dtype=np.uint32)})
    links, nt = link_table(meta, np.array(p, dtype=np.float32), truth, np.arange(n_ent, dtype=np.uint32))
    folds = np.arange(n_ent) % 3
    grid = np.round(np.arange(0.5, 0.99, 0.01), 2)
    rule, rows = select_rule(links, nt, folds, grid, margins=(0.0, 0.1, 0.2), penalties=(0.0,))
    assert rule["margin"] > 0 and rule["gain"] > 0
    base = next(r for r in rows if r["margin"] == 0.0)
    assert rule["nested_macro_f05"] > base["nested_macro_f05"]


In [ ]:
%%writefile /kaggle/working/er/tests/test_blocking.py
"""Multi-channel candidate generation: channel parsing, per-row top-k, the candidate audit and the
char n-gram features."""
import os
import sys

import numpy as np
import polars as pl
import pytest
import scipy.sparse as sp

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
import blocking  # noqa: E402
import textnorm  # noqa: E402


def test_parse_channels():
    assert blocking.parse_channels("combined=3") == {"combined": 3}
    assert blocking.parse_channels("combined=3,nchar=2, rev=1") == {"combined": 3, "nchar": 2, "rev": 1}
    assert blocking.parse_channels("combined=3,name=0") == {"combined": 3}  # k = 0 switches a channel off
    with pytest.raises(ValueError):
        blocking.parse_channels("name=2")  # combined is required
    with pytest.raises(ValueError):
        blocking.parse_channels("combined=3,bogus=2")


def test_topk_per_row_orders_and_offsets():
    c = sp.csr_matrix(np.array([[0.5, 0.9, 0.1, 0.0], [0.0, 0.0, 0.05, 0.3]]))
    rows, cols, score, rank = blocking.topk_per_row(c, k=2, min_score=0.1, offset=10)
    assert rows.tolist() == [10, 10, 11]
    assert cols.tolist() == [1, 0, 3]
    assert rank.tolist() == [0, 1, 0]
    assert np.allclose(score, [0.9, 0.5, 0.3])


def test_char_ngrams():
    assert textnorm.char_ngrams("dent", 4) == ["#den", "dent", "ent#"]
    assert textnorm.char_ngrams("ab", 4) == ["#ab#"]
    r = textnorm.normalize_record("Dent Diner", "1 Main St, Springfield, IL", "US")
    assert "g:#den" in r["feat_nchar"].split() and "g:ner#" in r["feat_nchar"].split()
    # a typo keeps most grams in common with the correct spelling
    a = set(textnorm.char_ngrams("college")); b = set(textnorm.char_ngrams("co1lege"))
    assert len(a & b) >= 2


def test_candidate_audit_recall_coverage_oracle():
    truth = pl.DataFrame({"t_rid": np.array([10, 11, 12, 13], dtype=np.uint32), "s_rid": np.array([1, 1, 2, 3], dtype=np.uint32)})
    # entity 1: both records found (one by the combined channel only, one by nchar only); entity 2: found;
    # entity 3: missed; entity 4: singleton
    cand = pl.DataFrame({"t_rid": np.array([10, 11, 12, 13], dtype=np.uint32), "s_rid": np.array([1, 1, 2, 9], dtype=np.uint32),
                         "n_ch": [1, 1, 2, 1], "r_combined": [0, -1, 0, 0], "r_nchar": [-1, 0, 1, -1]})
    audit = blocking.candidate_audit(cand, truth, np.array([1, 2, 3, 4], dtype=np.uint32), {"combined": 3, "nchar": 2},
                                     ambiguous=np.array([False, False, False, True]))
    assert audit["pair_recall"] == 0.75
    assert audit["pair_recall_by_channel"] == {"combined": 0.5, "nchar": 0.5}
    assert audit["true_pairs_found_by_one_channel_only"] == {"combined": 1, "nchar": 1}
    assert audit["entity_complete_coverage"] == pytest.approx(2 / 3)
    assert audit["entity_zero_coverage"] == pytest.approx(1 / 3)
    # oracle: entity 1 -> 1, entity 2 -> 1, entity 3 -> 0, singleton -> 1
    assert audit["oracle_macro_f05"] == pytest.approx(0.75)
    assert audit["misses"] == {"ambiguous_true_pairs": 1, "missed_ambiguous": 1, "missed_recoverable": 0, "recall_on_recoverable": 1.0}


def test_union_keeps_rank_per_channel(tmp_path):
    """End to end on a tiny in-memory country: every channel proposes its top-k, the union carries one
    row per pair with the rank in each channel and the exact combined cosine."""
    import argparse
    recs = [
        (1, 1, "Dent Diner", "12 Main St, Springfield, IL"),
        (2, 1, "Dent Diner", "900 Oak Ave, Peoria, IL"),
        (3, 1, "Acme Tools Inc", "5 Elm Rd, Peoria, IL"),
        (10, 2, "Dent Diner", "12 Main Street, Springfield, IL"),
        (11, 3, "Acme Too1s", None),
        (12, 2, "Zebra Unrelated", "77 Nowhere, Peoria, IL"),
    ]
    rows = []
    for rid, src, name, addr in recs:
        r = textnorm.normalize_record(name, addr, "US")
        rows.append({"rid": rid, "src": src, "country": "US", **{k: r[k] for k in blocking.BLOCKS}})
    rec = pl.DataFrame(rows).with_columns(pl.col("rid").cast(pl.UInt32), pl.col("src").cast(pl.Int8))
    args = argparse.Namespace(df_cap=1000, rare_df=5, chunk=4000)
    # serial retrieval (no process pool) keeps the test fast and deterministic
    def serial_pool(tmp, mats, channels, reverse, bounds, tag):
        blocking._init(tmp, mats, channels, reverse)
        out = []
        for b in bounds:
            out.extend(blocking._retrieve(b))
        return out
    blocking.run_pool, orig = serial_pool, blocking.run_pool
    try:
        out = blocking.run_country(rec, "US", args, str(tmp_path), blocking.parse_channels("combined=3,name=2,nchar=2,addr=1,cross=1,rare=2,hn=2,rev=2"))
    finally:
        blocking.run_pool = orig
    assert out.select("t_rid", "s_rid").unique().height == out.height
    pair = out.filter((pl.col("t_rid") == 10) & (pl.col("s_rid") == 1)).row(0, named=True)
    assert pair["rank"] == 0 and pair["r_combined"] == 0 and pair["n_ch"] >= 3
    assert pair["cos_nchar"] > 0.9  # identical compact name
    # the typo'd name still reaches its entity through the char n-gram / rev channels
    assert out.filter((pl.col("t_rid") == 11) & (pl.col("s_rid") == 3)).height == 1
    assert "r_rev" in out.columns and (out["r_rev"] >= 0).any()


In [ ]:
# ---------------------------------------------------------------- locate the dataset
DATA = f"{WORK}/dataset"
for split in ("train", "test"):
    os.makedirs(f"{DATA}/{split}", exist_ok=True)
names = {"train": ["train_source1", "train_source2", "train_source3", "train_ground_truth"],
         "test": ["test_source1", "test_source2", "test_source3"]}
missing = []
for split, fs in names.items():
    for f in fs:
        hits = glob.glob(f"{DATASET_ROOT}/**/{f}.tsv", recursive=True)
        if not hits:
            missing.append(f); continue
        dst = f"{DATA}/{split}/{f}.tsv"
        if os.path.lexists(dst): os.remove(dst)
        os.symlink(hits[0], dst)
        print(f"{f}.tsv  <-  {hits[0]}  ({os.path.getsize(hits[0]) / 1e6:.0f} MB)")
assert not missing, f"not found under {DATASET_ROOT}: {missing} - attach the challenge dataset to the notebook"

In [ ]:
# ---------------------------------------------------------------- step runner
os.chdir(ER)
def run_step(name, args, cwd=ER):
    """Runs one pipeline script as a subprocess with its own log; raises on failure."""
    log = f"{LOGS}/{name}.log"
    t0 = time.time()
    print(f"[{time.strftime('%H:%M:%S')}] start {name}", flush=True)
    with open(log, "w") as fh:
        r = subprocess.run([sys.executable] + args, cwd=cwd, stdout=fh, stderr=subprocess.STDOUT)
    dt = time.time() - t0
    if r.returncode != 0:
        print(open(log).read()[-4000:])
        raise RuntimeError(f"{name} failed after {dt:.0f}s (log: {log})")
    print(f"[{time.strftime('%H:%M:%S')}] done  {name} in {dt:.0f}s", flush=True)

print(subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], cwd=ER, capture_output=True, text=True).stdout[-400:])

## Smoke test
The whole pipeline on a 0.3 % name-group slice of the training data, scored on a disjoint labelled hold-out. Catches environment problems in minutes instead of hours.

In [ ]:
if not SKIP_SMOKE:
    SD, SW, SO = f"{WORK}/smoke/data", f"{WORK}/smoke/work", f"{WORK}/smoke/out"
    shutil.rmtree(f"{WORK}/smoke", ignore_errors=True)
    run_step("smoke_00_subset", ["tools/make_subset.py", "--data-dir", DATA, "--out-dir", SD, "--frac", "0.003", "--test-frac", "0.002"])
    run_step("smoke_01_folds", ["src/folds.py", "--data-dir", SD, "--work-dir", SW])
    run_step("smoke_02_lexicon", ["src/build_lexicon.py", "--data-dir", SD, "--work-dir", SW])
    for sp in ("train", "test"):
        run_step(f"smoke_03_prepare_{sp}", ["src/prepare.py", "--data-dir", SD, "--work-dir", SW, "--split", sp])
        run_step(f"smoke_04_blocking_{sp}", ["src/blocking.py", "--data-dir", SD, "--work-dir", SW, "--split", sp])
        run_step(f"smoke_05_features_{sp}", ["src/pair_features.py", "--data-dir", SD, "--work-dir", SW, "--split", sp])
    run_step("smoke_06_train", ["src/train.py", "--data-dir", SD, "--work-dir", SW, "--rounds1", "30", "--rounds2", "20", "--checkpoint-name", "smoke"])
    run_step("smoke_07_leakage_check", ["src/leakage_check.py", "--data-dir", SD, "--work-dir", SW, "--canary-rows", "50000", "--canary-rounds", "10"])
    run_step("smoke_08_select_threshold", ["src/select_threshold.py", "--data-dir", SD, "--work-dir", SW])
    run_step("smoke_08c_error_analysis", ["tools/error_analysis.py", "--data-dir", SD, "--work-dir", SW])
    run_step("smoke_09_predict", ["src/predict.py", "--data-dir", SD, "--work-dir", SW, "--out-dir", SO])
    run_step("smoke_10_validate", ["src/validate_submission.py", "--matching", f"{SO}/matching_results.tsv", "--candidate", f"{SO}/candidate_pairs.tsv", "--test-dir", f"{SD}/test"])
    run_step("smoke_11_evaluate", ["src/evaluate.py", "--pred", f"{SO}/matching_results.tsv", "--truth", f"{SD}/test/subset_ground_truth.tsv", "--out", f"{SW}/smoke_eval.json"])
    e = json.load(open(f"{SW}/smoke_eval.json"))["overall"]
    print(f"SMOKE hold-out: macro F0.5 {e['macro_f05']:.4f}  precision {e['micro_precision']:.4f}  recall {e['micro_recall']:.4f}")
    print("smoke test OK: every stage ran end to end in this environment")
if SMOKE_ONLY:
    print("SMOKE_ONLY=True: the full-pipeline cells below are skipped")

## Full pipeline

In [ ]:
if not SMOKE_ONLY:
    run_step("01_folds", ["src/folds.py", "--data-dir", DATA, "--work-dir", WORK])
    run_step("02_lexicon", ["src/build_lexicon.py", "--data-dir", DATA, "--work-dir", WORK])
    for sp in ("train", "test"):
        run_step(f"03_prepare_{sp}", ["src/prepare.py", "--data-dir", DATA, "--work-dir", WORK, "--split", sp])
        run_step(f"04_blocking_{sp}", ["src/blocking.py", "--data-dir", DATA, "--work-dir", WORK, "--split", sp] + (["--channels", CHANNELS] if CHANNELS else []))
        run_step(f"05_features_{sp}", ["src/pair_features.py", "--data-dir", DATA, "--work-dir", WORK, "--split", sp])
    run_step("06_train", ["src/train.py", "--data-dir", DATA, "--work-dir", WORK, "--rounds1", str(ROUNDS1), "--rounds2", str(ROUNDS2),
                          "--checkpoint-name", CKPT_NAME, "--resume"])
    run_step("07_leakage_check", ["src/leakage_check.py", "--data-dir", DATA, "--work-dir", WORK])
    run_step("08_select_threshold", ["src/select_threshold.py", "--data-dir", DATA, "--work-dir", WORK])
    run_step("08c_error_analysis", ["tools/error_analysis.py", "--data-dir", DATA, "--work-dir", WORK])   # entity-level error report -> work/error_analysis.json
    run_step("09_predict", ["src/predict.py", "--data-dir", DATA, "--work-dir", WORK, "--out-dir", OUT])
    run_step("10_validate", ["src/validate_submission.py", "--matching", f"{OUT}/matching_results.tsv", "--candidate", f"{OUT}/candidate_pairs.tsv", "--test-dir", f"{DATA}/test"])

## Results

In [ ]:
if not SMOKE_ONLY:
    for f in glob.glob(f"{WORK}/*.json") + glob.glob(f"{WORK}/*.md") + glob.glob(f"{WORK}/train/*.json") + [f"{WORK}/stage1.txt", f"{WORK}/stage2.txt", f"{WORK}/lexicon.json", f"{LOGS}/08c_error_analysis.log"]:
        if os.path.exists(f):
            shutil.copy(f, DIAG)
    # the checkpoint without its bulky OOF / fold models: manifest, models, calibration, every threshold policy, experiment log
    ck = f"{WORK}/checkpoints/{CKPT_NAME}"
    if os.path.isdir(ck):
        dst = f"{DIAG}/checkpoint_{CKPT_NAME}"
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(ck, dst, ignore=shutil.ignore_patterns("oof.parquet", "oof_stage1.npy", "*_fold*.txt"))
    shutil.copy(f"{OUT}/prediction_meta.json", DIAG) if os.path.exists(f"{OUT}/prediction_meta.json") else None
    summary = subprocess.run([sys.executable, "tools/summarize_reports.py", "--work-dir", WORK], capture_output=True, text=True).stdout
    open(f"{DIAG}/summary.md", "w").write(summary)
    print(summary)
    print(open(f"{LOGS}/10_validate.log").read()[-600:])
    for f in sorted(glob.glob(f"{OUT}/*")):
        print(f"{os.path.getsize(f) / 1e6:8.1f} MB  {f}")
    print("FULL RUN COMPLETE: submission in", OUT)